In [ ]:
# ============================================================
# REPRODUCIBILITY PATH SETUP
# ============================================================

from pathlib import Path
import os

# Change TRAFFIC_FORECASTING_ROOT only if the project is stored
# somewhere other than the default Colab location.
PROJECT_ROOT = Path(
    os.environ.get(
        "TRAFFIC_FORECASTING_ROOT",
        "/content/traffic-forecasting-failure-robustness"
    )
)

RAW_DATA_ROOT = PROJECT_ROOT / "data" / "raw"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"

RAW_DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Raw data     :", RAW_DATA_ROOT)
print("Outputs      :", OUTPUT_ROOT)

# ============================================================
# NOTEBOOK 4
# Robustness and Reliability Analysis Under
# Real-World Detector Failures
# ============================================================

import os
from pathlib import Path

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------
BASE_DIR = OUTPUT_ROOT

NOTEBOOK3_DIR = BASE_DIR / "Notebook_3_Output"
NOTEBOOK4_DIR = BASE_DIR / "Notebook_4_Output"

# Create Notebook 4 output structure
NOTEBOOK4_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_DIR = NOTEBOOK4_DIR / "Figures"
TABLES_DIR = NOTEBOOK4_DIR / "Tables"
DATA_DIR = NOTEBOOK4_DIR / "Data"
RESULTS_DIR = NOTEBOOK4_DIR / "Results"
MODELS_DIR = NOTEBOOK4_DIR / "Models"

for folder in [
    FIGURES_DIR,
    TABLES_DIR,
    DATA_DIR,
    RESULTS_DIR,
    MODELS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Display setup information
# ------------------------------------------------------------
print("=" * 90)
print("NOTEBOOK 4 — ROBUSTNESS AND RELIABILITY ANALYSIS")
print("=" * 90)

print(f"\nNotebook 3 input directory:")
print(NOTEBOOK3_DIR)

print(f"\nNotebook 4 output directory:")
print(NOTEBOOK4_DIR)

print("\nNotebook 4 folders created:")
for folder in [
    FIGURES_DIR,
    TABLES_DIR,
    DATA_DIR,
    RESULTS_DIR,
    MODELS_DIR
]:
    print(f"  ✓ {folder.name}")

# ------------------------------------------------------------
# Inspect Notebook 3 saved outputs
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("NOTEBOOK 3 SAVED FILES")
print("=" * 90)

if NOTEBOOK3_DIR.exists():
    files = sorted([
        p for p in NOTEBOOK3_DIR.rglob("*")
        if p.is_file()
    ])

    print(f"\nTotal files found: {len(files)}\n")

    for i, file in enumerate(files, start=1):
        relative_path = file.relative_to(NOTEBOOK3_DIR)
        size_mb = file.stat().st_size / (1024 ** 2)

        print(
            f"{i:03d}. {str(relative_path):70s} "
            f"{size_mb:10.2f} MB"
        )

else:
    print("\n❌ Notebook_3_Output directory was not found.")

print("\n" + "=" * 90)
print("SETUP COMPLETE")
print("=" * 90)

> **Reproducibility note.** This public version uses project-relative paths. By default, the project root is `/content/traffic-forecasting-failure-robustness`. To use another location (including Google Drive), set the environment variable `TRAFFIC_FORECASTING_ROOT` before running the notebook. Raw source archives belong in `data/raw/`; generated artifacts are written under `outputs/`.


In [ ]:
# ============================================================
# STEP 2 — INSPECT NOTEBOOK 3 DATA STRUCTURE
# Identify prediction, detector, alarm, and quality variables
# ============================================================

import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

print("=" * 95)
print("STEP 2 — NOTEBOOK 3 DATA STRUCTURE INSPECTION")
print("=" * 95)

# ------------------------------------------------------------
# Candidate files
# ------------------------------------------------------------
candidate_files = {
    "August_Final_All_Model_Predictions":
        NOTEBOOK3_DIR / "Data" / "August_Final_All_Model_Predictions.parquet",

    "August_Final_Test_Predictions":
        NOTEBOOK3_DIR / "Data" / "August_Final_Test_Predictions.parquet",

    "Forecasting_August":
        NOTEBOOK3_DIR / "Data" / "Forecasting_Monthly" / "FORECASTING_2026-08.parquet",

    "Validation_All_Model_Predictions":
        NOTEBOOK3_DIR / "Data" / "Validation_All_Model_Predictions.parquet"
}

# Keywords relevant to Notebook 4
keywords = [
    "alarm",
    "record",
    "volume",
    "actual",
    "target",
    "pred",
    "site",
    "detector",
    "date",
    "time",
    "timestamp",
    "missing",
    "valid",
    "quality",
    "error"
]

# ------------------------------------------------------------
# Schema inspection
# ------------------------------------------------------------
for label, path in candidate_files.items():

    print("\n" + "=" * 95)
    print(label)
    print("=" * 95)

    if not path.exists():
        print("❌ FILE NOT FOUND")
        print(path)
        continue

    parquet_file = pq.ParquetFile(path)

    columns = parquet_file.schema.names

    print(f"Path          : {path}")
    print(f"Rows          : {parquet_file.metadata.num_rows:,}")
    print(f"Columns       : {len(columns):,}")
    print(f"Row groups    : {parquet_file.metadata.num_row_groups:,}")

    print("\nALL COLUMNS:")
    print("-" * 95)

    for i, col in enumerate(columns, start=1):
        print(f"{i:03d}. {col}")

    # Relevant columns for robustness analysis
    relevant_columns = [
        col for col in columns
        if any(keyword.lower() in col.lower() for keyword in keywords)
    ]

    print("\nPOTENTIALLY RELEVANT NOTEBOOK 4 COLUMNS:")
    print("-" * 95)

    if relevant_columns:
        for col in relevant_columns:
            print(f"  • {col}")
    else:
        print("  No obvious alarm / quality / prediction columns detected.")


# ------------------------------------------------------------
# Inspect small sample from August forecasting data
# ------------------------------------------------------------
aug_path = candidate_files["Forecasting_August"]

print("\n" + "=" * 95)
print("AUGUST FORECASTING DATA — SMALL SAMPLE")
print("=" * 95)

if aug_path.exists():

    # Read only first row group to avoid loading entire dataset unnecessarily
    pf = pq.ParquetFile(aug_path)
    sample = pf.read_row_group(0).to_pandas()

    print(f"\nSample rows loaded: {len(sample):,}")

    print("\nData types:")
    print(sample.dtypes.to_string())

    print("\nFirst 5 rows:")
    display(sample.head())

    # Search specifically for possible failure/alarm variables
    possible_failure_cols = [
        c for c in sample.columns
        if any(
            x in c.lower()
            for x in [
                "alarm",
                "ct_records",
                "has_alarm",
                "missing",
                "quality",
                "valid"
            ]
        )
    ]

    print("\nPossible detector-failure / quality columns:")
    print("-" * 95)

    if possible_failure_cols:
        for c in possible_failure_cols:
            print(f"  ✓ {c}")

            try:
                print(sample[c].value_counts(dropna=False).head(10))
            except Exception:
                pass

            print()
    else:
        print("  ⚠️ No explicit failure/alarm columns found in this sample.")

else:
    print("❌ August forecasting file not found.")

print("\n" + "=" * 95)
print("STEP 2 COMPLETE")
print("=" * 95)

In [ ]:
# ============================================================
# STEP 3 — LOCATE DETECTOR MAPPING AND REAL FAILURE DATA
# ============================================================

import os
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

NOTEBOOK2_DIR = BASE_DIR / "Notebook_2_Output"

print("=" * 100)
print("STEP 3 — LOCATING DETECTOR MAPPING AND REAL FAILURE INFORMATION")
print("=" * 100)

print(f"\nNotebook 2 directory:\n{NOTEBOOK2_DIR}")

# ------------------------------------------------------------
# 1. List all Notebook 2 files
# ------------------------------------------------------------
if not NOTEBOOK2_DIR.exists():
    print("\n❌ Notebook_2_Output directory was not found.")
else:
    all_files = sorted([
        p for p in NOTEBOOK2_DIR.rglob("*")
        if p.is_file()
    ])

    print(f"\nTotal Notebook 2 files found: {len(all_files):,}")

    print("\n" + "=" * 100)
    print("NOTEBOOK 2 FILE INVENTORY")
    print("=" * 100)

    for i, path in enumerate(all_files, start=1):
        size_mb = path.stat().st_size / (1024 ** 2)
        rel = path.relative_to(NOTEBOOK2_DIR)

        print(
            f"{i:03d}. {str(rel):75s}"
            f"{size_mb:10.2f} MB"
        )

# ------------------------------------------------------------
# 2. Search schemas/headers for relevant Notebook 4 variables
# ------------------------------------------------------------
search_terms = [
    "detector_code",
    "scats",
    "site",
    "nb_detector",
    "alarm",
    "ct_alarm",
    "ct_records",
    "has_alarm",
    "record",
    "missing",
    "quality"
]

candidate_results = []

print("\n" + "=" * 100)
print("SEARCHING NOTEBOOK 2 FILE SCHEMAS")
print("=" * 100)

if NOTEBOOK2_DIR.exists():

    for path in all_files:

        suffix = path.suffix.lower()

        try:
            # --------------------------------------------
            # Parquet: schema only — does NOT load dataset
            # --------------------------------------------
            if suffix == ".parquet":

                pf = pq.ParquetFile(path)
                columns = pf.schema.names

            # --------------------------------------------
            # CSV: header only
            # --------------------------------------------
            elif suffix == ".csv":

                columns = list(pd.read_csv(path, nrows=0).columns)

            else:
                continue

            matched_columns = [
                col for col in columns
                if any(term in col.lower() for term in search_terms)
            ]

            if matched_columns:

                candidate_results.append({
                    "file": str(path.relative_to(NOTEBOOK2_DIR)),
                    "rows": (
                        pf.metadata.num_rows
                        if suffix == ".parquet"
                        else None
                    ),
                    "columns": columns,
                    "matched": matched_columns
                })

        except Exception as e:
            print(f"⚠️ Could not inspect: {path.name}")
            print(f"   Reason: {e}")

# ------------------------------------------------------------
# 3. Display candidate files
# ------------------------------------------------------------
print("\n" + "=" * 100)
print("CANDIDATE FILES FOR NOTEBOOK 4")
print("=" * 100)

if not candidate_results:

    print("\n⚠️ No relevant mapping/alarm columns were found in Notebook 2.")

else:

    print(f"\nCandidate files found: {len(candidate_results)}")

    for i, result in enumerate(candidate_results, start=1):

        print("\n" + "-" * 100)
        print(f"CANDIDATE {i}")
        print("-" * 100)

        print(f"File : {result['file']}")

        if result["rows"] is not None:
            print(f"Rows : {result['rows']:,}")

        print("\nRelevant columns:")

        for col in result["matched"]:
            print(f"  ✓ {col}")

        print("\nAll columns:")
        print(result["columns"])

# ------------------------------------------------------------
# 4. Search filenames themselves for likely mapping/quality files
# ------------------------------------------------------------
print("\n" + "=" * 100)
print("FILES WITH RELEVANT NAMES")
print("=" * 100)

filename_keywords = [
    "mapping",
    "detector",
    "alarm",
    "quality",
    "selected",
    "modeling",
    "clean"
]

name_matches = []

if NOTEBOOK2_DIR.exists():

    for path in all_files:

        if any(k in path.name.lower() for k in filename_keywords):
            name_matches.append(path)

if name_matches:

    for path in name_matches:
        print(f"  • {path.relative_to(NOTEBOOK2_DIR)}")

else:
    print("  No filename matches found.")

print("\n" + "=" * 100)
print("STEP 3 COMPLETE")
print("=" * 100)

In [ ]:
# ============================================================
# STEP 4 — FIX DUPLICATE MERGE COLUMN NAMES
# Continue failure audit after merge
# ============================================================

print("=" * 100)
print("STEP 4 — FIXING MERGED COLUMN NAMES")
print("=" * 100)

# ------------------------------------------------------------
# Inspect merged column names
# ------------------------------------------------------------
print("\nCurrent relevant columns in aug_failures:")

relevant_cols = [
    c for c in aug_failures.columns
    if any(
        key in c
        for key in [
            "Volume_Group",
            "Alarm_Exposure_Group",
            "NM_REGION",
            "Sensitivity_Cohort"
        ]
    )
]

for c in relevant_cols:
    print(f"  • {c}")

# ------------------------------------------------------------
# Create clean canonical column names
# Prefer mapping values when duplicate columns exist
# ------------------------------------------------------------

def create_canonical_column(df, base_name):

    candidates = [
        f"{base_name}_Mapping",
        f"{base_name}_Failure",
        base_name
    ]

    for c in candidates:
        if c in df.columns:
            df[base_name] = df[c]
            print(f"✓ {base_name} created from {c}")
            return

    print(f"⚠️ No source column found for {base_name}")


create_canonical_column(aug_failures, "Volume_Group")
create_canonical_column(aug_failures, "Alarm_Exposure_Group")
create_canonical_column(aug_failures, "NM_REGION")
create_canonical_column(aug_failures, "Sensitivity_Cohort")

# ------------------------------------------------------------
# Re-display August failure-event summary
# ------------------------------------------------------------
print("\n" + "=" * 100)
print("AUGUST FAILURE EVENTS — CORRECTED SAMPLE")
print("=" * 100)

sample_cols = [
    "Date",
    "Detector_Code",
    "NB_SCATS_SITE",
    "NB_DETECTOR",
    "Start_Interval_Index",
    "End_Interval_Index",
    "Run_Length",
    "CT_ALARM_24HOUR"
]

# Add optional metadata columns only if available
for c in [
    "NM_REGION",
    "Volume_Group",
    "Alarm_Exposure_Group",
    "Sensitivity_Cohort"
]:
    if c in aug_failures.columns:
        sample_cols.append(c)

display(
    aug_failures[sample_cols]
    .head(20)
)

# ------------------------------------------------------------
# Final August summary
# ------------------------------------------------------------
print("\n" + "=" * 100)
print("AUGUST TEST PERIOD — FINAL FAILURE SUMMARY")
print("=" * 100)

print(f"\nFailure runs          : {len(aug_failures):,}")
print(f"Failure detectors     : {aug_failures['Detector_Code'].nunique():,}")
print(f"Failure dates         : {aug_failures['Date'].nunique():,}")
print(f"Affected SCATS sites  : {aug_failures['NB_SCATS_SITE'].nunique():,}")
print(f"Failed 15-min intervals: {aug_failures['Run_Length'].sum():,}")

print("\nRun-length statistics:")
print(
    aug_failures["Run_Length"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

# ------------------------------------------------------------
# Save corrected audited file
# ------------------------------------------------------------
audit_output = DATA_DIR / "August_Real_Failure_Runs_Audited.csv"

aug_failures.to_csv(
    audit_output,
    index=False
)

print("\n" + "=" * 100)
print("CORRECTED AUDIT FILE SAVED")
print("=" * 100)

print(audit_output)

print("\n" + "=" * 100)
print("STEP 4 NOW COMPLETE")
print("=" * 100)

In [ ]:
# ============================================================
# STEP 5 — EXPAND REAL FAILURE RUNS TO 15-MIN INTERVALS
# AND AUDIT GROUND-TRUTH / PREDICTION AVAILABILITY
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 105)
print("STEP 5 — REAL FAILURE INTERVAL EXPANSION AND GROUND-TRUTH AUDIT")
print("=" * 105)

# ------------------------------------------------------------
# Reload audited failure runs if needed
# ------------------------------------------------------------
failure_run_file = DATA_DIR / "August_Real_Failure_Runs_Audited.csv"

aug_failures = pd.read_csv(failure_run_file)
aug_failures["Date"] = pd.to_datetime(aug_failures["Date"])

print(f"\nFailure runs loaded: {len(aug_failures):,}")

# ------------------------------------------------------------
# 1. Expand every contiguous run into exact 15-min timestamps
# ------------------------------------------------------------
expanded_rows = []

for run_id, row in aug_failures.reset_index(drop=True).iterrows():

    start_idx = int(row["Start_Interval_Index"])
    end_idx   = int(row["End_Interval_Index"])

    for interval_idx in range(start_idx, end_idx + 1):

        timestamp = (
            row["Date"]
            + pd.Timedelta(minutes=15 * interval_idx)
        )

        expanded_rows.append({
            "Failure_Run_ID": run_id + 1,
            "Date": row["Date"],
            "Timestamp": timestamp,
            "Detector_Code": int(row["Detector_Code"]),
            "NB_SCATS_SITE": int(row["NB_SCATS_SITE"]),
            "NB_DETECTOR": int(row["NB_DETECTOR"]),
            "Interval_Index": interval_idx,
            "Run_Length": int(row["Run_Length"]),
            "CT_ALARM_24HOUR": row["CT_ALARM_24HOUR"]
        })

failure_intervals = pd.DataFrame(expanded_rows)

print("\n" + "=" * 105)
print("EXPANDED REAL FAILURE INTERVALS")
print("=" * 105)

print(f"\nExpanded rows              : {len(failure_intervals):,}")
print(
    f"Unique Detector-Timestamps: "
    f"{failure_intervals[['Detector_Code','Timestamp']].drop_duplicates().shape[0]:,}"
)

duplicate_keys = failure_intervals.duplicated(
    subset=["Detector_Code", "Timestamp"]
).sum()

print(f"Duplicate Detector-Timestamps: {duplicate_keys:,}")

# Remove duplicates defensively if any exist
failure_intervals = (
    failure_intervals
    .drop_duplicates(
        subset=["Detector_Code", "Timestamp"],
        keep="first"
    )
    .reset_index(drop=True)
)

print(f"Final unique failure intervals: {len(failure_intervals):,}")

# ------------------------------------------------------------
# 2. Save interval-level real-failure library
# ------------------------------------------------------------
failure_interval_file = (
    DATA_DIR / "August_Real_Failure_Intervals_15min.parquet"
)

failure_intervals.to_parquet(
    failure_interval_file,
    index=False
)

print("\nSaved interval-level failure file:")
print(failure_interval_file)

# ------------------------------------------------------------
# 3. Load August forecasting dataset
# ------------------------------------------------------------
forecast_file = (
    NOTEBOOK3_DIR
    / "Data"
    / "Forecasting_Monthly"
    / "FORECASTING_2026-08.parquet"
)

forecast_cols = [
    "Timestamp",
    "Detector_Code",
    "Target_Volume",
    "All_Lags_Available"
]

forecast_aug = pd.read_parquet(
    forecast_file,
    columns=forecast_cols
)

forecast_aug["Timestamp"] = pd.to_datetime(
    forecast_aug["Timestamp"]
)

print("\n" + "=" * 105)
print("AUGUST FORECASTING DATA")
print("=" * 105)

print(f"\nRows loaded: {len(forecast_aug):,}")

# ------------------------------------------------------------
# 4. Match exact real-failure timestamps to forecasting rows
# ------------------------------------------------------------
failure_forecast = failure_intervals.merge(
    forecast_aug,
    on=["Timestamp", "Detector_Code"],
    how="left",
    indicator=True
)

matched_forecast = (
    failure_forecast["_merge"] == "both"
)

print("\n" + "=" * 105)
print("REAL FAILURE ↔ FORECASTING DATA COVERAGE")
print("=" * 105)

print(
    f"\nReal failure intervals             : "
    f"{len(failure_intervals):,}"
)

print(
    f"Found in forecasting dataset       : "
    f"{matched_forecast.sum():,}"
)

print(
    f"Not found in forecasting dataset   : "
    f"{(~matched_forecast).sum():,}"
)

# ------------------------------------------------------------
# 5. Ground-truth validity during exact failures
# ------------------------------------------------------------
matched = failure_forecast[
    failure_forecast["_merge"] == "both"
].copy()

if len(matched) > 0:

    target = matched["Target_Volume"]

    n_missing = target.isna().sum()
    n_negative = (target < 0).sum()
    n_zero = (target == 0).sum()
    n_positive = (target > 0).sum()

    print("\n" + "=" * 105)
    print("GROUND-TRUTH STATUS DURING REAL FAILURE INTERVALS")
    print("=" * 105)

    print(f"\nMatched failure intervals : {len(matched):,}")
    print(f"Missing Target_Volume     : {n_missing:,}")
    print(f"Negative Target_Volume    : {n_negative:,}")
    print(f"Zero Target_Volume        : {n_zero:,}")
    print(f"Positive Target_Volume    : {n_positive:,}")

    valid_ground_truth = (
        target.notna()
        & (target >= 0)
    )

    print(
        f"\nPotentially valid observed targets: "
        f"{valid_ground_truth.sum():,}"
    )

    print("\nAll_Lags_Available during failure intervals:")
    print(
        matched["All_Lags_Available"]
        .value_counts(dropna=False)
        .to_string()
    )

# ------------------------------------------------------------
# 6. Load final August predictions
# ------------------------------------------------------------
prediction_file = (
    NOTEBOOK3_DIR
    / "Data"
    / "August_Final_All_Model_Predictions.parquet"
)

pred_cols = [
    "Timestamp",
    "Detector_Code",
    "Actual",
    "LightGBM_Prediction",
    "XGBoost_Prediction"
]

pred_aug = pd.read_parquet(
    prediction_file,
    columns=pred_cols
)

pred_aug["Timestamp"] = pd.to_datetime(
    pred_aug["Timestamp"]
)

# ------------------------------------------------------------
# 7. Check whether exact failure intervals survived into
#    the final evaluation prediction dataset
# ------------------------------------------------------------
failure_predictions = failure_intervals.merge(
    pred_aug,
    on=["Timestamp", "Detector_Code"],
    how="left",
    indicator=True
)

matched_predictions = (
    failure_predictions["_merge"] == "both"
)

print("\n" + "=" * 105)
print("REAL FAILURE ↔ FINAL PREDICTION DATASET COVERAGE")
print("=" * 105)

print(
    f"\nFailure intervals represented in final predictions: "
    f"{matched_predictions.sum():,}"
)

print(
    f"Failure intervals absent from final predictions     : "
    f"{(~matched_predictions).sum():,}"
)

# ------------------------------------------------------------
# 8. Actual-value status if any failure intervals are present
# ------------------------------------------------------------
pred_failure_matched = failure_predictions[
    failure_predictions["_merge"] == "both"
].copy()

if len(pred_failure_matched) > 0:

    actual = pred_failure_matched["Actual"]

    print("\nActual values for failure intervals in prediction file:")
    print(f"  Missing : {actual.isna().sum():,}")
    print(f"  Negative: {(actual < 0).sum():,}")
    print(f"  Zero    : {(actual == 0).sum():,}")
    print(f"  Positive: {(actual > 0).sum():,}")

# ------------------------------------------------------------
# 9. Save audit table
# ------------------------------------------------------------
audit_summary = pd.DataFrame({
    "Metric": [
        "Failure_Runs",
        "Failure_Intervals",
        "Unique_Failure_Detectors",
        "Forecasting_Matches",
        "Forecasting_Nonmatches",
        "Prediction_Matches",
        "Prediction_Nonmatches"
    ],
    "Value": [
        len(aug_failures),
        len(failure_intervals),
        failure_intervals["Detector_Code"].nunique(),
        int(matched_forecast.sum()),
        int((~matched_forecast).sum()),
        int(matched_predictions.sum()),
        int((~matched_predictions).sum())
    ]
})

audit_summary_file = (
    TABLES_DIR / "Table_Real_Failure_GroundTruth_Audit.csv"
)

audit_summary.to_csv(
    audit_summary_file,
    index=False
)

print("\n" + "=" * 105)
print("AUDIT SUMMARY")
print("=" * 105)

display(audit_summary)

print("\nSaved:")
print(audit_summary_file)

print("\n" + "=" * 105)
print("STEP 5 COMPLETE")
print("=" * 105)

In [ ]:
# ============================================================
# STEP 6 — REAL-FAILURE FORECAST RECOVERY AUDIT
# How quickly does forecasting availability return
# after a real detector failure ends?
# ============================================================

import pandas as pd
import numpy as np

print("=" * 105)
print("STEP 6 — REAL-FAILURE FORECAST RECOVERY AUDIT")
print("=" * 105)

# ------------------------------------------------------------
# 1. Load audited August failure runs
# ------------------------------------------------------------
failure_run_file = DATA_DIR / "August_Real_Failure_Runs_Audited.csv"

runs = pd.read_csv(failure_run_file)
runs["Date"] = pd.to_datetime(runs["Date"])

# Exact start/end timestamps
runs["Failure_Start"] = (
    runs["Date"]
    + pd.to_timedelta(
        runs["Start_Interval_Index"] * 15,
        unit="m"
    )
)

runs["Failure_End"] = (
    runs["Date"]
    + pd.to_timedelta(
        runs["End_Interval_Index"] * 15,
        unit="m"
    )
)

# First theoretically healthy interval after failure
runs["First_Post_Failure_Interval"] = (
    runs["Failure_End"]
    + pd.Timedelta(minutes=15)
)

print(f"\nFailure runs loaded: {len(runs):,}")

# ------------------------------------------------------------
# 2. Load final August predictions
# ------------------------------------------------------------
prediction_file = (
    NOTEBOOK3_DIR
    / "Data"
    / "August_Final_All_Model_Predictions.parquet"
)

pred = pd.read_parquet(
    prediction_file,
    columns=[
        "Timestamp",
        "Detector_Code",
        "Actual",
        "LightGBM_Prediction",
        "XGBoost_Prediction",
        "Persistence_15min",
        "Seasonal_Naive_24h"
    ]
)

pred["Timestamp"] = pd.to_datetime(pred["Timestamp"])

print(f"Prediction rows loaded: {len(pred):,}")

# ------------------------------------------------------------
# 3. Organize prediction timestamps by detector
# ------------------------------------------------------------
pred_times = {
    int(det): np.sort(group["Timestamp"].values)
    for det, group in pred.groupby("Detector_Code")
}

# ------------------------------------------------------------
# 4. Find last forecast before and first forecast after failure
# ------------------------------------------------------------
results = []

for _, row in runs.iterrows():

    detector = int(row["Detector_Code"])

    failure_start = pd.Timestamp(row["Failure_Start"])
    failure_end = pd.Timestamp(row["Failure_End"])

    first_post = failure_end + pd.Timedelta(minutes=15)

    times = pred_times.get(detector)

    last_before = pd.NaT
    first_after = pd.NaT

    if times is not None and len(times) > 0:

        times_pd = pd.to_datetime(times)

        # Last available forecast before failure begins
        before = times_pd[times_pd < failure_start]

        if len(before) > 0:
            last_before = before.max()

        # First available forecast at/after first healthy interval
        after = times_pd[times_pd >= first_post]

        if len(after) > 0:
            first_after = after.min()

    # Recovery delay
    if pd.notna(first_after):

        recovery_minutes = (
            first_after - first_post
        ).total_seconds() / 60

        recovery_intervals = recovery_minutes / 15

    else:

        recovery_minutes = np.nan
        recovery_intervals = np.nan

    results.append({
        "Detector_Code": detector,
        "NB_SCATS_SITE": row["NB_SCATS_SITE"],
        "NB_DETECTOR": row["NB_DETECTOR"],
        "Date": row["Date"],
        "Failure_Run_Length": row["Run_Length"],
        "Failure_Start": failure_start,
        "Failure_End": failure_end,
        "First_Post_Failure_Interval": first_post,
        "Last_Prediction_Before": last_before,
        "First_Prediction_After": first_after,
        "Recovery_Delay_Minutes": recovery_minutes,
        "Recovery_Delay_Intervals": recovery_intervals
    })

recovery = pd.DataFrame(results)

# ------------------------------------------------------------
# 5. Restrict interpretation to failures where post-failure
#    time remains inside the August test period
# ------------------------------------------------------------
test_end = pd.Timestamp("2026-08-30 23:45:00")

recovery["Post_Failure_Inside_Test"] = (
    recovery["First_Post_Failure_Interval"] <= test_end
)

eligible = recovery[
    recovery["Post_Failure_Inside_Test"]
].copy()

print("\n" + "=" * 105)
print("RECOVERY COVERAGE")
print("=" * 105)

print(f"\nTotal failure runs                  : {len(recovery):,}")
print(f"Eligible for post-failure analysis : {len(eligible):,}")

print(
    f"Runs with forecast returning       : "
    f"{eligible['First_Prediction_After'].notna().sum():,}"
)

print(
    f"Runs with no later forecast        : "
    f"{eligible['First_Prediction_After'].isna().sum():,}"
)

# ------------------------------------------------------------
# 6. Recovery delay statistics
# ------------------------------------------------------------
valid_recovery = eligible[
    eligible["Recovery_Delay_Intervals"].notna()
].copy()

print("\n" + "=" * 105)
print("FORECAST RECOVERY DELAY")
print("=" * 105)

if len(valid_recovery) > 0:

    print("\nRecovery delay in 15-minute intervals:")
    print(
        valid_recovery["Recovery_Delay_Intervals"]
        .describe(
            percentiles=[
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99
            ]
        )
    )

    immediate = (
        valid_recovery["Recovery_Delay_Intervals"] == 0
    ).sum()

    within_1h = (
        valid_recovery["Recovery_Delay_Intervals"] <= 4
    ).sum()

    within_2h = (
        valid_recovery["Recovery_Delay_Intervals"] <= 8
    ).sum()

    within_6h = (
        valid_recovery["Recovery_Delay_Intervals"] <= 24
    ).sum()

    print("\nRecovery thresholds:")

    print(
        f"Immediate next interval : "
        f"{immediate:,} "
        f"({100 * immediate / len(valid_recovery):.2f}%)"
    )

    print(
        f"Within 1 hour           : "
        f"{within_1h:,} "
        f"({100 * within_1h / len(valid_recovery):.2f}%)"
    )

    print(
        f"Within 2 hours          : "
        f"{within_2h:,} "
        f"({100 * within_2h / len(valid_recovery):.2f}%)"
    )

    print(
        f"Within 6 hours          : "
        f"{within_6h:,} "
        f"({100 * within_6h / len(valid_recovery):.2f}%)"
    )

# ------------------------------------------------------------
# 7. Relationship between failure duration and recovery delay
# ------------------------------------------------------------
print("\n" + "=" * 105)
print("FAILURE LENGTH VS FORECAST RECOVERY")
print("=" * 105)

if len(valid_recovery) > 1:

    corr = valid_recovery[
        [
            "Failure_Run_Length",
            "Recovery_Delay_Intervals"
        ]
    ].corr().iloc[0, 1]

    print(
        f"\nCorrelation between failure run length "
        f"and recovery delay: {corr:.4f}"
    )

# ------------------------------------------------------------
# 8. Recovery-delay frequency table
# ------------------------------------------------------------
print("\nMost common recovery delays:")

delay_distribution = (
    valid_recovery["Recovery_Delay_Intervals"]
    .value_counts()
    .sort_index()
    .head(30)
)

print(delay_distribution.to_string())

# ------------------------------------------------------------
# 9. Show examples
# ------------------------------------------------------------
print("\n" + "=" * 105)
print("SAMPLE RECOVERY EVENTS")
print("=" * 105)

display(
    valid_recovery[
        [
            "Detector_Code",
            "NB_SCATS_SITE",
            "Date",
            "Failure_Run_Length",
            "Failure_Start",
            "Failure_End",
            "First_Post_Failure_Interval",
            "First_Prediction_After",
            "Recovery_Delay_Intervals"
        ]
    ]
    .sort_values(
        "Recovery_Delay_Intervals",
        ascending=False
    )
    .head(20)
)

# ------------------------------------------------------------
# 10. Save recovery audit
# ------------------------------------------------------------
recovery_file = (
    DATA_DIR
    / "August_Real_Failure_Forecast_Recovery_Audit.csv"
)

recovery.to_csv(
    recovery_file,
    index=False
)

summary_file = (
    TABLES_DIR
    / "Table_Real_Failure_Forecast_Recovery_Summary.csv"
)

summary = pd.DataFrame({
    "Metric": [
        "Total_Failure_Runs",
        "Post_Failure_Eligible_Runs",
        "Runs_With_Forecast_Return",
        "Immediate_Return_Runs",
        "Median_Recovery_Intervals",
        "Mean_Recovery_Intervals",
        "Maximum_Recovery_Intervals"
    ],
    "Value": [
        len(recovery),
        len(eligible),
        valid_recovery.shape[0],
        (
            valid_recovery[
                "Recovery_Delay_Intervals"
            ].eq(0).sum()
            if len(valid_recovery) > 0
            else np.nan
        ),
        (
            valid_recovery[
                "Recovery_Delay_Intervals"
            ].median()
            if len(valid_recovery) > 0
            else np.nan
        ),
        (
            valid_recovery[
                "Recovery_Delay_Intervals"
            ].mean()
            if len(valid_recovery) > 0
            else np.nan
        ),
        (
            valid_recovery[
                "Recovery_Delay_Intervals"
            ].max()
            if len(valid_recovery) > 0
            else np.nan
        )
    ]
})

summary.to_csv(
    summary_file,
    index=False
)

print("\n" + "=" * 105)
print("FILES SAVED")
print("=" * 105)

print(recovery_file)
print(summary_file)

print("\n" + "=" * 105)
print("STEP 6 COMPLETE")
print("=" * 105)

In [ ]:
# ============================================================
# STEP 7 — RECOVERY MECHANISM DIAGNOSTIC
# Separate:
# 1) Clean target-data return
# 2) Lag-feature readiness
# 3) Final prediction return
# ============================================================

import pandas as pd
import numpy as np

print("=" * 110)
print("STEP 7 — RECOVERY MECHANISM DIAGNOSTIC")
print("=" * 110)

# ------------------------------------------------------------
# 1. Load failure runs
# ------------------------------------------------------------
run_file = DATA_DIR / "August_Real_Failure_Runs_Audited.csv"

runs = pd.read_csv(run_file)
runs["Date"] = pd.to_datetime(runs["Date"])

runs["Failure_Start"] = (
    runs["Date"]
    + pd.to_timedelta(
        runs["Start_Interval_Index"] * 15,
        unit="m"
    )
)

runs["Failure_End"] = (
    runs["Date"]
    + pd.to_timedelta(
        runs["End_Interval_Index"] * 15,
        unit="m"
    )
)

runs["First_Post_Failure_Interval"] = (
    runs["Failure_End"]
    + pd.Timedelta(minutes=15)
)

print(f"\nFailure runs: {len(runs):,}")

# ------------------------------------------------------------
# 2. Load August forecasting dataset
# ------------------------------------------------------------
forecast_file = (
    NOTEBOOK3_DIR
    / "Data"
    / "Forecasting_Monthly"
    / "FORECASTING_2026-08.parquet"
)

forecast = pd.read_parquet(
    forecast_file,
    columns=[
        "Timestamp",
        "Detector_Code",
        "Target_Volume",
        "All_Lags_Available",
        "Lag_1",
        "Lag_2",
        "Lag_4",
        "Lag_8",
        "Lag_96"
    ]
)

forecast["Timestamp"] = pd.to_datetime(
    forecast["Timestamp"]
)

forecast = forecast.sort_values(
    ["Detector_Code", "Timestamp"]
)

print(f"Forecasting rows: {len(forecast):,}")

# ------------------------------------------------------------
# 3. Load final prediction timestamps
# ------------------------------------------------------------
prediction_file = (
    NOTEBOOK3_DIR
    / "Data"
    / "August_Final_All_Model_Predictions.parquet"
)

pred = pd.read_parquet(
    prediction_file,
    columns=[
        "Timestamp",
        "Detector_Code"
    ]
)

pred["Timestamp"] = pd.to_datetime(
    pred["Timestamp"]
)

pred = pred.sort_values(
    ["Detector_Code", "Timestamp"]
)

print(f"Final prediction rows: {len(pred):,}")

# ------------------------------------------------------------
# 4. Build detector-level lookup dictionaries
# ------------------------------------------------------------
forecast_groups = {
    int(det): grp.reset_index(drop=True)
    for det, grp in forecast.groupby("Detector_Code")
}

prediction_times = {
    int(det): grp["Timestamp"].to_numpy()
    for det, grp in pred.groupby("Detector_Code")
}

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------
def delay_intervals(start_time, return_time):

    if pd.isna(return_time):
        return np.nan

    return (
        (return_time - start_time).total_seconds()
        / 60
        / 15
    )

# ------------------------------------------------------------
# 5. Diagnose every failure run
# ------------------------------------------------------------
diagnostics = []

for run_id, row in runs.reset_index(drop=True).iterrows():

    detector = int(row["Detector_Code"])

    failure_start = pd.Timestamp(row["Failure_Start"])
    failure_end = pd.Timestamp(row["Failure_End"])

    first_post = pd.Timestamp(
        row["First_Post_Failure_Interval"]
    )

    first_clean_target = pd.NaT
    first_all_lags = pd.NaT
    first_prediction = pd.NaT

    # --------------------------------------------------------
    # Forecasting dataset
    # --------------------------------------------------------
    fg = forecast_groups.get(detector)

    if fg is not None:

        post_fg = fg[
            fg["Timestamp"] >= first_post
        ]

        if len(post_fg) > 0:

            # First clean target row after failure
            first_clean_target = (
                post_fg.iloc[0]["Timestamp"]
            )

            # First row where every required lag exists
            ready = post_fg[
                post_fg["All_Lags_Available"] == True
            ]

            if len(ready) > 0:
                first_all_lags = (
                    ready.iloc[0]["Timestamp"]
                )

    # --------------------------------------------------------
    # Final prediction dataset
    # --------------------------------------------------------
    ptimes = prediction_times.get(detector)

    if ptimes is not None:

        pos = np.searchsorted(
            ptimes,
            np.datetime64(first_post),
            side="left"
        )

        if pos < len(ptimes):
            first_prediction = pd.Timestamp(
                ptimes[pos]
            )

    diagnostics.append({

        "Failure_Run_ID":
            run_id + 1,

        "Detector_Code":
            detector,

        "NB_SCATS_SITE":
            row["NB_SCATS_SITE"],

        "NB_DETECTOR":
            row["NB_DETECTOR"],

        "Failure_Start":
            failure_start,

        "Failure_End":
            failure_end,

        "Failure_Run_Length":
            row["Run_Length"],

        "First_Post_Failure":
            first_post,

        "First_Clean_Target":
            first_clean_target,

        "First_All_Lags_Available":
            first_all_lags,

        "First_Final_Prediction":
            first_prediction,

        "Clean_Target_Delay_Intervals":
            delay_intervals(
                first_post,
                first_clean_target
            ),

        "Lag_Readiness_Delay_Intervals":
            delay_intervals(
                first_post,
                first_all_lags
            ),

        "Prediction_Delay_Intervals":
            delay_intervals(
                first_post,
                first_prediction
            )
    })

diag = pd.DataFrame(diagnostics)

# ------------------------------------------------------------
# 6. Overall mechanism summary
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("RECOVERY COMPONENT SUMMARY")
print("=" * 110)

for col, label in [
    (
        "Clean_Target_Delay_Intervals",
        "Clean target return"
    ),
    (
        "Lag_Readiness_Delay_Intervals",
        "All-lag readiness"
    ),
    (
        "Prediction_Delay_Intervals",
        "Final prediction return"
    )
]:

    valid = diag[col].dropna()

    print(f"\n{label}")
    print("-" * 70)

    print(f"Available events : {len(valid):,}")

    if len(valid) > 0:

        print(
            valid.describe(
                percentiles=[
                    0.25,
                    0.50,
                    0.75,
                    0.90,
                    0.95,
                    0.99
                ]
            )
        )

        print(
            f"Immediate return : "
            f"{(valid == 0).sum():,} "
            f"({100*(valid == 0).mean():.2f}%)"
        )

# ------------------------------------------------------------
# 7. Does final prediction return exactly when lags are ready?
# ------------------------------------------------------------
both = diag[
    diag["First_All_Lags_Available"].notna()
    & diag["First_Final_Prediction"].notna()
].copy()

both["Lag_vs_Prediction_Difference"] = (
    (
        both["First_Final_Prediction"]
        - both["First_All_Lags_Available"]
    )
    .dt.total_seconds()
    / 60
    / 15
)

print("\n" + "=" * 110)
print("ALL-LAG READINESS VS FINAL PREDICTION")
print("=" * 110)

print(f"\nComparable events: {len(both):,}")

if len(both) > 0:

    exact = (
        both["Lag_vs_Prediction_Difference"] == 0
    ).sum()

    print(
        f"Prediction starts exactly when all lags become ready: "
        f"{exact:,} / {len(both):,} "
        f"({100*exact/len(both):.2f}%)"
    )

    print("\nDifference distribution:")
    print(
        both[
            "Lag_vs_Prediction_Difference"
        ]
        .value_counts()
        .sort_index()
        .head(20)
        .to_string()
    )

# ------------------------------------------------------------
# 8. How often does the clean target return immediately?
# ------------------------------------------------------------
clean_valid = diag[
    "Clean_Target_Delay_Intervals"
].dropna()

print("\n" + "=" * 110)
print("CLEAN TARGET RETURN BEHAVIOR")
print("=" * 110)

if len(clean_valid) > 0:

    immediate_clean = (
        clean_valid == 0
    ).sum()

    delayed_clean = (
        clean_valid > 0
    ).sum()

    print(
        f"\nImmediate clean-target return : "
        f"{immediate_clean:,} "
        f"({100*immediate_clean/len(clean_valid):.2f}%)"
    )

    print(
        f"Delayed clean-target return   : "
        f"{delayed_clean:,} "
        f"({100*delayed_clean/len(clean_valid):.2f}%)"
    )

    print("\nMost common clean-target delays:")

    print(
        clean_valid
        .value_counts()
        .sort_index()
        .head(30)
        .to_string()
    )

# ------------------------------------------------------------
# 9. Sample events for manual inspection
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("SAMPLE DIAGNOSTIC EVENTS")
print("=" * 110)

display(
    diag[
        [
            "Detector_Code",
            "NB_SCATS_SITE",
            "Failure_Run_Length",
            "Failure_Start",
            "Failure_End",
            "First_Post_Failure",
            "First_Clean_Target",
            "First_All_Lags_Available",
            "First_Final_Prediction",
            "Clean_Target_Delay_Intervals",
            "Lag_Readiness_Delay_Intervals",
            "Prediction_Delay_Intervals"
        ]
    ]
    .sort_values(
        "Prediction_Delay_Intervals",
        ascending=False
    )
    .head(25)
)

# ------------------------------------------------------------
# 10. Special check for the 96-interval floor
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("96-INTERVAL (24-HOUR) FLOOR CHECK")
print("=" * 110)

prediction_valid = diag[
    "Prediction_Delay_Intervals"
].dropna()

if len(prediction_valid) > 0:

    print(
        f"\nMinimum prediction delay: "
        f"{prediction_valid.min():.0f} intervals "
        f"({prediction_valid.min()*15/60:.2f} hours)"
    )

    n_96_or_more = (
        prediction_valid >= 96
    ).sum()

    print(
        f"Events with delay >= 96 intervals: "
        f"{n_96_or_more:,} / {len(prediction_valid):,} "
        f"({100*n_96_or_more/len(prediction_valid):.2f}%)"
    )

# ------------------------------------------------------------
# 11. Observation-window / right-censoring check
# ------------------------------------------------------------
test_end = pd.Timestamp(
    "2026-08-30 23:45:00"
)

diag["Observable_PostFailure_Intervals"] = (
    (
        test_end
        - diag["First_Post_Failure"]
    )
    .dt.total_seconds()
    / 60
    / 15
)

diag["At_Least_24h_Observable"] = (
    diag["Observable_PostFailure_Intervals"] >= 96
)

print("\n" + "=" * 110)
print("RIGHT-CENSORING CHECK")
print("=" * 110)

print(
    f"\nEvents with at least 24 h observable after failure: "
    f"{diag['At_Least_24h_Observable'].sum():,} / {len(diag):,}"
)

no_return = diag[
    diag["First_Final_Prediction"].isna()
]

print(
    f"Events with no prediction return in August: "
    f"{len(no_return):,}"
)

if len(no_return) > 0:

    print(
        "Of those, events with <24 h observation remaining: "
        f"{(~no_return['At_Least_24h_Observable']).sum():,}"
    )

# ------------------------------------------------------------
# 12. Save diagnostic outputs
# ------------------------------------------------------------
diag_file = (
    DATA_DIR
    / "August_Real_Failure_Recovery_Mechanism_Diagnostic.csv"
)

diag.to_csv(
    diag_file,
    index=False
)

mechanism_summary = pd.DataFrame({

    "Metric": [
        "Median_Clean_Target_Delay_Intervals",
        "Median_Lag_Readiness_Delay_Intervals",
        "Median_Prediction_Delay_Intervals",
        "Min_Prediction_Delay_Intervals",
        "Events_At_Least_24h_Observable",
        "Events_No_Prediction_Return"
    ],

    "Value": [
        diag[
            "Clean_Target_Delay_Intervals"
        ].median(),

        diag[
            "Lag_Readiness_Delay_Intervals"
        ].median(),

        diag[
            "Prediction_Delay_Intervals"
        ].median(),

        diag[
            "Prediction_Delay_Intervals"
        ].min(),

        diag[
            "At_Least_24h_Observable"
        ].sum(),

        diag[
            "First_Final_Prediction"
        ].isna().sum()
    ]
})

summary_file = (
    TABLES_DIR
    / "Table_Recovery_Mechanism_Diagnostic.csv"
)

mechanism_summary.to_csv(
    summary_file,
    index=False
)

print("\n" + "=" * 110)
print("FILES SAVED")
print("=" * 110)

print(diag_file)
print(summary_file)

print("\n" + "=" * 110)
print("STEP 7 COMPLETE")
print("=" * 110)

In [ ]:
# ============================================================
# STEP 8 — ISOLATED VS RECURRENT FAILURE ANALYSIS
# Remove contamination from subsequent failure events
# ============================================================

import pandas as pd
import numpy as np

print("=" * 110)
print("STEP 8 — ISOLATED VS RECURRENT FAILURE ANALYSIS")
print("=" * 110)

# ------------------------------------------------------------
# 1. Load Step 7 diagnostic file
# ------------------------------------------------------------
diag_file = (
    DATA_DIR
    / "August_Real_Failure_Recovery_Mechanism_Diagnostic.csv"
)

diag = pd.read_csv(
    diag_file,
    parse_dates=[
        "Failure_Start",
        "Failure_End",
        "First_Post_Failure",
        "First_Clean_Target",
        "First_All_Lags_Available",
        "First_Final_Prediction"
    ]
)

diag = diag.sort_values(
    ["Detector_Code", "Failure_Start"]
).reset_index(drop=True)

print(f"\nFailure runs loaded: {len(diag):,}")

# ------------------------------------------------------------
# 2. Identify next failure on same detector
# ------------------------------------------------------------
diag["Next_Failure_Start"] = (
    diag.groupby("Detector_Code")["Failure_Start"]
    .shift(-1)
)

diag["Next_Failure_End"] = (
    diag.groupby("Detector_Code")["Failure_End"]
    .shift(-1)
)

# ------------------------------------------------------------
# 3. Determine whether another failure occurs before recovery
# ------------------------------------------------------------
diag["Subsequent_Failure_Before_Prediction"] = (
    diag["Next_Failure_Start"].notna()
    & diag["First_Final_Prediction"].notna()
    & (
        diag["Next_Failure_Start"]
        < diag["First_Final_Prediction"]
    )
)

# For cases with no prediction return, any later failure
# inside the August test period also contaminates recovery
test_end = pd.Timestamp("2026-08-30 23:45:00")

no_prediction = (
    diag["First_Final_Prediction"].isna()
)

diag.loc[
    no_prediction
    & diag["Next_Failure_Start"].notna()
    & (diag["Next_Failure_Start"] <= test_end),
    "Subsequent_Failure_Before_Prediction"
] = True

# ------------------------------------------------------------
# 4. Define clean isolated events
# ------------------------------------------------------------
diag["Isolated_Recovery_Event"] = (
    diag["First_Final_Prediction"].notna()
    & ~diag["Subsequent_Failure_Before_Prediction"]
)

isolated = diag[
    diag["Isolated_Recovery_Event"]
].copy()

recurrent = diag[
    diag["Subsequent_Failure_Before_Prediction"]
].copy()

print("\n" + "=" * 110)
print("EVENT CLASSIFICATION")
print("=" * 110)

print(f"\nTotal failure runs                    : {len(diag):,}")
print(f"Isolated recovery events             : {len(isolated):,}")
print(f"Recurrent/contaminated events        : {len(recurrent):,}")
print(
    f"No prediction and no later failure  : "
    f"{(
        diag['First_Final_Prediction'].isna()
        & ~diag['Subsequent_Failure_Before_Prediction']
    ).sum():,}"
)

# ------------------------------------------------------------
# 5. Quantify cleaning delay and lag delay separately
# ------------------------------------------------------------
isolated["Post_Clean_Lag_Delay_Intervals"] = (
    isolated["Lag_Readiness_Delay_Intervals"]
    - isolated["Clean_Target_Delay_Intervals"]
)

isolated["Post_Clean_Lag_Delay_Hours"] = (
    isolated["Post_Clean_Lag_Delay_Intervals"]
    * 15 / 60
)

isolated["Total_Recovery_Hours"] = (
    isolated["Prediction_Delay_Intervals"]
    * 15 / 60
)

isolated["Clean_Target_Recovery_Hours"] = (
    isolated["Clean_Target_Delay_Intervals"]
    * 15 / 60
)

print("\n" + "=" * 110)
print("ISOLATED FAILURE RECOVERY — CORE RESULTS")
print("=" * 110)

for col, label in [
    (
        "Clean_Target_Delay_Intervals",
        "Clean-target delay"
    ),
    (
        "Post_Clean_Lag_Delay_Intervals",
        "Additional lag-readiness delay"
    ),
    (
        "Prediction_Delay_Intervals",
        "Total prediction recovery delay"
    )
]:

    valid = isolated[col].dropna()

    print(f"\n{label}")
    print("-" * 65)

    if len(valid) > 0:
        print(
            valid.describe(
                percentiles=[
                    0.25,
                    0.50,
                    0.75,
                    0.90,
                    0.95,
                    0.99
                ]
            )
        )

# ------------------------------------------------------------
# 6. Test the expected Lag_96 effect
# ------------------------------------------------------------
lag_delay = isolated[
    "Post_Clean_Lag_Delay_Intervals"
].dropna()

print("\n" + "=" * 110)
print("LAG-96 RECOVERY SIGNATURE")
print("=" * 110)

if len(lag_delay) > 0:

    exactly_96 = (lag_delay == 96).sum()
    at_least_96 = (lag_delay >= 96).sum()

    print(
        f"\nExactly 96 additional intervals : "
        f"{exactly_96:,} / {len(lag_delay):,} "
        f"({100*exactly_96/len(lag_delay):.2f}%)"
    )

    print(
        f"At least 96 intervals           : "
        f"{at_least_96:,} / {len(lag_delay):,} "
        f"({100*at_least_96/len(lag_delay):.2f}%)"
    )

    print("\nMost common additional lag delays:")
    print(
        lag_delay
        .value_counts()
        .sort_index()
        .head(30)
        .to_string()
    )

# ------------------------------------------------------------
# 7. Does clean target return on same calendar day?
# ------------------------------------------------------------
isolated["Clean_Return_Same_Day"] = (
    isolated["First_Clean_Target"].dt.date
    == isolated["Failure_End"].dt.date
)

isolated["Clean_Return_Next_Day"] = (
    isolated["First_Clean_Target"].dt.date
    == (
        isolated["Failure_End"]
        + pd.Timedelta(days=1)
    ).dt.date
)

print("\n" + "=" * 110)
print("CLEANING PIPELINE RETURN PATTERN")
print("=" * 110)

valid_clean = isolated[
    isolated["First_Clean_Target"].notna()
].copy()

if len(valid_clean) > 0:

    same_day = valid_clean[
        "Clean_Return_Same_Day"
    ].sum()

    next_day = valid_clean[
        "Clean_Return_Next_Day"
    ].sum()

    print(
        f"\nClean target returns same calendar day : "
        f"{same_day:,} / {len(valid_clean):,} "
        f"({100*same_day/len(valid_clean):.2f}%)"
    )

    print(
        f"Clean target returns next calendar day : "
        f"{next_day:,} / {len(valid_clean):,} "
        f"({100*next_day/len(valid_clean):.2f}%)"
    )

    print("\nHour of first clean-target return:")

    hour_distribution = (
        valid_clean[
            "First_Clean_Target"
        ]
        .dt.hour
        .value_counts()
        .sort_index()
    )

    print(hour_distribution.to_string())

# ------------------------------------------------------------
# 8. Relationship between real failure duration and recovery
#    using isolated events only
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("FAILURE DURATION VS RECOVERY — ISOLATED EVENTS ONLY")
print("=" * 110)

corr_data = isolated[
    [
        "Failure_Run_Length",
        "Clean_Target_Delay_Intervals",
        "Prediction_Delay_Intervals"
    ]
].dropna()

if len(corr_data) > 2:

    corr_clean = corr_data[
        "Failure_Run_Length"
    ].corr(
        corr_data[
            "Clean_Target_Delay_Intervals"
        ]
    )

    corr_pred = corr_data[
        "Failure_Run_Length"
    ].corr(
        corr_data[
            "Prediction_Delay_Intervals"
        ]
    )

    print(
        f"\nFailure length vs clean-target delay : "
        f"{corr_clean:.4f}"
    )

    print(
        f"Failure length vs prediction delay   : "
        f"{corr_pred:.4f}"
    )

# ------------------------------------------------------------
# 9. Compare isolated vs recurrent events
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("ISOLATED VS RECURRENT SUMMARY")
print("=" * 110)

comparison = pd.DataFrame({
    "Group": [
        "Isolated",
        "Recurrent/Contaminated"
    ],
    "Events": [
        len(isolated),
        len(recurrent)
    ],
    "Median_Failure_Run_Length": [
        isolated["Failure_Run_Length"].median(),
        recurrent["Failure_Run_Length"].median()
        if len(recurrent) > 0 else np.nan
    ],
    "Median_Prediction_Delay_Intervals": [
        isolated[
            "Prediction_Delay_Intervals"
        ].median(),
        recurrent[
            "Prediction_Delay_Intervals"
        ].median()
        if len(recurrent) > 0 else np.nan
    ]
})

display(comparison)

# ------------------------------------------------------------
# 10. Show longest isolated recovery events
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("LONGEST ISOLATED RECOVERY EVENTS")
print("=" * 110)

display(
    isolated[
        [
            "Detector_Code",
            "NB_SCATS_SITE",
            "Failure_Run_Length",
            "Failure_Start",
            "Failure_End",
            "First_Clean_Target",
            "First_All_Lags_Available",
            "First_Final_Prediction",
            "Clean_Target_Delay_Intervals",
            "Post_Clean_Lag_Delay_Intervals",
            "Prediction_Delay_Intervals"
        ]
    ]
    .sort_values(
        "Prediction_Delay_Intervals",
        ascending=False
    )
    .head(20)
)

# ------------------------------------------------------------
# 11. Save results
# ------------------------------------------------------------
isolated_file = (
    DATA_DIR
    / "August_Isolated_Real_Failure_Recovery_Events.csv"
)

isolated.to_csv(
    isolated_file,
    index=False
)

classification_file = (
    DATA_DIR
    / "August_Failure_Event_Classification.csv"
)

diag.to_csv(
    classification_file,
    index=False
)

comparison_file = (
    TABLES_DIR
    / "Table_Isolated_vs_Recurrent_Failures.csv"
)

comparison.to_csv(
    comparison_file,
    index=False
)

print("\n" + "=" * 110)
print("FILES SAVED")
print("=" * 110)

print(isolated_file)
print(classification_file)
print(comparison_file)

print("\n" + "=" * 110)
print("STEP 8 COMPLETE")
print("=" * 110)

In [ ]:
# ============================================================
# STEP 9 — FAILURE-TO-FORECAST BLACKOUT AMPLIFICATION
# Quantify how short real detector failures propagate through
# strict cleaning + lag-based forecasting requirements
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("=" * 110)
print("STEP 9 — FAILURE-TO-FORECAST BLACKOUT AMPLIFICATION")
print("=" * 110)

# ------------------------------------------------------------
# 1. Load isolated events
# ------------------------------------------------------------
isolated_file = (
    DATA_DIR
    / "August_Isolated_Real_Failure_Recovery_Events.csv"
)

iso = pd.read_csv(
    isolated_file,
    parse_dates=[
        "Failure_Start",
        "Failure_End",
        "First_Clean_Target",
        "First_All_Lags_Available",
        "First_Final_Prediction"
    ]
)

print(f"\nIsolated events loaded: {len(iso):,}")

# ------------------------------------------------------------
# 2. Calculate failure and blackout durations
# ------------------------------------------------------------

# Actual physical/data failure duration
iso["Failure_Duration_Hours"] = (
    iso["Failure_Run_Length"] * 15 / 60
)

# Total clean-data blackout:
# failure itself + delay until clean target returns
iso["Clean_Data_Blackout_Intervals"] = (
    iso["Failure_Run_Length"]
    + iso["Clean_Target_Delay_Intervals"]
)

iso["Clean_Data_Blackout_Hours"] = (
    iso["Clean_Data_Blackout_Intervals"] * 15 / 60
)

# Total forecasting blackout:
# failure itself + delay until predictions become available
iso["Forecast_Blackout_Intervals"] = (
    iso["Failure_Run_Length"]
    + iso["Prediction_Delay_Intervals"]
)

iso["Forecast_Blackout_Hours"] = (
    iso["Forecast_Blackout_Intervals"] * 15 / 60
)

# Extra blackout introduced after the actual failure ends
iso["Pipeline_Extension_Hours"] = (
    iso["Prediction_Delay_Intervals"] * 15 / 60
)

# Amplification ratio
iso["Blackout_Amplification_Factor"] = (
    iso["Forecast_Blackout_Intervals"]
    / iso["Failure_Run_Length"]
)

# ------------------------------------------------------------
# 3. Canonical mechanism subset
#    Exactly 96 intervals added after clean-data return
# ------------------------------------------------------------
canonical = iso[
    iso["Post_Clean_Lag_Delay_Intervals"] == 96
].copy()

print("\n" + "=" * 110)
print("CANONICAL LAG-96 EVENTS")
print("=" * 110)

print(
    f"\nCanonical events: {len(canonical):,} / {len(iso):,} "
    f"({100*len(canonical)/len(iso):.2f}%)"
)

# ------------------------------------------------------------
# 4. Main descriptive results
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("BLACKOUT AMPLIFICATION — ALL ISOLATED EVENTS")
print("=" * 110)

metrics = [
    ("Failure_Duration_Hours", "Actual failure duration (hours)"),
    ("Clean_Data_Blackout_Hours", "Clean-data blackout (hours)"),
    ("Pipeline_Extension_Hours", "Post-failure pipeline extension (hours)"),
    ("Forecast_Blackout_Hours", "Total forecast blackout (hours)"),
    ("Blackout_Amplification_Factor", "Blackout amplification factor")
]

for col, label in metrics:

    values = iso[col].dropna()

    print(f"\n{label}")
    print("-" * 70)

    print(
        values.describe(
            percentiles=[
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99
            ]
        )
    )

# ------------------------------------------------------------
# 5. Headline median values
# ------------------------------------------------------------
median_failure = iso["Failure_Duration_Hours"].median()
median_blackout = iso["Forecast_Blackout_Hours"].median()
median_amplification = iso["Blackout_Amplification_Factor"].median()

print("\n" + "=" * 110)
print("HEADLINE RESULTS")
print("=" * 110)

print(
    f"\nMedian actual failure duration : "
    f"{median_failure:.2f} hours"
)

print(
    f"Median forecast blackout       : "
    f"{median_blackout:.2f} hours"
)

print(
    f"Median blackout amplification  : "
    f"{median_amplification:.2f}×"
)

# ------------------------------------------------------------
# 6. Special focus on very short failures
# ------------------------------------------------------------
short = iso[
    iso["Failure_Run_Length"] <= 4
].copy()

print("\n" + "=" * 110)
print("SHORT FAILURES — ≤ 1 HOUR")
print("=" * 110)

print(f"\nEvents: {len(short):,}")

if len(short) > 0:

    print(
        f"Median actual failure duration : "
        f"{short['Failure_Duration_Hours'].median():.2f} h"
    )

    print(
        f"Median forecast blackout       : "
        f"{short['Forecast_Blackout_Hours'].median():.2f} h"
    )

    print(
        f"Median amplification factor    : "
        f"{short['Blackout_Amplification_Factor'].median():.2f}×"
    )

# ------------------------------------------------------------
# 7. Failure-duration categories
# ------------------------------------------------------------
bins = [0, 1, 4, 8, 24, 48, 96, np.inf]

labels = [
    "15 min",
    "30–60 min",
    "1.25–2 h",
    "2.25–6 h",
    "6.25–12 h",
    "12.25–24 h",
    ">24 h"
]

iso["Failure_Duration_Group"] = pd.cut(
    iso["Failure_Run_Length"],
    bins=bins,
    labels=labels,
    right=True
)

group_summary = (
    iso.groupby(
        "Failure_Duration_Group",
        observed=True
    )
    .agg(
        Events=("Failure_Run_Length", "size"),
        Median_Failure_Hours=(
            "Failure_Duration_Hours",
            "median"
        ),
        Median_Clean_Blackout_Hours=(
            "Clean_Data_Blackout_Hours",
            "median"
        ),
        Median_Forecast_Blackout_Hours=(
            "Forecast_Blackout_Hours",
            "median"
        ),
        Median_Amplification=(
            "Blackout_Amplification_Factor",
            "median"
        )
    )
    .reset_index()
)

print("\n" + "=" * 110)
print("BLACKOUT BY FAILURE DURATION GROUP")
print("=" * 110)

display(group_summary)

# ------------------------------------------------------------
# 8. Correlation check
# ------------------------------------------------------------
corr = iso[
    "Failure_Duration_Hours"
].corr(
    iso["Forecast_Blackout_Hours"]
)

print("\n" + "=" * 110)
print("FAILURE DURATION VS TOTAL FORECAST BLACKOUT")
print("=" * 110)

print(f"\nPearson correlation: {corr:.4f}")

# ------------------------------------------------------------
# 9. Publication-ready scatter figure
# ------------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(8.2, 6.2)
)

ax.scatter(
    iso["Failure_Duration_Hours"],
    iso["Forecast_Blackout_Hours"],
    alpha=0.65,
    s=42,
    edgecolors="none"
)

# Equality reference line
max_value = max(
    iso["Failure_Duration_Hours"].max(),
    iso["Forecast_Blackout_Hours"].max()
)

ax.plot(
    [0, max_value],
    [0, max_value],
    linestyle="--",
    linewidth=1.2,
    label="No propagation (blackout = failure duration)"
)

ax.set_xlabel(
    "Observed Real Failure Duration (hours)",
    fontsize=12
)

ax.set_ylabel(
    "Total Forecast Unavailability (hours)",
    fontsize=12
)

ax.set_title(
    "Propagation of Real Detector Failures into Forecast Unavailability",
    fontsize=13,
    pad=12
)

ax.grid(
    True,
    alpha=0.25
)

ax.legend(
    frameon=False,
    fontsize=9
)

ax.tick_params(
    axis="both",
    labelsize=10
)

fig.tight_layout()

png_file = (
    FIGURES_DIR
    / "Figure_Real_Failure_Forecast_Blackout_Amplification.png"
)

pdf_file = (
    FIGURES_DIR
    / "Figure_Real_Failure_Forecast_Blackout_Amplification.pdf"
)

fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    pdf_file,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 10. Save publication-ready tables
# ------------------------------------------------------------
event_output = (
    DATA_DIR
    / "August_Isolated_Failure_Blackout_Amplification.csv"
)

iso.to_csv(
    event_output,
    index=False
)

group_output = (
    TABLES_DIR
    / "Table_Blackout_by_Failure_Duration.csv"
)

group_summary.to_csv(
    group_output,
    index=False
)

headline_summary = pd.DataFrame({
    "Metric": [
        "Isolated_Failure_Events",
        "Canonical_Lag96_Events",
        "Median_Actual_Failure_Hours",
        "Median_Clean_Data_Blackout_Hours",
        "Median_Total_Forecast_Blackout_Hours",
        "Median_Blackout_Amplification_Factor",
        "Failure_vs_Blackout_Correlation"
    ],
    "Value": [
        len(iso),
        len(canonical),
        iso["Failure_Duration_Hours"].median(),
        iso["Clean_Data_Blackout_Hours"].median(),
        iso["Forecast_Blackout_Hours"].median(),
        iso["Blackout_Amplification_Factor"].median(),
        corr
    ]
})

headline_output = (
    TABLES_DIR
    / "Table_Failure_Blackout_Headline_Results.csv"
)

headline_summary.to_csv(
    headline_output,
    index=False
)

print("\n" + "=" * 110)
print("FILES SAVED")
print("=" * 110)

print(event_output)
print(group_output)
print(headline_output)
print(png_file)
print(pdf_file)

print("\n" + "=" * 110)
print("STEP 9 COMPLETE")
print("=" * 110)

In [ ]:
# ============================================================
# STEP 10 — MODEL REPRODUCIBILITY AND FEATURE AUDIT
# Confirm that Notebook 4 can reproduce Notebook 3 predictions
# before controlled failure injection
# ============================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path

import lightgbm as lgb
import xgboost as xgb

print("=" * 110)
print("STEP 10 — MODEL REPRODUCIBILITY AND FEATURE AUDIT")
print("=" * 110)

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------
lgb_model_file = (
    NOTEBOOK3_DIR
    / "Models"
    / "LightGBM_FINAL_JanJul_1200.txt"
)

xgb_model_file = (
    NOTEBOOK3_DIR
    / "Models"
    / "XGBoost_FINAL_JanJul_624.json"
)

feature_config_file = (
    NOTEBOOK3_DIR
    / "Results"
    / "Forecasting_Feature_Configuration.json"
)

forecast_file = (
    NOTEBOOK3_DIR
    / "Data"
    / "Forecasting_Monthly"
    / "FORECASTING_2026-08.parquet"
)

prediction_file = (
    NOTEBOOK3_DIR
    / "Data"
    / "August_Final_All_Model_Predictions.parquet"
)

# ------------------------------------------------------------
# 2. Load feature configuration
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("NOTEBOOK 3 FEATURE CONFIGURATION")
print("=" * 110)

if feature_config_file.exists():

    with open(feature_config_file, "r") as f:
        feature_config = json.load(f)

    print(json.dumps(
        feature_config,
        indent=2
    ))

else:
    print("⚠️ Feature configuration JSON not found.")
    feature_config = {}

# ------------------------------------------------------------
# 3. Load trained models
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("LOADING FINAL MODELS")
print("=" * 110)

lgb_model = lgb.Booster(
    model_file=str(lgb_model_file)
)

xgb_model = xgb.Booster()
xgb_model.load_model(
    str(xgb_model_file)
)

print("✓ LightGBM final model loaded")
print("✓ XGBoost final model loaded")

# ------------------------------------------------------------
# 4. Inspect exact model feature names
# ------------------------------------------------------------
lgb_features = lgb_model.feature_name()

xgb_features = xgb_model.feature_names

print("\nLightGBM model features:")
for i, f in enumerate(lgb_features, start=1):
    print(f"  {i:02d}. {f}")

print("\nXGBoost model features:")

if xgb_features is not None:
    for i, f in enumerate(xgb_features, start=1):
        print(f"  {i:02d}. {f}")
else:
    print("  ⚠️ XGBoost model does not contain saved feature names.")

# ------------------------------------------------------------
# 5. Determine required forecasting columns
# ------------------------------------------------------------
base_cols = [
    "Timestamp",
    "Detector_Code",
    "All_Lags_Available"
]

required_model_features = list(
    dict.fromkeys(
        lgb_features
        + (
            xgb_features
            if xgb_features is not None
            else []
        )
    )
)

forecast_columns = list(
    dict.fromkeys(
        base_cols
        + required_model_features
    )
)

print("\n" + "=" * 110)
print("REQUIRED FORECAST DATA COLUMNS")
print("=" * 110)

for c in forecast_columns:
    print(f"  • {c}")

# ------------------------------------------------------------
# 6. Load only required August columns
# ------------------------------------------------------------
forecast = pd.read_parquet(
    forecast_file,
    columns=forecast_columns
)

forecast["Timestamp"] = pd.to_datetime(
    forecast["Timestamp"]
)

print("\n" + "=" * 110)
print("AUGUST FORECASTING AVAILABILITY")
print("=" * 110)

print(f"\nTotal forecasting rows : {len(forecast):,}")

print("\nAll_Lags_Available:")
print(
    forecast["All_Lags_Available"]
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------
# 7. Load saved final predictions
# ------------------------------------------------------------
saved_pred = pd.read_parquet(
    prediction_file,
    columns=[
        "Timestamp",
        "Detector_Code",
        "LightGBM_Prediction",
        "XGBoost_Prediction"
    ]
)

saved_pred["Timestamp"] = pd.to_datetime(
    saved_pred["Timestamp"]
)

print(f"\nSaved final prediction rows: {len(saved_pred):,}")

# ------------------------------------------------------------
# 8. Compare prediction keys with all-lag-ready rows
# ------------------------------------------------------------
ready = forecast[
    forecast["All_Lags_Available"] == True
].copy()

print(f"All-lag-ready rows       : {len(ready):,}")

key_check = saved_pred[
    ["Timestamp", "Detector_Code"]
].merge(
    ready[
        ["Timestamp", "Detector_Code"]
    ],
    on=[
        "Timestamp",
        "Detector_Code"
    ],
    how="outer",
    indicator=True
)

print("\nPrediction-key consistency:")

print(
    key_check["_merge"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 9. Build reproducibility sample
# ------------------------------------------------------------
SAMPLE_SIZE = min(
    100_000,
    len(saved_pred)
)

sample_keys = (
    saved_pred
    .sample(
        n=SAMPLE_SIZE,
        random_state=42
    )
    [
        [
            "Timestamp",
            "Detector_Code",
            "LightGBM_Prediction",
            "XGBoost_Prediction"
        ]
    ]
)

sample = sample_keys.merge(
    ready,
    on=[
        "Timestamp",
        "Detector_Code"
    ],
    how="inner"
)

print("\n" + "=" * 110)
print("REPRODUCIBILITY SAMPLE")
print("=" * 110)

print(f"\nRequested sample rows : {SAMPLE_SIZE:,}")
print(f"Matched sample rows   : {len(sample):,}")

# ------------------------------------------------------------
# 10. Reproduce LightGBM predictions
# ------------------------------------------------------------
X_lgb = sample[
    lgb_features
].copy()

lgb_reproduced = lgb_model.predict(
    X_lgb
)

sample[
    "LightGBM_Reproduced"
] = lgb_reproduced

sample[
    "LightGBM_Reproduction_Difference"
] = (
    sample["LightGBM_Reproduced"]
    - sample["LightGBM_Prediction"]
)

# ------------------------------------------------------------
# 11. Reproduce XGBoost predictions
# ------------------------------------------------------------
if xgb_features is not None:

    X_xgb = sample[
        xgb_features
    ].copy()

    dmatrix = xgb.DMatrix(
        X_xgb,
        feature_names=xgb_features
    )

else:

    # Fallback only if feature names were not saved
    X_xgb = sample[
        lgb_features
    ].copy()

    dmatrix = xgb.DMatrix(
        X_xgb
    )

xgb_reproduced = xgb_model.predict(
    dmatrix
)

sample[
    "XGBoost_Reproduced"
] = xgb_reproduced

sample[
    "XGBoost_Reproduction_Difference"
] = (
    sample["XGBoost_Reproduced"]
    - sample["XGBoost_Prediction"]
)

# ------------------------------------------------------------
# 12. Reproducibility statistics
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("LIGHTGBM REPRODUCIBILITY")
print("=" * 110)

lgb_abs_diff = (
    sample[
        "LightGBM_Reproduction_Difference"
    ]
    .abs()
)

print(
    f"\nMean absolute reproduction difference : "
    f"{lgb_abs_diff.mean():.12f}"
)

print(
    f"Maximum absolute difference           : "
    f"{lgb_abs_diff.max():.12f}"
)

print(
    f"Rows within 1e-6                      : "
    f"{(lgb_abs_diff <= 1e-6).sum():,} / {len(sample):,}"
)

print("\n" + "=" * 110)
print("XGBOOST REPRODUCIBILITY")
print("=" * 110)

xgb_abs_diff = (
    sample[
        "XGBoost_Reproduction_Difference"
    ]
    .abs()
)

print(
    f"\nMean absolute reproduction difference : "
    f"{xgb_abs_diff.mean():.12f}"
)

print(
    f"Maximum absolute difference           : "
    f"{xgb_abs_diff.max():.12f}"
)

print(
    f"Rows within 1e-6                      : "
    f"{(xgb_abs_diff <= 1e-6).sum():,} / {len(sample):,}"
)

# ------------------------------------------------------------
# 13. Controlled NaN capability smoke test
#     This does NOT form part of the experiment yet.
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("MISSING-LAG SCORING CAPABILITY — SMOKE TEST")
print("=" * 110)

lag_features = [
    f for f in [
        "Lag_1",
        "Lag_2",
        "Lag_4",
        "Lag_8",
        "Lag_96"
    ]
    if f in lgb_features
]

test_row = sample.iloc[[0]].copy()

print("\nLag features available for stress testing:")
print(lag_features)

if len(lag_features) > 0:

    # Set Lag_1 missing as a harmless scoring test
    test_row.loc[
        test_row.index,
        lag_features[0]
    ] = np.nan

    # LightGBM
    lgb_nan_test = lgb_model.predict(
        test_row[lgb_features]
    )[0]

    print(
        f"\nLightGBM prediction with "
        f"{lag_features[0]} = NaN: "
        f"{lgb_nan_test:.6f}"
    )

    # XGBoost
    if xgb_features is not None:

        xgb_nan_matrix = xgb.DMatrix(
            test_row[xgb_features],
            feature_names=xgb_features
        )

    else:

        xgb_nan_matrix = xgb.DMatrix(
            test_row[lgb_features]
        )

    xgb_nan_test = xgb_model.predict(
        xgb_nan_matrix
    )[0]

    print(
        f"XGBoost prediction with "
        f"{lag_features[0]} = NaN: "
        f"{xgb_nan_test:.6f}"
    )

    print("\n✓ Both models successfully scored a row containing a missing lag.")

# ------------------------------------------------------------
# 14. Save reproducibility audit
# ------------------------------------------------------------
audit = pd.DataFrame({
    "Metric": [
        "Total_August_Forecasting_Rows",
        "All_Lags_Available_Rows",
        "Final_Prediction_Rows",
        "Reproducibility_Sample_Rows",
        "LightGBM_Mean_Abs_Reproduction_Diff",
        "LightGBM_Max_Abs_Reproduction_Diff",
        "XGBoost_Mean_Abs_Reproduction_Diff",
        "XGBoost_Max_Abs_Reproduction_Diff"
    ],
    "Value": [
        len(forecast),
        len(ready),
        len(saved_pred),
        len(sample),
        lgb_abs_diff.mean(),
        lgb_abs_diff.max(),
        xgb_abs_diff.mean(),
        xgb_abs_diff.max()
    ]
})

audit_file = (
    TABLES_DIR
    / "Table_Model_Reproducibility_Audit.csv"
)

audit.to_csv(
    audit_file,
    index=False
)

print("\n" + "=" * 110)
print("REPRODUCIBILITY AUDIT")
print("=" * 110)

display(audit)

print("\nSaved:")
print(audit_file)

print("\n" + "=" * 110)
print("STEP 10 COMPLETE")
print("=" * 110)

In [ ]:
# ============================================================
# STEP 10A — REPRODUCTION DIFFERENCE DIAGNOSTIC
# Determine whether Notebook 3 saved predictions were clipped
# (e.g., negative predictions forced to zero)
# ============================================================

import pandas as pd
import numpy as np

print("=" * 110)
print("STEP 10A — REPRODUCTION DIFFERENCE DIAGNOSTIC")
print("=" * 110)

# ------------------------------------------------------------
# 1. Absolute differences
# ------------------------------------------------------------
sample["LGB_Abs_Diff_Raw"] = (
    sample["LightGBM_Reproduced"]
    - sample["LightGBM_Prediction"]
).abs()

sample["XGB_Abs_Diff_Raw"] = (
    sample["XGBoost_Reproduced"]
    - sample["XGBoost_Prediction"]
).abs()

# ------------------------------------------------------------
# 2. Difference distributions
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("RAW DIFFERENCE DISTRIBUTIONS")
print("=" * 110)

for col, label in [
    ("LGB_Abs_Diff_Raw", "LightGBM"),
    ("XGB_Abs_Diff_Raw", "XGBoost")
]:
    print(f"\n{label}")
    print("-" * 60)

    print(
        sample[col].describe(
            percentiles=[
                0.50,
                0.90,
                0.95,
                0.99,
                0.999
            ]
        )
    )

    print(
        f"Rows > 1e-6 : "
        f"{(sample[col] > 1e-6).sum():,}"
    )

    print(
        f"Rows > 0.01 : "
        f"{(sample[col] > 0.01).sum():,}"
    )

    print(
        f"Rows > 0.1  : "
        f"{(sample[col] > 0.1).sum():,}"
    )

    print(
        f"Rows > 1.0  : "
        f"{(sample[col] > 1.0).sum():,}"
    )

# ------------------------------------------------------------
# 3. Check negative raw model outputs
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("NEGATIVE RAW MODEL OUTPUTS")
print("=" * 110)

lgb_negative = (
    sample["LightGBM_Reproduced"] < 0
)

xgb_negative = (
    sample["XGBoost_Reproduced"] < 0
)

print(
    f"\nLightGBM negative raw predictions: "
    f"{lgb_negative.sum():,}"
)

print(
    f"XGBoost negative raw predictions : "
    f"{xgb_negative.sum():,}"
)

# Among those, check whether Notebook 3 saved zero
if lgb_negative.sum() > 0:

    lgb_saved_zero_when_negative = (
        sample.loc[
            lgb_negative,
            "LightGBM_Prediction"
        ].abs() <= 1e-12
    ).sum()

    print(
        f"\nLightGBM negative raw → saved as zero: "
        f"{lgb_saved_zero_when_negative:,} / "
        f"{lgb_negative.sum():,}"
    )

if xgb_negative.sum() > 0:

    xgb_saved_zero_when_negative = (
        sample.loc[
            xgb_negative,
            "XGBoost_Prediction"
        ].abs() <= 1e-12
    ).sum()

    print(
        f"XGBoost negative raw → saved as zero: "
        f"{xgb_saved_zero_when_negative:,} / "
        f"{xgb_negative.sum():,}"
    )

# ------------------------------------------------------------
# 4. Apply zero clipping and compare again
# ------------------------------------------------------------
sample["LightGBM_Reproduced_Clipped"] = np.maximum(
    sample["LightGBM_Reproduced"],
    0
)

sample["XGBoost_Reproduced_Clipped"] = np.maximum(
    sample["XGBoost_Reproduced"],
    0
)

sample["LGB_Abs_Diff_Clipped"] = (
    sample["LightGBM_Reproduced_Clipped"]
    - sample["LightGBM_Prediction"]
).abs()

sample["XGB_Abs_Diff_Clipped"] = (
    sample["XGBoost_Reproduced_Clipped"]
    - sample["XGBoost_Prediction"]
).abs()

print("\n" + "=" * 110)
print("AFTER ZERO-CLIPPING REPRODUCED PREDICTIONS")
print("=" * 110)

for col, label in [
    ("LGB_Abs_Diff_Clipped", "LightGBM"),
    ("XGB_Abs_Diff_Clipped", "XGBoost")
]:
    print(f"\n{label}")
    print("-" * 60)

    print(
        f"Mean absolute difference : "
        f"{sample[col].mean():.12f}"
    )

    print(
        f"Maximum absolute difference: "
        f"{sample[col].max():.12f}"
    )

    print(
        f"Rows within 1e-6: "
        f"{(sample[col] <= 1e-6).sum():,} / "
        f"{len(sample):,}"
    )

    print(
        f"Rows within 1e-5: "
        f"{(sample[col] <= 1e-5).sum():,} / "
        f"{len(sample):,}"
    )

    print(
        f"Rows within 1e-4: "
        f"{(sample[col] <= 1e-4).sum():,} / "
        f"{len(sample):,}"
    )

# ------------------------------------------------------------
# 5. Inspect largest mismatches after clipping
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("LARGEST LIGHTGBM MISMATCHES AFTER CLIPPING")
print("=" * 110)

display(
    sample.nlargest(
        15,
        "LGB_Abs_Diff_Clipped"
    )[
        [
            "Timestamp",
            "Detector_Code",
            "LightGBM_Prediction",
            "LightGBM_Reproduced",
            "LightGBM_Reproduced_Clipped",
            "LGB_Abs_Diff_Clipped"
        ]
    ]
)

print("\n" + "=" * 110)
print("LARGEST XGBOOST MISMATCHES AFTER CLIPPING")
print("=" * 110)

display(
    sample.nlargest(
        15,
        "XGB_Abs_Diff_Clipped"
    )[
        [
            "Timestamp",
            "Detector_Code",
            "XGBoost_Prediction",
            "XGBoost_Reproduced",
            "XGBoost_Reproduced_Clipped",
            "XGB_Abs_Diff_Clipped"
        ]
    ]
)

# ------------------------------------------------------------
# 6. Check numerical rounding hypotheses
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("NUMERICAL PRECISION CHECK")
print("=" * 110)

for decimals in [6, 5, 4, 3]:

    lgb_match = np.isclose(
        sample["LightGBM_Reproduced_Clipped"].round(decimals),
        sample["LightGBM_Prediction"].round(decimals),
        atol=10**(-decimals),
        rtol=0
    ).mean()

    xgb_match = np.isclose(
        sample["XGBoost_Reproduced_Clipped"].round(decimals),
        sample["XGBoost_Prediction"].round(decimals),
        atol=10**(-decimals),
        rtol=0
    ).mean()

    print(
        f"\nRounded to {decimals} decimals:"
    )

    print(
        f"  LightGBM match: "
        f"{100*lgb_match:.4f}%"
    )

    print(
        f"  XGBoost match : "
        f"{100*xgb_match:.4f}%"
    )

# ------------------------------------------------------------
# 7. Create concise audit summary
# ------------------------------------------------------------
diagnostic_summary = pd.DataFrame({

    "Metric": [
        "LGB_Negative_Raw_Predictions",
        "XGB_Negative_Raw_Predictions",

        "LGB_Raw_Mean_Abs_Diff",
        "LGB_Clipped_Mean_Abs_Diff",
        "LGB_Clipped_Max_Abs_Diff",

        "XGB_Raw_Mean_Abs_Diff",
        "XGB_Clipped_Mean_Abs_Diff",
        "XGB_Clipped_Max_Abs_Diff",

        "LGB_Clipped_Within_1e6_Percent",
        "XGB_Clipped_Within_1e6_Percent"
    ],

    "Value": [
        int(lgb_negative.sum()),
        int(xgb_negative.sum()),

        sample["LGB_Abs_Diff_Raw"].mean(),
        sample["LGB_Abs_Diff_Clipped"].mean(),
        sample["LGB_Abs_Diff_Clipped"].max(),

        sample["XGB_Abs_Diff_Raw"].mean(),
        sample["XGB_Abs_Diff_Clipped"].mean(),
        sample["XGB_Abs_Diff_Clipped"].max(),

        100 * (
            sample["LGB_Abs_Diff_Clipped"] <= 1e-6
        ).mean(),

        100 * (
            sample["XGB_Abs_Diff_Clipped"] <= 1e-6
        ).mean()
    ]
})

print("\n" + "=" * 110)
print("DIAGNOSTIC SUMMARY")
print("=" * 110)

display(diagnostic_summary)

# ------------------------------------------------------------
# 8. Save only the important audit result
# ------------------------------------------------------------
diagnostic_file = (
    TABLES_DIR
    / "Table_Model_Reproduction_Discrepancy_Diagnostic.csv"
)

diagnostic_summary.to_csv(
    diagnostic_file,
    index=False
)

print("\nSaved:")
print(diagnostic_file)

print("\n" + "=" * 110)
print("STEP 10A COMPLETE")
print("=" * 110)

In [ ]:
# ============================================================
# STEP 11 — PRE-TEST EMPIRICAL FAILURE PATTERN LIBRARY
# Build real failure patterns from Jan–Jul only
# for controlled injection into the held-out August test set
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import ks_2samp

print("=" * 110)
print("STEP 11 — PRE-TEST EMPIRICAL FAILURE PATTERN LIBRARY")
print("=" * 110)

# ------------------------------------------------------------
# 1. Load full real-failure library + detector mapping
# ------------------------------------------------------------
failure_file = (
    NOTEBOOK2_DIR
    / "REAL_FAILURE_CONTIGUOUS_RUN_LIBRARY.csv"
)

mapping_file = (
    NOTEBOOK2_DIR
    / "DETECTOR_CODE_MAPPING_800.csv"
)

failures_all = pd.read_csv(
    failure_file,
    parse_dates=["Date"]
)

mapping = pd.read_csv(
    mapping_file
)

print(f"\nFull real-failure runs loaded : {len(failures_all):,}")
print(f"800-detector mapping rows     : {len(mapping):,}")

# ------------------------------------------------------------
# 2. Restrict to the exact 800-detector modeling cohort
# ------------------------------------------------------------
mapping_small = mapping[
    [
        "NB_SCATS_SITE",
        "NB_DETECTOR",
        "Detector_Code",
        "Sensitivity_Cohort"
    ]
].copy()

failure_800 = failures_all.merge(
    mapping_small,
    on=[
        "NB_SCATS_SITE",
        "NB_DETECTOR"
    ],
    how="inner"
)

print(
    f"Failure runs belonging to modeling cohort: "
    f"{len(failure_800):,}"
)

print(
    f"Unique affected modeling detectors       : "
    f"{failure_800['Detector_Code'].nunique():,}"
)

# ------------------------------------------------------------
# 3. Validate failure geometry
# ------------------------------------------------------------
expected_length = (
    failure_800["End_Interval_Index"]
    - failure_800["Start_Interval_Index"]
    + 1
)

invalid_start = (
    ~failure_800["Start_Interval_Index"]
    .between(0, 95)
).sum()

invalid_end = (
    ~failure_800["End_Interval_Index"]
    .between(0, 95)
).sum()

length_mismatch = (
    expected_length
    != failure_800["Run_Length"]
).sum()

print("\n" + "=" * 110)
print("PATTERN VALIDITY")
print("=" * 110)

print(f"\nInvalid start indices : {invalid_start:,}")
print(f"Invalid end indices   : {invalid_end:,}")
print(f"Run-length mismatches : {length_mismatch:,}")

# ------------------------------------------------------------
# 4. Split pattern-development period from held-out August
# ------------------------------------------------------------
pretest = failure_800[
    failure_800["Date"].between(
        "2026-01-01",
        "2026-07-31"
    )
].copy()

august_observed = failure_800[
    failure_800["Date"].between(
        "2026-08-01",
        "2026-08-30"
    )
].copy()

print("\n" + "=" * 110)
print("TEMPORAL SPLIT")
print("=" * 110)

print(
    f"\nPre-test real failure runs (Jan–Jul): "
    f"{len(pretest):,}"
)

print(
    f"Pre-test affected detectors          : "
    f"{pretest['Detector_Code'].nunique():,}"
)

print(
    f"Held-out August observed runs        : "
    f"{len(august_observed):,}"
)

print(
    f"August affected detectors            : "
    f"{august_observed['Detector_Code'].nunique():,}"
)

# ------------------------------------------------------------
# 5. Add useful pattern descriptors
# ------------------------------------------------------------
def enrich_patterns(df):

    df = df.copy()

    df["Failure_Duration_Minutes"] = (
        df["Run_Length"] * 15
    )

    df["Failure_Duration_Hours"] = (
        df["Run_Length"] * 15 / 60
    )

    df["Start_Hour"] = (
        df["Start_Interval_Index"] // 4
    )

    df["Start_Minute"] = (
        (df["Start_Interval_Index"] % 4) * 15
    )

    conditions = [
        df["Run_Length"] == 1,
        df["Run_Length"].between(2, 4),
        df["Run_Length"].between(5, 8),
        df["Run_Length"].between(9, 24),
        df["Run_Length"].between(25, 48),
        df["Run_Length"].between(49, 95),
        df["Run_Length"] == 96
    ]

    labels = [
        "15 min",
        "30–60 min",
        "1.25–2 h",
        "2.25–6 h",
        "6.25–12 h",
        "12.25–23.75 h",
        "24 h"
    ]

    df["Failure_Duration_Group"] = np.select(
        conditions,
        labels,
        default="Other"
    )

    return df


pretest = enrich_patterns(pretest)
august_observed = enrich_patterns(august_observed)

# ------------------------------------------------------------
# 6. Descriptive statistics
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("PRE-TEST FAILURE RUN-LENGTH DISTRIBUTION")
print("=" * 110)

print(
    pretest["Run_Length"].describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

print("\nPre-test duration groups:")

pretest_group_counts = (
    pretest["Failure_Duration_Group"]
    .value_counts()
    .reindex(
        [
            "15 min",
            "30–60 min",
            "1.25–2 h",
            "2.25–6 h",
            "6.25–12 h",
            "12.25–23.75 h",
            "24 h"
        ],
        fill_value=0
    )
)

print(pretest_group_counts.to_string())

# ------------------------------------------------------------
# 7. Compare pre-test patterns with held-out August failures
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("PRE-TEST VS AUGUST FAILURE-PATTERN COMPARISON")
print("=" * 110)

comparison_rows = []

for label, df in [
    ("Jan–Jul Pattern Library", pretest),
    ("August Observed Failures", august_observed)
]:

    comparison_rows.append({
        "Period": label,
        "Failure_Runs": len(df),
        "Affected_Detectors":
            df["Detector_Code"].nunique(),
        "Mean_Run_Length":
            df["Run_Length"].mean(),
        "Median_Run_Length":
            df["Run_Length"].median(),
        "P75_Run_Length":
            df["Run_Length"].quantile(0.75),
        "P90_Run_Length":
            df["Run_Length"].quantile(0.90),
        "P95_Run_Length":
            df["Run_Length"].quantile(0.95),
        "P99_Run_Length":
            df["Run_Length"].quantile(0.99),
        "Maximum_Run_Length":
            df["Run_Length"].max(),
        "Percent_LE_1h":
            100 * (
                df["Run_Length"] <= 4
            ).mean(),
        "Percent_Full_Day":
            100 * (
                df["Run_Length"] == 96
            ).mean()
    })

comparison = pd.DataFrame(
    comparison_rows
)

display(comparison)

# ------------------------------------------------------------
# 8. Distributional similarity test
# ------------------------------------------------------------
ks = ks_2samp(
    pretest["Run_Length"],
    august_observed["Run_Length"],
    alternative="two-sided",
    mode="auto"
)

print("\nKolmogorov–Smirnov comparison:")
print(
    f"  KS statistic : {ks.statistic:.4f}"
)
print(
    f"  p-value      : {ks.pvalue:.6f}"
)

# ------------------------------------------------------------
# 9. Start-time distribution
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("PRE-TEST FAILURE START-TIME DISTRIBUTION")
print("=" * 110)

start_hour_summary = (
    pretest.groupby("Start_Hour")
    .size()
    .rename("Failure_Runs")
    .reset_index()
)

display(start_hour_summary)

# ------------------------------------------------------------
# 10. Detector-level failure-pattern availability
# ------------------------------------------------------------
detector_pattern_summary = (
    pretest.groupby("Detector_Code")
    .agg(
        PreTest_Failure_Runs=(
            "Run_Length",
            "size"
        ),
        Median_Run_Length=(
            "Run_Length",
            "median"
        ),
        Maximum_Run_Length=(
            "Run_Length",
            "max"
        ),
        Unique_Failure_Dates=(
            "Date",
            "nunique"
        )
    )
    .reset_index()
)

print("\n" + "=" * 110)
print("DETECTOR-LEVEL PATTERN AVAILABILITY")
print("=" * 110)

print(
    f"\nDetectors with ≥1 pre-test failure pattern : "
    f"{len(detector_pattern_summary):,}"
)

print(
    f"Detectors with ≥2 pre-test failure runs    : "
    f"{(detector_pattern_summary['PreTest_Failure_Runs'] >= 2).sum():,}"
)

print(
    f"Detectors with ≥5 pre-test failure runs    : "
    f"{(detector_pattern_summary['PreTest_Failure_Runs'] >= 5).sum():,}"
)

print(
    f"Detectors with ≥10 pre-test failure runs   : "
    f"{(detector_pattern_summary['PreTest_Failure_Runs'] >= 10).sum():,}"
)

# ------------------------------------------------------------
# 11. Save empirical pattern library
# ------------------------------------------------------------
pattern_columns = [
    "Date",
    "Detector_Code",
    "NB_SCATS_SITE",
    "NB_DETECTOR",
    "NM_REGION",
    "Start_Interval_Index",
    "End_Interval_Index",
    "Start_Interval",
    "End_Interval",
    "Run_Length",
    "Failure_Duration_Minutes",
    "Failure_Duration_Hours",
    "Failure_Duration_Group",
    "Start_Hour",
    "Start_Minute",
    "CT_ALARM_24HOUR",
    "Volume_Group",
    "Alarm_Exposure_Group",
    "Sensitivity_Cohort"
]

pattern_columns = [
    c for c in pattern_columns
    if c in pretest.columns
]

pattern_file = (
    DATA_DIR
    / "PreTest_Real_Failure_Pattern_Library_JanJul.csv"
)

pretest[
    pattern_columns
].to_csv(
    pattern_file,
    index=False
)

comparison_file = (
    TABLES_DIR
    / "Table_PreTest_vs_August_Failure_Patterns.csv"
)

comparison.to_csv(
    comparison_file,
    index=False
)

detector_summary_file = (
    TABLES_DIR
    / "Table_PreTest_Failure_Patterns_by_Detector.csv"
)

detector_pattern_summary.to_csv(
    detector_summary_file,
    index=False
)

# ------------------------------------------------------------
# 12. Save experiment provenance
# ------------------------------------------------------------
import json

provenance = {
    "controlled_failure_injection_design": {
        "failure_pattern_source":
            "Observed real SCATS detector failure runs",
        "pattern_development_period":
            "2026-01-01 to 2026-07-31",
        "held_out_test_period":
            "2026-08-01 to 2026-08-30",
        "time_resolution_minutes":
            15,
        "pattern_variables":
            [
                "Detector_Code",
                "Start_Interval_Index",
                "Run_Length"
            ],
        "purpose":
            "Empirically grounded controlled robustness experiment"
    }
}

provenance_file = (
    RESULTS_DIR
    / "Controlled_Failure_Injection_Design.json"
)

with open(
    provenance_file,
    "w"
) as f:
    json.dump(
        provenance,
        f,
        indent=2
    )

print("\n" + "=" * 110)
print("FILES SAVED")
print("=" * 110)

print(pattern_file)
print(comparison_file)
print(detector_summary_file)
print(provenance_file)

print("\n" + "=" * 110)
print("STEP 11 COMPLETE")
print("=" * 110)

In [ ]:
# ============================================================
# STEP 12 — CONTROLLED FAILURE-INJECTION EVENT DESIGN
#
# Select isolated August injection events using real Jan–Jul
# failure patterns while preserving:
#   - same detector
#   - real failure run length
#   - real start time
#   - same day of week
#
# Every selected August window must have complete baseline
# predictions before any artificial failure is introduced.
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import ks_2samp
import json

print("=" * 112)
print("STEP 12 — CONTROLLED FAILURE-INJECTION EVENT DESIGN")
print("=" * 112)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# ------------------------------------------------------------
# 1. Load pre-test empirical failure patterns
# ------------------------------------------------------------
pattern_file = (
    DATA_DIR
    / "PreTest_Real_Failure_Pattern_Library_JanJul.csv"
)

patterns = pd.read_csv(
    pattern_file,
    parse_dates=["Date"]
)

patterns["Source_DayOfWeek"] = (
    patterns["Date"].dt.dayofweek
)

print(f"\nPre-test failure patterns: {len(patterns):,}")
print(
    f"Unique source detectors : "
    f"{patterns['Detector_Code'].nunique():,}"
)

# ------------------------------------------------------------
# 2. Load baseline August prediction keys
# ------------------------------------------------------------
prediction_file = (
    NOTEBOOK3_DIR
    / "Data"
    / "August_Final_All_Model_Predictions.parquet"
)

baseline_keys = pd.read_parquet(
    prediction_file,
    columns=[
        "Timestamp",
        "Detector_Code"
    ]
)

baseline_keys["Timestamp"] = pd.to_datetime(
    baseline_keys["Timestamp"]
)

baseline_keys = baseline_keys.sort_values(
    ["Detector_Code", "Timestamp"]
)

print(
    f"Baseline August prediction rows: "
    f"{len(baseline_keys):,}"
)

# ------------------------------------------------------------
# 3. Create fast timestamp lookup by detector
# ------------------------------------------------------------
prediction_times = {}

for detector, group in baseline_keys.groupby(
    "Detector_Code",
    sort=False
):

    prediction_times[int(detector)] = (
        group["Timestamp"]
        .to_numpy(
            dtype="datetime64[ns]"
        )
    )

print(
    f"Detectors represented in baseline predictions: "
    f"{len(prediction_times):,}"
)

# ------------------------------------------------------------
# 4. Candidate August dates
# ------------------------------------------------------------
august_dates = pd.date_range(
    "2026-08-01",
    "2026-08-30",
    freq="D"
)

august_calendar = pd.DataFrame({
    "Target_Date": august_dates
})

august_calendar["DayOfWeek"] = (
    august_calendar["Target_Date"]
    .dt.dayofweek
)

test_end = pd.Timestamp(
    "2026-08-30 23:45:00"
)

# ------------------------------------------------------------
# 5. Helper: complete baseline coverage check
# ------------------------------------------------------------
def has_complete_baseline_window(
    detector,
    window_start,
    window_end
):

    det_times = prediction_times.get(
        int(detector)
    )

    if det_times is None:
        return False

    expected_times = pd.date_range(
        window_start,
        window_end,
        freq="15min"
    ).to_numpy(
        dtype="datetime64[ns]"
    )

    # Window must be completely inside held-out test period
    if (
        window_start
        < pd.Timestamp("2026-08-01 00:00:00")
        or
        window_end
        > test_end
    ):
        return False

    # Search exact timestamps efficiently
    positions = np.searchsorted(
        det_times,
        expected_times
    )

    valid_positions = (
        positions < len(det_times)
    )

    if not valid_positions.all():
        return False

    return np.array_equal(
        det_times[
            positions
        ],
        expected_times
    )

# ------------------------------------------------------------
# 6. Select ONE isolated injection event per detector
#
# For each detector:
# - randomly order its real Jan–Jul patterns
# - preserve source day-of-week
# - randomly try clean August dates
# - require complete baseline predictions from:
#
# failure start → failure end + 24 hours
#
# This covers all possible Lag_1/2/4/8/96 effects.
# ------------------------------------------------------------
selected_events = []
unselected_detectors = []

detector_ids = (
    patterns["Detector_Code"]
    .drop_duplicates()
    .astype(int)
    .to_numpy()
)

rng.shuffle(detector_ids)

for detector in detector_ids:

    detector_patterns = (
        patterns[
            patterns["Detector_Code"]
            == detector
        ]
        .copy()
        .reset_index(drop=True)
    )

    pattern_order = rng.permutation(
        len(detector_patterns)
    )

    event_found = False

    for p_idx in pattern_order:

        pattern = detector_patterns.iloc[
            int(p_idx)
        ]

        source_dow = int(
            pattern["Source_DayOfWeek"]
        )

        # Preserve exact day-of-week
        candidate_dates = (
            august_calendar.loc[
                august_calendar["DayOfWeek"]
                == source_dow,
                "Target_Date"
            ]
            .to_numpy()
            .copy()
        )

        rng.shuffle(candidate_dates)

        for target_date_np in candidate_dates:

            target_date = pd.Timestamp(
                target_date_np
            )

            start_idx = int(
                pattern[
                    "Start_Interval_Index"
                ]
            )

            end_idx = int(
                pattern[
                    "End_Interval_Index"
                ]
            )

            run_length = int(
                pattern["Run_Length"]
            )

            failure_start = (
                target_date
                + pd.Timedelta(
                    minutes=15 * start_idx
                )
            )

            failure_end = (
                target_date
                + pd.Timedelta(
                    minutes=15 * end_idx
                )
            )

            # Last target potentially affected by Lag_96
            evaluation_end = (
                failure_end
                + pd.Timedelta(hours=24)
            )

            # Need complete clean baseline predictions
            # throughout the entire affected window
            valid_window = (
                has_complete_baseline_window(
                    detector=detector,
                    window_start=failure_start,
                    window_end=evaluation_end
                )
            )

            if not valid_window:
                continue

            evaluation_rows = int(
                (
                    evaluation_end
                    - failure_start
                ).total_seconds()
                / (15 * 60)
            ) + 1

            selected_events.append({

                "Injection_Event_ID":
                    len(selected_events) + 1,

                "Detector_Code":
                    detector,

                "Source_Failure_Date":
                    pattern["Date"],

                "Source_DayOfWeek":
                    source_dow,

                "Target_Injection_Date":
                    target_date,

                "Target_DayOfWeek":
                    target_date.dayofweek,

                "Start_Interval_Index":
                    start_idx,

                "End_Interval_Index":
                    end_idx,

                "Run_Length":
                    run_length,

                "Failure_Duration_Minutes":
                    run_length * 15,

                "Failure_Duration_Hours":
                    run_length * 15 / 60,

                "Failure_Start":
                    failure_start,

                "Failure_End":
                    failure_end,

                "Evaluation_End":
                    evaluation_end,

                "Evaluation_Window_Rows":
                    evaluation_rows,

                "NM_REGION":
                    pattern.get(
                        "NM_REGION",
                        np.nan
                    ),

                "Volume_Group":
                    pattern.get(
                        "Volume_Group",
                        np.nan
                    ),

                "Alarm_Exposure_Group":
                    pattern.get(
                        "Alarm_Exposure_Group",
                        np.nan
                    ),

                "Sensitivity_Cohort":
                    pattern.get(
                        "Sensitivity_Cohort",
                        np.nan
                    )
            })

            event_found = True
            break

        if event_found:
            break

    if not event_found:
        unselected_detectors.append(
            detector
        )

design = pd.DataFrame(
    selected_events
)

# ------------------------------------------------------------
# 7. Basic design integrity checks
# ------------------------------------------------------------
print("\n" + "=" * 112)
print("CONTROLLED INJECTION DESIGN — COVERAGE")
print("=" * 112)

print(
    f"\nEligible source detectors      : "
    f"{len(detector_ids):,}"
)

print(
    f"Selected injection events      : "
    f"{len(design):,}"
)

print(
    f"Unique injected detectors      : "
    f"{design['Detector_Code'].nunique():,}"
)

print(
    f"Detectors without valid window : "
    f"{len(unselected_detectors):,}"
)

duplicate_detector_events = (
    design["Detector_Code"]
    .duplicated()
    .sum()
)

print(
    f"Duplicate detector assignments : "
    f"{duplicate_detector_events:,}"
)

dow_mismatches = (
    design["Source_DayOfWeek"]
    != design["Target_DayOfWeek"]
).sum()

print(
    f"Day-of-week mismatches         : "
    f"{dow_mismatches:,}"
)

# ------------------------------------------------------------
# 8. Selected pattern distribution
# ------------------------------------------------------------
print("\n" + "=" * 112)
print("SELECTED FAILURE-PATTERN DISTRIBUTION")
print("=" * 112)

print(
    design["Run_Length"].describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

print("\nFailure duration categories:")

def duration_group(run_length):

    if run_length == 1:
        return "15 min"

    elif run_length <= 4:
        return "30–60 min"

    elif run_length <= 8:
        return "1.25–2 h"

    elif run_length <= 24:
        return "2.25–6 h"

    elif run_length <= 48:
        return "6.25–12 h"

    elif run_length <= 95:
        return "12.25–23.75 h"

    elif run_length == 96:
        return "24 h"

    return "Other"


design[
    "Failure_Duration_Group"
] = (
    design["Run_Length"]
    .apply(duration_group)
)

duration_order = [
    "15 min",
    "30–60 min",
    "1.25–2 h",
    "2.25–6 h",
    "6.25–12 h",
    "12.25–23.75 h",
    "24 h"
]

duration_counts = (
    design[
        "Failure_Duration_Group"
    ]
    .value_counts()
    .reindex(
        duration_order,
        fill_value=0
    )
)

print(
    duration_counts.to_string()
)

# ------------------------------------------------------------
# 9. Compare selected-event distribution with full
#    Jan–Jul empirical pattern library
# ------------------------------------------------------------
ks_selected = ks_2samp(
    patterns["Run_Length"],
    design["Run_Length"],
    alternative="two-sided",
    mode="auto"
)

print("\n" + "=" * 112)
print("REPRESENTATIVENESS OF SELECTED INJECTION EVENTS")
print("=" * 112)

print(
    f"\nKS statistic : "
    f"{ks_selected.statistic:.4f}"
)

print(
    f"KS p-value   : "
    f"{ks_selected.pvalue:.6f}"
)

print(
    f"\nFull pattern median run length : "
    f"{patterns['Run_Length'].median():.2f}"
)

print(
    f"Selected-event median          : "
    f"{design['Run_Length'].median():.2f}"
)

print(
    f"\nFull pattern % ≤1 hour         : "
    f"{100*(patterns['Run_Length'] <= 4).mean():.2f}%"
)

print(
    f"Selected-event % ≤1 hour       : "
    f"{100*(design['Run_Length'] <= 4).mean():.2f}%"
)

# ------------------------------------------------------------
# 10. Injection dates
# ------------------------------------------------------------
date_summary = (
    design.groupby(
        "Target_Injection_Date"
    )
    .agg(
        Injection_Events=(
            "Injection_Event_ID",
            "size"
        ),
        Injected_Detectors=(
            "Detector_Code",
            "nunique"
        ),
        Total_Failed_Intervals=(
            "Run_Length",
            "sum"
        )
    )
    .reset_index()
)

print("\n" + "=" * 112)
print("AUGUST INJECTION-DATE DISTRIBUTION")
print("=" * 112)

display(date_summary)

# ------------------------------------------------------------
# 11. Quantify total controlled corruption
# ------------------------------------------------------------
total_failed_intervals = (
    design["Run_Length"].sum()
)

print("\n" + "=" * 112)
print("CONTROLLED CORRUPTION SCALE")
print("=" * 112)

print(
    f"\nInjected failure events     : "
    f"{len(design):,}"
)

print(
    f"Injected detector intervals : "
    f"{total_failed_intervals:,}"
)

print(
    f"Median failure length       : "
    f"{design['Run_Length'].median():.2f} intervals"
)

print(
    f"Median duration             : "
    f"{design['Failure_Duration_Hours'].median():.2f} h"
)

# ------------------------------------------------------------
# 12. Save event design
# ------------------------------------------------------------
design_file = (
    DATA_DIR
    / "Controlled_August_Failure_Injection_Events.csv"
)

design.to_csv(
    design_file,
    index=False
)

date_summary_file = (
    TABLES_DIR
    / "Table_Controlled_Injection_Date_Distribution.csv"
)

date_summary.to_csv(
    date_summary_file,
    index=False
)

design_summary = pd.DataFrame({

    "Metric": [
        "Eligible_Source_Detectors",
        "Selected_Injection_Events",
        "Unique_Injected_Detectors",
        "Unselected_Detectors",
        "Total_Injected_Failure_Intervals",
        "Median_Selected_Run_Length",
        "Selected_Percent_LE_1h",
        "Selection_KS_Statistic",
        "Selection_KS_PValue"
    ],

    "Value": [
        len(detector_ids),
        len(design),
        design[
            "Detector_Code"
        ].nunique(),
        len(unselected_detectors),
        total_failed_intervals,
        design[
            "Run_Length"
        ].median(),
        100 * (
            design[
                "Run_Length"
            ] <= 4
        ).mean(),
        ks_selected.statistic,
        ks_selected.pvalue
    ]
})

design_summary_file = (
    TABLES_DIR
    / "Table_Controlled_Injection_Design_Summary.csv"
)

design_summary.to_csv(
    design_summary_file,
    index=False
)

# ------------------------------------------------------------
# 13. Update experiment provenance
# ------------------------------------------------------------
provenance_update = {

    "random_seed":
        RANDOM_SEED,

    "event_selection":
        "Maximum one injected event per detector",

    "source_pattern_period":
        "2026-01-01 to 2026-07-31",

    "injection_period":
        "2026-08-01 to 2026-08-30",

    "day_of_week_preserved":
        True,

    "start_interval_preserved":
        True,

    "run_length_preserved":
        True,

    "same_detector_preserved":
        True,

    "baseline_window_requirement":
        (
            "Complete baseline predictions from failure "
            "start through failure end + 24 hours"
        ),

    "purpose":
        (
            "Isolated empirically grounded robustness "
            "evaluation under missing historical detector data"
        )
}

provenance_file = (
    RESULTS_DIR
    / "Controlled_Failure_Injection_Event_Selection.json"
)

with open(
    provenance_file,
    "w"
) as f:

    json.dump(
        provenance_update,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 14. Final output
# ------------------------------------------------------------
print("\n" + "=" * 112)
print("FILES SAVED")
print("=" * 112)

print(design_file)
print(date_summary_file)
print(design_summary_file)
print(provenance_file)

print("\n" + "=" * 112)
print("STEP 12 COMPLETE")
print("=" * 112)

In [ ]:
# ============================================================
# STEP 13 — EMPIRICALLY GROUNDED FAILURE INJECTION
#
# Inject missing historical observations according to
# real Jan–Jul SCATS failure patterns into CLEAN August windows.
#
# IMPORTANT:
# - Actual target values are NEVER modified.
# - Only lag inputs whose timestamps fall inside the injected
#   detector-failure interval are set to NaN.
# - Models are NOT retrained.
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import json

print("=" * 115)
print("STEP 13 — EMPIRICALLY GROUNDED FAILURE INJECTION")
print("=" * 115)

# ------------------------------------------------------------
# 1. Load controlled injection design
# ------------------------------------------------------------
design_file = (
    DATA_DIR
    / "Controlled_August_Failure_Injection_Events.csv"
)

design = pd.read_csv(
    design_file,
    parse_dates=[
        "Source_Failure_Date",
        "Target_Injection_Date",
        "Failure_Start",
        "Failure_End",
        "Evaluation_End"
    ]
)

print(f"\nControlled injection events: {len(design):,}")
print(
    f"Unique injected detectors : "
    f"{design['Detector_Code'].nunique():,}"
)

# ------------------------------------------------------------
# 2. Load final trained models
# ------------------------------------------------------------
lgb_model_file = (
    NOTEBOOK3_DIR
    / "Models"
    / "LightGBM_FINAL_JanJul_1200.txt"
)

xgb_model_file = (
    NOTEBOOK3_DIR
    / "Models"
    / "XGBoost_FINAL_JanJul_624.json"
)

lgb_model = lgb.Booster(
    model_file=str(lgb_model_file)
)

xgb_model = xgb.Booster()
xgb_model.load_model(
    str(xgb_model_file)
)

lgb_features = lgb_model.feature_name()
xgb_features = xgb_model.feature_names

print("\n✓ LightGBM loaded")
print("✓ XGBoost loaded")

print("\nModel features:")
for feature in lgb_features:
    print(f"  • {feature}")

# ------------------------------------------------------------
# 3. Define lag structure
# ------------------------------------------------------------
lag_offsets = {
    "Lag_1": 1,
    "Lag_2": 2,
    "Lag_4": 4,
    "Lag_8": 8,
    "Lag_96": 96
}

# ------------------------------------------------------------
# 4. Build evaluation timestamps
#
# Each event is evaluated from the failure start through
# failure end + 24 hours.
# ------------------------------------------------------------
event_windows = []

for _, event in design.iterrows():

    timestamps = pd.date_range(
        event["Failure_Start"],
        event["Evaluation_End"],
        freq="15min"
    )

    event_df = pd.DataFrame({
        "Timestamp": timestamps
    })

    event_df["Injection_Event_ID"] = int(
        event["Injection_Event_ID"]
    )

    event_df["Detector_Code"] = int(
        event["Detector_Code"]
    )

    event_df["Failure_Start"] = (
        event["Failure_Start"]
    )

    event_df["Failure_End"] = (
        event["Failure_End"]
    )

    event_df["Run_Length"] = int(
        event["Run_Length"]
    )

    event_df["Failure_Duration_Hours"] = (
        event["Failure_Duration_Hours"]
    )

    event_df["Failure_Duration_Group"] = (
        event["Failure_Duration_Group"]
    )

    event_windows.append(
        event_df
    )

evaluation_keys = pd.concat(
    event_windows,
    ignore_index=True
)

print("\n" + "=" * 115)
print("CONTROLLED EVALUATION WINDOWS")
print("=" * 115)

print(
    f"\nEvaluation rows requested : "
    f"{len(evaluation_keys):,}"
)

print(
    f"Unique events             : "
    f"{evaluation_keys['Injection_Event_ID'].nunique():,}"
)

print(
    f"Unique detectors          : "
    f"{evaluation_keys['Detector_Code'].nunique():,}"
)

duplicate_keys = (
    evaluation_keys
    .duplicated(
        subset=[
            "Detector_Code",
            "Timestamp"
        ]
    )
    .sum()
)

print(
    f"Duplicate detector-times  : "
    f"{duplicate_keys:,}"
)

# ------------------------------------------------------------
# 5. Load only required forecasting features
# ------------------------------------------------------------
forecast_file = (
    NOTEBOOK3_DIR
    / "Data"
    / "Forecasting_Monthly"
    / "FORECASTING_2026-08.parquet"
)

forecast_columns = [
    "Timestamp",
    "Detector_Code",
    "Target_Volume",
    "Interval_Index",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_2",
    "Lag_4",
    "Lag_8",
    "Lag_96"
]

forecast = pd.read_parquet(
    forecast_file,
    columns=forecast_columns
)

forecast["Timestamp"] = pd.to_datetime(
    forecast["Timestamp"]
)

# Only retain requested event windows
experiment = evaluation_keys.merge(
    forecast,
    on=[
        "Timestamp",
        "Detector_Code"
    ],
    how="left",
    validate="one_to_one"
)

missing_feature_rows = (
    experiment[
        "Target_Volume"
    ]
    .isna()
    .sum()
)

print("\nForecast feature merge:")
print(
    f"  Matched rows : "
    f"{len(experiment) - missing_feature_rows:,}"
)
print(
    f"  Missing rows : "
    f"{missing_feature_rows:,}"
)

# ------------------------------------------------------------
# 6. Load clean Notebook 3 baseline predictions
# ------------------------------------------------------------
baseline_file = (
    NOTEBOOK3_DIR
    / "Data"
    / "August_Final_All_Model_Predictions.parquet"
)

baseline = pd.read_parquet(
    baseline_file,
    columns=[
        "Timestamp",
        "Detector_Code",
        "Actual",
        "LightGBM_Prediction",
        "XGBoost_Prediction",
        "Persistence_15min",
        "Seasonal_Naive_24h"
    ]
)

baseline["Timestamp"] = pd.to_datetime(
    baseline["Timestamp"]
)

experiment = experiment.merge(
    baseline,
    on=[
        "Timestamp",
        "Detector_Code"
    ],
    how="left",
    validate="one_to_one"
)

baseline_missing = (
    experiment[
        "Actual"
    ]
    .isna()
    .sum()
)

print("\nBaseline prediction merge:")
print(
    f"  Matched rows : "
    f"{len(experiment) - baseline_missing:,}"
)
print(
    f"  Missing rows : "
    f"{baseline_missing:,}"
)

# ------------------------------------------------------------
# 7. Ground-truth consistency
# ------------------------------------------------------------
target_difference = (
    experiment["Target_Volume"]
    - experiment["Actual"]
).abs()

print("\n" + "=" * 115)
print("GROUND-TRUTH CONSISTENCY")
print("=" * 115)

print(
    f"\nMaximum |Target_Volume - Actual| : "
    f"{target_difference.max():.12f}"
)

print(
    f"Exact matching rows              : "
    f"{(target_difference == 0).sum():,} / "
    f"{len(experiment):,}"
)

# ------------------------------------------------------------
# 8. Inject missing historical observations
# ------------------------------------------------------------
print("\n" + "=" * 115)
print("INJECTING REAL FAILURE PATTERNS INTO LAG FEATURES")
print("=" * 115)

for lag_name, lag_steps in lag_offsets.items():

    lag_timestamp = (
        experiment["Timestamp"]
        - pd.to_timedelta(
            lag_steps * 15,
            unit="m"
        )
    )

    impacted = (
        (lag_timestamp >= experiment["Failure_Start"])
        &
        (lag_timestamp <= experiment["Failure_End"])
    )

    impact_col = (
        f"{lag_name}_Injected_Missing"
    )

    injected_col = (
        f"{lag_name}_Injected"
    )

    experiment[impact_col] = impacted

    experiment[injected_col] = (
        experiment[lag_name]
    )

    experiment.loc[
        impacted,
        injected_col
    ] = np.nan

    print(
        f"{lag_name:7s} → "
        f"{impacted.sum():,} injected missing values"
    )

# ------------------------------------------------------------
# 9. Overall injected-missing pattern
# ------------------------------------------------------------
impact_columns = [
    f"{lag}_Injected_Missing"
    for lag in lag_offsets
]

experiment[
    "Missing_Lag_Count"
] = (
    experiment[
        impact_columns
    ]
    .sum(axis=1)
    .astype("int8")
)

experiment[
    "Any_Injected_Lag_Missing"
] = (
    experiment[
        "Missing_Lag_Count"
    ] > 0
)

def create_missing_pattern(row):

    missing = [
        lag
        for lag in lag_offsets
        if row[
            f"{lag}_Injected_Missing"
        ]
    ]

    if not missing:
        return "None"

    return "+".join(missing)


experiment[
    "Missing_Lag_Pattern"
] = experiment.apply(
    create_missing_pattern,
    axis=1
)

affected = experiment[
    experiment[
        "Any_Injected_Lag_Missing"
    ]
].copy()

print("\n" + "=" * 115)
print("INJECTION COVERAGE")
print("=" * 115)

print(
    f"\nTotal evaluation rows : "
    f"{len(experiment):,}"
)

print(
    f"Affected target rows  : "
    f"{len(affected):,}"
)

print(
    f"Unaffected rows       : "
    f"{len(experiment) - len(affected):,}"
)

print(
    f"Affected events       : "
    f"{affected['Injection_Event_ID'].nunique():,}"
)

print(
    f"Affected detectors    : "
    f"{affected['Detector_Code'].nunique():,}"
)

print("\nMissing-lag-count distribution:")
print(
    affected[
        "Missing_Lag_Count"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nMost common missing-lag patterns:")
print(
    affected[
        "Missing_Lag_Pattern"
    ]
    .value_counts()
    .head(20)
    .to_string()
)

# ------------------------------------------------------------
# 10. Prepare injected model feature matrices
# ------------------------------------------------------------
X_lgb = affected[
    lgb_features
].copy()

X_xgb = affected[
    xgb_features
].copy()

# Replace clean lag features with injected versions
for lag_name in lag_offsets:

    if lag_name in X_lgb.columns:

        X_lgb[lag_name] = (
            affected[
                f"{lag_name}_Injected"
            ]
            .to_numpy()
        )

    if lag_name in X_xgb.columns:

        X_xgb[lag_name] = (
            affected[
                f"{lag_name}_Injected"
            ]
            .to_numpy()
        )

# ------------------------------------------------------------
# 11. Generate injected LightGBM predictions
# ------------------------------------------------------------
lgb_injected_raw = lgb_model.predict(
    X_lgb
)

affected[
    "LightGBM_Injected_Prediction"
] = np.maximum(
    lgb_injected_raw,
    0
)

# ------------------------------------------------------------
# 12. Generate injected XGBoost predictions
# ------------------------------------------------------------
xgb_matrix = xgb.DMatrix(
    X_xgb,
    feature_names=xgb_features
)

xgb_injected_raw = xgb_model.predict(
    xgb_matrix
)

affected[
    "XGBoost_Injected_Prediction"
] = np.maximum(
    xgb_injected_raw,
    0
)

# ------------------------------------------------------------
# 13. Reconstruct naive baselines under missing data
# ------------------------------------------------------------
affected[
    "Persistence_Injected_Prediction"
] = (
    affected["Lag_1_Injected"]
)

affected[
    "Seasonal_Naive_Injected_Prediction"
] = (
    affected["Lag_96_Injected"]
)

# ------------------------------------------------------------
# 14. Prediction shifts caused only by failure injection
# ------------------------------------------------------------
affected[
    "LightGBM_Prediction_Shift"
] = (
    affected[
        "LightGBM_Injected_Prediction"
    ]
    - affected[
        "LightGBM_Prediction"
    ]
)

affected[
    "XGBoost_Prediction_Shift"
] = (
    affected[
        "XGBoost_Injected_Prediction"
    ]
    - affected[
        "XGBoost_Prediction"
    ]
)

# ------------------------------------------------------------
# 15. Scoring availability
# ------------------------------------------------------------
print("\n" + "=" * 115)
print("MODEL SCORING AVAILABILITY AFTER FAILURE INJECTION")
print("=" * 115)

availability = pd.DataFrame({

    "Model": [
        "LightGBM",
        "XGBoost",
        "Persistence_15min",
        "Seasonal_Naive_24h"
    ],

    "Affected_Rows": [
        len(affected),
        len(affected),
        len(affected),
        len(affected)
    ],

    "Predictions_Available": [
        affected[
            "LightGBM_Injected_Prediction"
        ].notna().sum(),

        affected[
            "XGBoost_Injected_Prediction"
        ].notna().sum(),

        affected[
            "Persistence_Injected_Prediction"
        ].notna().sum(),

        affected[
            "Seasonal_Naive_Injected_Prediction"
        ].notna().sum()
    ]
})

availability[
    "Availability_Percent"
] = (
    100
    * availability[
        "Predictions_Available"
    ]
    / availability[
        "Affected_Rows"
    ]
)

display(availability)

# ------------------------------------------------------------
# 16. Prediction-shift diagnostics
# ------------------------------------------------------------
print("\n" + "=" * 115)
print("FAILURE-INDUCED MODEL PREDICTION SHIFTS")
print("=" * 115)

for col, label in [
    (
        "LightGBM_Prediction_Shift",
        "LightGBM"
    ),
    (
        "XGBoost_Prediction_Shift",
        "XGBoost"
    )
]:

    values = affected[col]

    print(f"\n{label}")
    print("-" * 65)

    print(
        values.describe(
            percentiles=[
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99
            ]
        )
    )

    print(
        f"Rows with any prediction change: "
        f"{(values.abs() > 1e-8).sum():,}"
    )

# ------------------------------------------------------------
# 17. Event-level injection integrity summary
# ------------------------------------------------------------
event_summary = (
    affected.groupby(
        "Injection_Event_ID"
    )
    .agg(
        Detector_Code=(
            "Detector_Code",
            "first"
        ),
        Failure_Run_Length=(
            "Run_Length",
            "first"
        ),
        Failure_Duration_Hours=(
            "Failure_Duration_Hours",
            "first"
        ),
        Failure_Duration_Group=(
            "Failure_Duration_Group",
            "first"
        ),
        Affected_Target_Rows=(
            "Timestamp",
            "size"
        ),
        Maximum_Missing_Lags=(
            "Missing_Lag_Count",
            "max"
        ),
        Mean_Missing_Lags=(
            "Missing_Lag_Count",
            "mean"
        ),
        Mean_Absolute_LGB_Shift=(
            "LightGBM_Prediction_Shift",
            lambda x: x.abs().mean()
        ),
        Mean_Absolute_XGB_Shift=(
            "XGBoost_Prediction_Shift",
            lambda x: x.abs().mean()
        )
    )
    .reset_index()
)

print("\n" + "=" * 115)
print("EVENT-LEVEL INJECTION SUMMARY")
print("=" * 115)

print(
    event_summary[
        "Affected_Target_Rows"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

# ------------------------------------------------------------
# 18. Save ONLY critical experiment outputs
# ------------------------------------------------------------

# Affected rows only — no need to duplicate 2.1M clean rows
affected_output_columns = [
    "Injection_Event_ID",
    "Detector_Code",
    "Timestamp",
    "Failure_Start",
    "Failure_End",
    "Run_Length",
    "Failure_Duration_Hours",
    "Failure_Duration_Group",
    "Actual",
    "Target_Volume",
    "Missing_Lag_Count",
    "Missing_Lag_Pattern",
    "Lag_1",
    "Lag_1_Injected",
    "Lag_2",
    "Lag_2_Injected",
    "Lag_4",
    "Lag_4_Injected",
    "Lag_8",
    "Lag_8_Injected",
    "Lag_96",
    "Lag_96_Injected",
    "LightGBM_Prediction",
    "LightGBM_Injected_Prediction",
    "LightGBM_Prediction_Shift",
    "XGBoost_Prediction",
    "XGBoost_Injected_Prediction",
    "XGBoost_Prediction_Shift",
    "Persistence_15min",
    "Persistence_Injected_Prediction",
    "Seasonal_Naive_24h",
    "Seasonal_Naive_Injected_Prediction"
]

affected_file = (
    DATA_DIR
    / "Controlled_Failure_Injected_Affected_Predictions.parquet"
)

affected[
    affected_output_columns
].to_parquet(
    affected_file,
    index=False
)

event_summary_file = (
    TABLES_DIR
    / "Table_Controlled_Failure_Injection_Event_Summary.csv"
)

event_summary.to_csv(
    event_summary_file,
    index=False
)

availability_file = (
    TABLES_DIR
    / "Table_Injected_Model_Prediction_Availability.csv"
)

availability.to_csv(
    availability_file,
    index=False
)

# ------------------------------------------------------------
# 19. Save experiment metadata
# ------------------------------------------------------------
experiment_metadata = {

    "experiment_name":
        "Empirically Grounded Real-Failure Injection",

    "injection_events":
        int(len(design)),

    "unique_detectors":
        int(
            design[
                "Detector_Code"
            ].nunique()
        ),

    "failure_pattern_source":
        "Observed Jan-Jul 2026 SCATS failure runs",

    "evaluation_period":
        "August 2026 held-out test set",

    "target_modified":
        False,

    "injected_variables": [
        "Lag_1",
        "Lag_2",
        "Lag_4",
        "Lag_8",
        "Lag_96"
    ],

    "lag_offsets_minutes": {
        "Lag_1": 15,
        "Lag_2": 30,
        "Lag_4": 60,
        "Lag_8": 120,
        "Lag_96": 1440
    },

    "model_retraining":
        False,

    "negative_prediction_postprocessing":
        "Clipped to zero, identical to Notebook 3",

    "saved_dataset_scope":
        "Only target rows affected by at least one injected missing lag"
}

metadata_file = (
    RESULTS_DIR
    / "Controlled_Failure_Injection_Experiment_Metadata.json"
)

with open(
    metadata_file,
    "w"
) as f:

    json.dump(
        experiment_metadata,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 20. Final output
# ------------------------------------------------------------
print("\n" + "=" * 115)
print("FILES SAVED")
print("=" * 115)

print(affected_file)
print(event_summary_file)
print(availability_file)
print(metadata_file)

print("\n" + "=" * 115)
print("STEP 13 COMPLETE")
print("=" * 115)

In [ ]:
# ============================================================
# STEP 14 — ROBUSTNESS PERFORMANCE ANALYSIS
#
# Compare clean baseline forecasting performance against
# empirically grounded failure-injected performance.
#
# Main objectives:
#   1. Quantify accuracy degradation for LightGBM and XGBoost
#   2. Separate accuracy robustness from prediction availability
#   3. Examine degradation by:
#        - number of missing lags
#        - missing-lag pattern
#        - real failure duration
# ============================================================

import pandas as pd
import numpy as np
import json

print("=" * 115)
print("STEP 14 — ROBUSTNESS PERFORMANCE ANALYSIS")
print("=" * 115)

# ------------------------------------------------------------
# 1. Load affected controlled-experiment predictions
# ------------------------------------------------------------
affected_file = (
    DATA_DIR
    / "Controlled_Failure_Injected_Affected_Predictions.parquet"
)

df = pd.read_parquet(
    affected_file
)

df["Timestamp"] = pd.to_datetime(
    df["Timestamp"]
)

print(f"\nAffected forecasting rows : {len(df):,}")
print(
    f"Injection events          : "
    f"{df['Injection_Event_ID'].nunique():,}"
)
print(
    f"Detectors                 : "
    f"{df['Detector_Code'].nunique():,}"
)

# ------------------------------------------------------------
# 2. Metric function
# ------------------------------------------------------------
def calculate_metrics(
    actual,
    prediction
):

    actual = np.asarray(
        actual,
        dtype=float
    )

    prediction = np.asarray(
        prediction,
        dtype=float
    )

    valid = (
        np.isfinite(actual)
        & np.isfinite(prediction)
    )

    actual = actual[valid]
    prediction = prediction[valid]

    if len(actual) == 0:

        return {
            "N": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "WAPE": np.nan,
            "Bias": np.nan,
            "Median_AE": np.nan
        }

    error = prediction - actual
    abs_error = np.abs(error)

    denominator = np.abs(actual).sum()

    wape = (
        100 * abs_error.sum() / denominator
        if denominator > 0
        else np.nan
    )

    return {
        "N": len(actual),

        "MAE":
            abs_error.mean(),

        "RMSE":
            np.sqrt(
                np.mean(
                    error ** 2
                )
            ),

        "WAPE":
            wape,

        "Bias":
            error.mean(),

        "Median_AE":
            np.median(
                abs_error
            )
    }

# ------------------------------------------------------------
# 3. Model configuration
# ------------------------------------------------------------
models = {

    "LightGBM": {
        "baseline":
            "LightGBM_Prediction",

        "injected":
            "LightGBM_Injected_Prediction"
    },

    "XGBoost": {
        "baseline":
            "XGBoost_Prediction",

        "injected":
            "XGBoost_Injected_Prediction"
    },

    "Persistence_15min": {
        "baseline":
            "Persistence_15min",

        "injected":
            "Persistence_Injected_Prediction"
    },

    "Seasonal_Naive_24h": {
        "baseline":
            "Seasonal_Naive_24h",

        "injected":
            "Seasonal_Naive_Injected_Prediction"
    }
}

# ------------------------------------------------------------
# 4. Overall performance comparison
#
# Important:
# For each model we calculate:
#
# A) CLEAN performance on ALL affected rows
#
# B) Paired CLEAN performance on exactly the rows where an
#    injected prediction remains available
#
# C) Injected performance on those same rows
#
# This keeps accuracy comparison fair for models whose
# prediction becomes unavailable.
# ------------------------------------------------------------
overall_rows = []

for model_name, cols in models.items():

    baseline_col = cols["baseline"]
    injected_col = cols["injected"]

    total_rows = len(df)

    # ----------------------------------------
    # Clean baseline — all affected rows
    # ----------------------------------------
    clean_all = calculate_metrics(
        df["Actual"],
        df[baseline_col]
    )

    # ----------------------------------------
    # Rows where injected prediction exists
    # ----------------------------------------
    available_mask = (
        df[injected_col].notna()
    )

    paired_df = df.loc[
        available_mask
    ]

    clean_paired = calculate_metrics(
        paired_df["Actual"],
        paired_df[baseline_col]
    )

    injected = calculate_metrics(
        paired_df["Actual"],
        paired_df[injected_col]
    )

    availability_percent = (
        100
        * available_mask.sum()
        / total_rows
    )

    # ----------------------------------------
    # Degradation
    # ----------------------------------------
    mae_delta = (
        injected["MAE"]
        - clean_paired["MAE"]
    )

    rmse_delta = (
        injected["RMSE"]
        - clean_paired["RMSE"]
    )

    wape_delta = (
        injected["WAPE"]
        - clean_paired["WAPE"]
    )

    mae_degradation_percent = (
        100
        * mae_delta
        / clean_paired["MAE"]
        if clean_paired["MAE"] > 0
        else np.nan
    )

    rmse_degradation_percent = (
        100
        * rmse_delta
        / clean_paired["RMSE"]
        if clean_paired["RMSE"] > 0
        else np.nan
    )

    wape_degradation_percent = (
        100
        * wape_delta
        / clean_paired["WAPE"]
        if clean_paired["WAPE"] > 0
        else np.nan
    )

    overall_rows.append({

        "Model":
            model_name,

        "Affected_Rows":
            total_rows,

        "Injected_Predictions_Available":
            int(
                available_mask.sum()
            ),

        "Availability_Percent":
            availability_percent,

        # Clean performance on full affected set
        "Clean_All_MAE":
            clean_all["MAE"],

        "Clean_All_RMSE":
            clean_all["RMSE"],

        "Clean_All_WAPE":
            clean_all["WAPE"],

        "Clean_All_Bias":
            clean_all["Bias"],

        # Paired clean performance
        "Paired_Clean_MAE":
            clean_paired["MAE"],

        "Paired_Clean_RMSE":
            clean_paired["RMSE"],

        "Paired_Clean_WAPE":
            clean_paired["WAPE"],

        "Paired_Clean_Bias":
            clean_paired["Bias"],

        # Injected
        "Injected_MAE":
            injected["MAE"],

        "Injected_RMSE":
            injected["RMSE"],

        "Injected_WAPE":
            injected["WAPE"],

        "Injected_Bias":
            injected["Bias"],

        # Absolute degradation
        "MAE_Delta":
            mae_delta,

        "RMSE_Delta":
            rmse_delta,

        "WAPE_Delta_pp":
            wape_delta,

        # Relative degradation
        "MAE_Degradation_Percent":
            mae_degradation_percent,

        "RMSE_Degradation_Percent":
            rmse_degradation_percent,

        "WAPE_Degradation_Percent":
            wape_degradation_percent
    })

overall = pd.DataFrame(
    overall_rows
)

print("\n" + "=" * 115)
print("OVERALL ROBUSTNESS PERFORMANCE")
print("=" * 115)

display(
    overall[
        [
            "Model",
            "Availability_Percent",
            "Paired_Clean_MAE",
            "Injected_MAE",
            "MAE_Degradation_Percent",
            "Paired_Clean_RMSE",
            "Injected_RMSE",
            "Paired_Clean_WAPE",
            "Injected_WAPE",
            "WAPE_Delta_pp",
            "Injected_Bias"
        ]
    ]
)

# ------------------------------------------------------------
# 5. Row-level paired error degradation for ML models
# ------------------------------------------------------------
for model_name, baseline_col, injected_col in [

    (
        "LightGBM",
        "LightGBM_Prediction",
        "LightGBM_Injected_Prediction"
    ),

    (
        "XGBoost",
        "XGBoost_Prediction",
        "XGBoost_Injected_Prediction"
    )
]:

    prefix = (
        "LGB"
        if model_name == "LightGBM"
        else "XGB"
    )

    df[
        f"{prefix}_Baseline_AE"
    ] = (
        df[baseline_col]
        - df["Actual"]
    ).abs()

    df[
        f"{prefix}_Injected_AE"
    ] = (
        df[injected_col]
        - df["Actual"]
    ).abs()

    df[
        f"{prefix}_AE_Delta"
    ] = (
        df[
            f"{prefix}_Injected_AE"
        ]
        - df[
            f"{prefix}_Baseline_AE"
        ]
    )

# ------------------------------------------------------------
# 6. Row-level degradation summary
# ------------------------------------------------------------
print("\n" + "=" * 115)
print("ROW-LEVEL PAIRED ERROR RESPONSE")
print("=" * 115)

row_response = []

for prefix, model_name in [
    ("LGB", "LightGBM"),
    ("XGB", "XGBoost")
]:

    delta = df[
        f"{prefix}_AE_Delta"
    ]

    worsened = (
        delta > 1e-8
    ).sum()

    improved = (
        delta < -1e-8
    ).sum()

    unchanged = (
        delta.abs() <= 1e-8
    ).sum()

    row_response.append({

        "Model":
            model_name,

        "Rows":
            len(delta),

        "Error_Worsened_Rows":
            worsened,

        "Error_Worsened_Percent":
            100 * worsened / len(delta),

        "Error_Improved_Rows":
            improved,

        "Error_Improved_Percent":
            100 * improved / len(delta),

        "Error_Unchanged_Rows":
            unchanged,

        "Error_Unchanged_Percent":
            100 * unchanged / len(delta),

        "Mean_AE_Delta":
            delta.mean(),

        "Median_AE_Delta":
            delta.median(),

        "P90_AE_Delta":
            delta.quantile(0.90),

        "P95_AE_Delta":
            delta.quantile(0.95),

        "P99_AE_Delta":
            delta.quantile(0.99)
    })

row_response = pd.DataFrame(
    row_response
)

display(row_response)

# ------------------------------------------------------------
# 7. Generic grouped ML performance function
# ------------------------------------------------------------
def grouped_ml_performance(
    dataframe,
    group_col
):

    output = []

    for group_value, g in dataframe.groupby(
        group_col,
        observed=True,
        dropna=False
    ):

        for (
            model_name,
            baseline_col,
            injected_col
        ) in [

            (
                "LightGBM",
                "LightGBM_Prediction",
                "LightGBM_Injected_Prediction"
            ),

            (
                "XGBoost",
                "XGBoost_Prediction",
                "XGBoost_Injected_Prediction"
            )
        ]:

            clean = calculate_metrics(
                g["Actual"],
                g[baseline_col]
            )

            injected = calculate_metrics(
                g["Actual"],
                g[injected_col]
            )

            mae_delta = (
                injected["MAE"]
                - clean["MAE"]
            )

            wape_delta = (
                injected["WAPE"]
                - clean["WAPE"]
            )

            output.append({

                group_col:
                    group_value,

                "Model":
                    model_name,

                "Rows":
                    len(g),

                "Clean_MAE":
                    clean["MAE"],

                "Injected_MAE":
                    injected["MAE"],

                "MAE_Delta":
                    mae_delta,

                "MAE_Degradation_Percent":
                    (
                        100
                        * mae_delta
                        / clean["MAE"]
                        if clean["MAE"] > 0
                        else np.nan
                    ),

                "Clean_RMSE":
                    clean["RMSE"],

                "Injected_RMSE":
                    injected["RMSE"],

                "Clean_WAPE":
                    clean["WAPE"],

                "Injected_WAPE":
                    injected["WAPE"],

                "WAPE_Delta_pp":
                    wape_delta,

                "Clean_Bias":
                    clean["Bias"],

                "Injected_Bias":
                    injected["Bias"]
            })

    return pd.DataFrame(
        output
    )

# ------------------------------------------------------------
# 8. Performance by number of missing lag features
# ------------------------------------------------------------
by_missing_count = grouped_ml_performance(
    df,
    "Missing_Lag_Count"
)

print("\n" + "=" * 115)
print("PERFORMANCE BY NUMBER OF MISSING LAGS")
print("=" * 115)

display(
    by_missing_count[
        [
            "Missing_Lag_Count",
            "Model",
            "Rows",
            "Clean_MAE",
            "Injected_MAE",
            "MAE_Degradation_Percent",
            "Clean_WAPE",
            "Injected_WAPE",
            "WAPE_Delta_pp"
        ]
    ]
)

# ------------------------------------------------------------
# 9. Performance by exact missing-lag pattern
# ------------------------------------------------------------
by_pattern = grouped_ml_performance(
    df,
    "Missing_Lag_Pattern"
)

pattern_frequency = (
    df[
        "Missing_Lag_Pattern"
    ]
    .value_counts()
)

by_pattern[
    "Pattern_Total_Rows"
] = (
    by_pattern[
        "Missing_Lag_Pattern"
    ]
    .map(
        pattern_frequency
    )
)

by_pattern = (
    by_pattern
    .sort_values(
        [
            "Pattern_Total_Rows",
            "Missing_Lag_Pattern",
            "Model"
        ],
        ascending=[
            False,
            True,
            True
        ]
    )
)

print("\n" + "=" * 115)
print("TOP MISSING-LAG PATTERNS — PERFORMANCE")
print("=" * 115)

top_patterns = (
    pattern_frequency
    .head(10)
    .index
)

display(
    by_pattern[
        by_pattern[
            "Missing_Lag_Pattern"
        ].isin(
            top_patterns
        )
    ][
        [
            "Missing_Lag_Pattern",
            "Model",
            "Rows",
            "Clean_MAE",
            "Injected_MAE",
            "MAE_Degradation_Percent",
            "Clean_WAPE",
            "Injected_WAPE",
            "WAPE_Delta_pp",
            "Injected_Bias"
        ]
    ]
)

# ------------------------------------------------------------
# 10. Performance by real failure duration group
# ------------------------------------------------------------
by_duration = grouped_ml_performance(
    df,
    "Failure_Duration_Group"
)

duration_order = [
    "15 min",
    "30–60 min",
    "1.25–2 h",
    "2.25–6 h",
    "6.25–12 h",
    "12.25–23.75 h",
    "24 h"
]

by_duration[
    "Failure_Duration_Group"
] = pd.Categorical(
    by_duration[
        "Failure_Duration_Group"
    ],
    categories=duration_order,
    ordered=True
)

by_duration = (
    by_duration
    .sort_values(
        [
            "Failure_Duration_Group",
            "Model"
        ]
    )
)

print("\n" + "=" * 115)
print("PERFORMANCE BY REAL FAILURE DURATION")
print("=" * 115)

display(
    by_duration[
        [
            "Failure_Duration_Group",
            "Model",
            "Rows",
            "Clean_MAE",
            "Injected_MAE",
            "MAE_Degradation_Percent",
            "Clean_WAPE",
            "Injected_WAPE",
            "WAPE_Delta_pp"
        ]
    ]
)

# ------------------------------------------------------------
# 11. Event-level paired robustness
# ------------------------------------------------------------
event_rows = []

for event_id, g in df.groupby(
    "Injection_Event_ID"
):

    base_info = {

        "Injection_Event_ID":
            int(event_id),

        "Detector_Code":
            int(
                g[
                    "Detector_Code"
                ].iloc[0]
            ),

        "Failure_Run_Length":
            int(
                g[
                    "Run_Length"
                ].iloc[0]
            ),

        "Failure_Duration_Hours":
            float(
                g[
                    "Failure_Duration_Hours"
                ].iloc[0]
            ),

        "Failure_Duration_Group":
            g[
                "Failure_Duration_Group"
            ].iloc[0],

        "Affected_Rows":
            len(g),

        "Maximum_Missing_Lag_Count":
            int(
                g[
                    "Missing_Lag_Count"
                ].max()
            )
    }

    for (
        prefix,
        model_name,
        baseline_col,
        injected_col
    ) in [

        (
            "LGB",
            "LightGBM",
            "LightGBM_Prediction",
            "LightGBM_Injected_Prediction"
        ),

        (
            "XGB",
            "XGBoost",
            "XGBoost_Prediction",
            "XGBoost_Injected_Prediction"
        )
    ]:

        clean = calculate_metrics(
            g["Actual"],
            g[baseline_col]
        )

        injected = calculate_metrics(
            g["Actual"],
            g[injected_col]
        )

        base_info[
            f"{prefix}_Clean_MAE"
        ] = clean["MAE"]

        base_info[
            f"{prefix}_Injected_MAE"
        ] = injected["MAE"]

        base_info[
            f"{prefix}_MAE_Delta"
        ] = (
            injected["MAE"]
            - clean["MAE"]
        )

        base_info[
            f"{prefix}_Clean_WAPE"
        ] = clean["WAPE"]

        base_info[
            f"{prefix}_Injected_WAPE"
        ] = injected["WAPE"]

        base_info[
            f"{prefix}_WAPE_Delta_pp"
        ] = (
            injected["WAPE"]
            - clean["WAPE"]
        )

    event_rows.append(
        base_info
    )

event_performance = pd.DataFrame(
    event_rows
)

print("\n" + "=" * 115)
print("EVENT-LEVEL ROBUSTNESS")
print("=" * 115)

for prefix, model_name in [
    ("LGB", "LightGBM"),
    ("XGB", "XGBoost")
]:

    values = event_performance[
        f"{prefix}_MAE_Delta"
    ]

    print(f"\n{model_name}")
    print("-" * 65)

    print(
        values.describe(
            percentiles=[
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99
            ]
        )
    )

    print(
        f"Events with worsened MAE: "
        f"{(values > 0).sum():,} / {len(values):,} "
        f"({100*(values > 0).mean():.2f}%)"
    )

# ------------------------------------------------------------
# 12. Direct LightGBM vs XGBoost robustness comparison
# ------------------------------------------------------------
lgb_row = overall[
    overall["Model"] == "LightGBM"
].iloc[0]

xgb_row = overall[
    overall["Model"] == "XGBoost"
].iloc[0]

comparison = pd.DataFrame({

    "Metric": [
        "Availability_Percent",
        "Clean_WAPE",
        "Injected_WAPE",
        "WAPE_Delta_pp",
        "WAPE_Degradation_Percent",
        "Clean_MAE",
        "Injected_MAE",
        "MAE_Degradation_Percent",
        "Injected_Bias"
    ],

    "LightGBM": [
        lgb_row[
            "Availability_Percent"
        ],
        lgb_row[
            "Paired_Clean_WAPE"
        ],
        lgb_row[
            "Injected_WAPE"
        ],
        lgb_row[
            "WAPE_Delta_pp"
        ],
        lgb_row[
            "WAPE_Degradation_Percent"
        ],
        lgb_row[
            "Paired_Clean_MAE"
        ],
        lgb_row[
            "Injected_MAE"
        ],
        lgb_row[
            "MAE_Degradation_Percent"
        ],
        lgb_row[
            "Injected_Bias"
        ]
    ],

    "XGBoost": [
        xgb_row[
            "Availability_Percent"
        ],
        xgb_row[
            "Paired_Clean_WAPE"
        ],
        xgb_row[
            "Injected_WAPE"
        ],
        xgb_row[
            "WAPE_Delta_pp"
        ],
        xgb_row[
            "WAPE_Degradation_Percent"
        ],
        xgb_row[
            "Paired_Clean_MAE"
        ],
        xgb_row[
            "Injected_MAE"
        ],
        xgb_row[
            "MAE_Degradation_Percent"
        ],
        xgb_row[
            "Injected_Bias"
        ]
    ]
})

print("\n" + "=" * 115)
print("LIGHTGBM VS XGBOOST — ROBUSTNESS SUMMARY")
print("=" * 115)

display(comparison)

# ------------------------------------------------------------
# 13. Save critical outputs
# ------------------------------------------------------------
overall_file = (
    TABLES_DIR
    / "Table_Controlled_Failure_Overall_Robustness_Performance.csv"
)

overall.to_csv(
    overall_file,
    index=False
)

row_response_file = (
    TABLES_DIR
    / "Table_Controlled_Failure_Row_Level_Response.csv"
)

row_response.to_csv(
    row_response_file,
    index=False
)

missing_count_file = (
    TABLES_DIR
    / "Table_Robustness_by_Missing_Lag_Count.csv"
)

by_missing_count.to_csv(
    missing_count_file,
    index=False
)

pattern_file = (
    TABLES_DIR
    / "Table_Robustness_by_Missing_Lag_Pattern.csv"
)

by_pattern.to_csv(
    pattern_file,
    index=False
)

duration_file = (
    TABLES_DIR
    / "Table_Robustness_by_Failure_Duration.csv"
)

by_duration.to_csv(
    duration_file,
    index=False
)

event_file = (
    DATA_DIR
    / "Controlled_Failure_Event_Level_Robustness.parquet"
)

event_performance.to_parquet(
    event_file,
    index=False
)

comparison_file = (
    TABLES_DIR
    / "Table_LightGBM_vs_XGBoost_Robustness.csv"
)

comparison.to_csv(
    comparison_file,
    index=False
)

# ------------------------------------------------------------
# 14. Save concise results metadata
# ------------------------------------------------------------
results_metadata = {

    "affected_rows":
        int(len(df)),

    "injection_events":
        int(
            df[
                "Injection_Event_ID"
            ].nunique()
        ),

    "primary_accuracy_metrics": [
        "MAE",
        "RMSE",
        "WAPE"
    ],

    "availability_metric":
        "Prediction Availability Percent",

    "paired_comparison":
        True,

    "robustness_dimensions": [
        "Overall",
        "Missing lag count",
        "Missing lag pattern",
        "Real failure duration",
        "Event level"
    ]
}

metadata_file = (
    RESULTS_DIR
    / "Robustness_Performance_Analysis_Metadata.json"
)

with open(
    metadata_file,
    "w"
) as f:

    json.dump(
        results_metadata,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 15. Final saved files
# ------------------------------------------------------------
print("\n" + "=" * 115)
print("FILES SAVED")
print("=" * 115)

for path in [
    overall_file,
    row_response_file,
    missing_count_file,
    pattern_file,
    duration_file,
    event_file,
    comparison_file,
    metadata_file
]:
    print(path)

print("\n" + "=" * 115)
print("STEP 14 COMPLETE")
print("=" * 115)

In [ ]:
# ============================================================
# STEP 15 — EVENT-LEVEL STATISTICAL INFERENCE
#
# Use the 604 controlled failure events as the statistical
# unit to avoid treating correlated 15-min rows as independent.
#
# Analyses:
#   1. Bootstrap 95% CIs for event-level MAE degradation
#   2. Wilcoxon signed-rank test vs zero degradation
#   3. Direct paired LightGBM vs XGBoost robustness comparison
#   4. Effect-size summaries
# ============================================================

import pandas as pd
import numpy as np

from scipy.stats import (
    wilcoxon,
    binomtest
)

print("=" * 115)
print("STEP 15 — EVENT-LEVEL STATISTICAL INFERENCE")
print("=" * 115)

# ------------------------------------------------------------
# 1. Load event-level robustness dataset
# ------------------------------------------------------------
event_file = (
    DATA_DIR
    / "Controlled_Failure_Event_Level_Robustness.parquet"
)

events = pd.read_parquet(
    event_file
)

print(f"\nControlled failure events: {len(events):,}")
print(
    f"Unique detectors        : "
    f"{events['Detector_Code'].nunique():,}"
)

# ------------------------------------------------------------
# 2. Bootstrap configuration
# ------------------------------------------------------------
RANDOM_SEED = 42
N_BOOTSTRAP = 10000

rng = np.random.default_rng(
    RANDOM_SEED
)

# ------------------------------------------------------------
# 3. Bootstrap helper
# ------------------------------------------------------------
def bootstrap_ci(
    values,
    statistic="mean",
    n_bootstrap=N_BOOTSTRAP,
    random_state=None
):

    values = np.asarray(
        values,
        dtype=float
    )

    values = values[
        np.isfinite(values)
    ]

    local_rng = np.random.default_rng(
        random_state
    )

    n = len(values)

    boot_stats = np.empty(
        n_bootstrap
    )

    for i in range(n_bootstrap):

        sample = local_rng.choice(
            values,
            size=n,
            replace=True
        )

        if statistic == "mean":
            boot_stats[i] = np.mean(
                sample
            )

        elif statistic == "median":
            boot_stats[i] = np.median(
                sample
            )

        else:
            raise ValueError(
                "statistic must be mean or median"
            )

    estimate = (
        np.mean(values)
        if statistic == "mean"
        else np.median(values)
    )

    lower = np.percentile(
        boot_stats,
        2.5
    )

    upper = np.percentile(
        boot_stats,
        97.5
    )

    return (
        estimate,
        lower,
        upper
    )

# ------------------------------------------------------------
# 4. Event-level degradation distributions
# ------------------------------------------------------------
model_config = {

    "LightGBM":
        "LGB_MAE_Delta",

    "XGBoost":
        "XGB_MAE_Delta"
}

inference_rows = []

print("\n" + "=" * 115)
print("MODEL-SPECIFIC EVENT-LEVEL INFERENCE")
print("=" * 115)

for i, (
    model_name,
    delta_col
) in enumerate(
    model_config.items()
):

    values = (
        events[
            delta_col
        ]
        .dropna()
        .to_numpy()
    )

    # --------------------------------------------------------
    # Bootstrap mean CI
    # --------------------------------------------------------
    (
        mean_est,
        mean_low,
        mean_high
    ) = bootstrap_ci(
        values,
        statistic="mean",
        random_state=RANDOM_SEED + i
    )

    # --------------------------------------------------------
    # Bootstrap median CI
    # --------------------------------------------------------
    (
        median_est,
        median_low,
        median_high
    ) = bootstrap_ci(
        values,
        statistic="median",
        random_state=RANDOM_SEED + 100 + i
    )

    # --------------------------------------------------------
    # Wilcoxon test:
    # H0 = median paired degradation is zero
    # --------------------------------------------------------
    nonzero_values = values[
        np.abs(values) > 1e-12
    ]

    if len(nonzero_values) > 0:

        wilcox = wilcoxon(
            nonzero_values,
            alternative="greater",
            zero_method="wilcox"
        )

        wilcoxon_stat = (
            wilcox.statistic
        )

        wilcoxon_p = (
            wilcox.pvalue
        )

    else:

        wilcoxon_stat = np.nan
        wilcoxon_p = np.nan

    # --------------------------------------------------------
    # Event-level sign test
    # --------------------------------------------------------
    worsened = int(
        (values > 0).sum()
    )

    improved = int(
        (values < 0).sum()
    )

    non_ties = (
        worsened
        + improved
    )

    if non_ties > 0:

        sign_test = binomtest(
            worsened,
            n=non_ties,
            p=0.5,
            alternative="greater"
        )

        sign_p = (
            sign_test.pvalue
        )

    else:

        sign_p = np.nan

    inference_rows.append({

        "Model":
            model_name,

        "Events":
            len(values),

        "Mean_MAE_Delta":
            mean_est,

        "Mean_MAE_Delta_CI_Lower":
            mean_low,

        "Mean_MAE_Delta_CI_Upper":
            mean_high,

        "Median_MAE_Delta":
            median_est,

        "Median_MAE_Delta_CI_Lower":
            median_low,

        "Median_MAE_Delta_CI_Upper":
            median_high,

        "Worsened_Events":
            worsened,

        "Improved_Events":
            improved,

        "Worsened_Percent":
            100
            * worsened
            / len(values),

        "Wilcoxon_Statistic":
            wilcoxon_stat,

        "Wilcoxon_OneSided_P":
            wilcoxon_p,

        "Sign_Test_OneSided_P":
            sign_p
    })

    print(
        f"\n{model_name}"
    )
    print("-" * 70)

    print(
        f"Mean MAE degradation   : "
        f"{mean_est:.4f}"
    )

    print(
        f"95% bootstrap CI       : "
        f"[{mean_low:.4f}, "
        f"{mean_high:.4f}]"
    )

    print(
        f"Median MAE degradation : "
        f"{median_est:.4f}"
    )

    print(
        f"95% bootstrap CI       : "
        f"[{median_low:.4f}, "
        f"{median_high:.4f}]"
    )

    print(
        f"Worsened events        : "
        f"{worsened:,} / "
        f"{len(values):,}"
    )

    print(
        f"Wilcoxon one-sided p   : "
        f"{wilcoxon_p:.6e}"
    )

    print(
        f"Sign-test one-sided p  : "
        f"{sign_p:.6e}"
    )

inference = pd.DataFrame(
    inference_rows
)

# ------------------------------------------------------------
# 5. Direct paired robustness comparison
#
# Positive value means XGBoost suffers greater degradation
# than LightGBM for the same injected failure event.
# ------------------------------------------------------------
events[
    "XGB_minus_LGB_MAE_Degradation"
] = (
    events["XGB_MAE_Delta"]
    - events["LGB_MAE_Delta"]
)

direct = (
    events[
        "XGB_minus_LGB_MAE_Degradation"
    ]
    .dropna()
    .to_numpy()
)

(
    direct_mean,
    direct_mean_low,
    direct_mean_high
) = bootstrap_ci(
    direct,
    statistic="mean",
    random_state=500
)

(
    direct_median,
    direct_median_low,
    direct_median_high
) = bootstrap_ci(
    direct,
    statistic="median",
    random_state=501
)

direct_nonzero = direct[
    np.abs(direct) > 1e-12
]

direct_test = wilcoxon(
    direct_nonzero,
    alternative="greater",
    zero_method="wilcox"
)

xgb_worse_count = int(
    (direct > 0).sum()
)

lgb_worse_count = int(
    (direct < 0).sum()
)

direct_non_ties = (
    xgb_worse_count
    + lgb_worse_count
)

direct_sign = binomtest(
    xgb_worse_count,
    n=direct_non_ties,
    p=0.5,
    alternative="greater"
)

print("\n" + "=" * 115)
print("DIRECT PAIRED MODEL ROBUSTNESS COMPARISON")
print("=" * 115)

print(
    "\nMetric: XGBoost MAE degradation "
    "− LightGBM MAE degradation"
)

print(
    f"\nMean paired difference   : "
    f"{direct_mean:.4f}"
)

print(
    f"95% bootstrap CI         : "
    f"[{direct_mean_low:.4f}, "
    f"{direct_mean_high:.4f}]"
)

print(
    f"Median paired difference : "
    f"{direct_median:.4f}"
)

print(
    f"95% bootstrap CI         : "
    f"[{direct_median_low:.4f}, "
    f"{direct_median_high:.4f}]"
)

print(
    f"\nEvents where XGBoost degraded more: "
    f"{xgb_worse_count:,} / "
    f"{len(direct):,} "
    f"({100*xgb_worse_count/len(direct):.2f}%)"
)

print(
    f"Events where LightGBM degraded more: "
    f"{lgb_worse_count:,} / "
    f"{len(direct):,} "
    f"({100*lgb_worse_count/len(direct):.2f}%)"
)

print(
    f"\nPaired Wilcoxon one-sided p : "
    f"{direct_test.pvalue:.6e}"
)

print(
    f"Paired sign-test p          : "
    f"{direct_sign.pvalue:.6e}"
)

# ------------------------------------------------------------
# 6. Direct-comparison summary table
# ------------------------------------------------------------
direct_summary = pd.DataFrame({

    "Metric": [
        "Events",
        "Mean_XGB_minus_LGB_MAE_Degradation",
        "Mean_95CI_Lower",
        "Mean_95CI_Upper",
        "Median_XGB_minus_LGB_MAE_Degradation",
        "Median_95CI_Lower",
        "Median_95CI_Upper",
        "Events_XGB_Degraded_More",
        "Events_LGB_Degraded_More",
        "XGB_Degraded_More_Percent",
        "Paired_Wilcoxon_OneSided_P",
        "Paired_Sign_Test_OneSided_P"
    ],

    "Value": [
        len(direct),
        direct_mean,
        direct_mean_low,
        direct_mean_high,
        direct_median,
        direct_median_low,
        direct_median_high,
        xgb_worse_count,
        lgb_worse_count,
        100
        * xgb_worse_count
        / len(direct),
        direct_test.pvalue,
        direct_sign.pvalue
    ]
})

# ------------------------------------------------------------
# 7. Failure-duration relationship at event level
# ------------------------------------------------------------
duration_corr = events[
    [
        "Failure_Duration_Hours",
        "LGB_MAE_Delta",
        "XGB_MAE_Delta"
    ]
].corr()

print("\n" + "=" * 115)
print("EVENT-LEVEL FAILURE DURATION ASSOCIATION")
print("=" * 115)

print(
    "\nFailure duration vs LightGBM MAE degradation: "
    f"{duration_corr.loc['Failure_Duration_Hours', 'LGB_MAE_Delta']:.4f}"
)

print(
    "Failure duration vs XGBoost MAE degradation : "
    f"{duration_corr.loc['Failure_Duration_Hours', 'XGB_MAE_Delta']:.4f}"
)

# ------------------------------------------------------------
# 8. Save only publication-relevant statistical outputs
# ------------------------------------------------------------
inference_file = (
    TABLES_DIR
    / "Table_Event_Level_Robustness_Statistical_Inference.csv"
)

inference.to_csv(
    inference_file,
    index=False
)

direct_file = (
    TABLES_DIR
    / "Table_Paired_LightGBM_vs_XGBoost_Robustness_Inference.csv"
)

direct_summary.to_csv(
    direct_file,
    index=False
)

# Update event dataset because direct paired metric is useful
updated_event_file = (
    DATA_DIR
    / "Controlled_Failure_Event_Level_Robustness_With_Paired_Comparison.parquet"
)

events.to_parquet(
    updated_event_file,
    index=False
)

# ------------------------------------------------------------
# 9. Save statistical-analysis metadata
# ------------------------------------------------------------
metadata = {

    "statistical_unit":
        "Controlled failure event",

    "number_of_events":
        int(len(events)),

    "bootstrap_resamples":
        N_BOOTSTRAP,

    "bootstrap_confidence_level":
        0.95,

    "primary_event_metric":
        "MAE degradation",

    "tests": [
        "One-sided Wilcoxon signed-rank vs zero degradation",
        "One-sided event-level sign test",
        "Paired LightGBM vs XGBoost Wilcoxon comparison"
    ],

    "reason_for_event_level_analysis":
        (
            "15-minute observations within the same "
            "failure event are correlated and should not "
            "be treated as independent statistical units."
        )
}

metadata_file = (
    RESULTS_DIR
    / "Event_Level_Statistical_Inference_Metadata.json"
)

with open(
    metadata_file,
    "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 10. Final outputs
# ------------------------------------------------------------
print("\n" + "=" * 115)
print("STATISTICAL INFERENCE SUMMARY")
print("=" * 115)

display(inference)

print("\nDirect model comparison:")
display(direct_summary)

print("\n" + "=" * 115)
print("FILES SAVED")
print("=" * 115)

print(inference_file)
print(direct_file)
print(updated_event_file)
print(metadata_file)

print("\n" + "=" * 115)
print("STEP 15 COMPLETE")
print("=" * 115)

In [ ]:
# ============================================================
# STEP 16 — CORE PUBLICATION-READY ROBUSTNESS FIGURES
#
# Figure 1:
# Clean vs failure-injected WAPE
#
# Figure 2:
# WAPE degradation vs number of missing lags
#
# Figure 3:
# Event-level paired robustness:
# LightGBM vs XGBoost MAE degradation
#
# All figures saved as:
#   - PNG 600 dpi
#   - PDF vector format
#
# Figure source data also saved.
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("=" * 115)
print("STEP 16 — CORE PUBLICATION-READY ROBUSTNESS FIGURES")
print("=" * 115)

# ------------------------------------------------------------
# Global publication settings
# ------------------------------------------------------------
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.dpi": 120
})

# ============================================================
# FIGURE 1
# CLEAN VS FAILURE-INJECTED WAPE
# ============================================================

overall_file = (
    TABLES_DIR
    / "Table_Controlled_Failure_Overall_Robustness_Performance.csv"
)

overall = pd.read_csv(overall_file)

ml_overall = (
    overall[
        overall["Model"].isin(
            ["LightGBM", "XGBoost"]
        )
    ]
    .copy()
    .reset_index(drop=True)
)

figure1_data = ml_overall[
    [
        "Model",
        "Paired_Clean_WAPE",
        "Injected_WAPE",
        "WAPE_Delta_pp",
        "Availability_Percent"
    ]
].copy()

fig1_data_file = (
    TABLES_DIR
    / "Figure_Data_Clean_vs_Injected_WAPE.csv"
)

figure1_data.to_csv(
    fig1_data_file,
    index=False
)

x = np.arange(
    len(figure1_data)
)

width = 0.34

fig, ax = plt.subplots(
    figsize=(7.6, 5.8)
)

bars_clean = ax.bar(
    x - width / 2,
    figure1_data["Paired_Clean_WAPE"],
    width,
    label="Clean baseline"
)

bars_injected = ax.bar(
    x + width / 2,
    figure1_data["Injected_WAPE"],
    width,
    label="Failure-injected"
)

ax.set_ylabel(
    "WAPE (%)"
)

ax.set_xticks(
    x
)

ax.set_xticklabels(
    figure1_data["Model"]
)

ax.set_title(
    "Forecasting Accuracy Under Empirically Grounded Detector Failures"
)

ax.grid(
    axis="y",
    alpha=0.25
)

ax.legend(
    frameon=False
)

# Value labels
for bars in [
    bars_clean,
    bars_injected
]:

    for bar in bars:

        height = bar.get_height()

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,
            height,
            f"{height:.1f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

fig.tight_layout()

fig1_png = (
    FIGURES_DIR
    / "Figure_Clean_vs_Injected_WAPE.png"
)

fig1_pdf = (
    FIGURES_DIR
    / "Figure_Clean_vs_Injected_WAPE.pdf"
)

fig.savefig(
    fig1_png,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    fig1_pdf,
    bbox_inches="tight"
)

plt.show()

print("\n✓ Figure 1 saved")

# ============================================================
# FIGURE 2
# ROBUSTNESS DEGRADATION VS NUMBER OF MISSING LAGS
# ============================================================

missing_file = (
    TABLES_DIR
    / "Table_Robustness_by_Missing_Lag_Count.csv"
)

missing = pd.read_csv(
    missing_file
)

figure2_data = missing[
    [
        "Missing_Lag_Count",
        "Model",
        "Rows",
        "Clean_WAPE",
        "Injected_WAPE",
        "WAPE_Delta_pp",
        "MAE_Degradation_Percent"
    ]
].copy()

fig2_data_file = (
    TABLES_DIR
    / "Figure_Data_Robustness_by_Missing_Lag_Count.csv"
)

figure2_data.to_csv(
    fig2_data_file,
    index=False
)

fig, ax = plt.subplots(
    figsize=(7.8, 5.8)
)

for model_name in [
    "LightGBM",
    "XGBoost"
]:

    subset = (
        figure2_data[
            figure2_data["Model"]
            == model_name
        ]
        .sort_values(
            "Missing_Lag_Count"
        )
    )

    ax.plot(
        subset["Missing_Lag_Count"],
        subset["Injected_WAPE"],
        marker="o",
        linewidth=2,
        markersize=6,
        label=model_name
    )

ax.set_xlabel(
    "Number of Missing Historical Lag Features"
)

ax.set_ylabel(
    "Failure-Injected WAPE (%)"
)

ax.set_xticks(
    sorted(
        figure2_data[
            "Missing_Lag_Count"
        ].unique()
    )
)

ax.set_title(
    "Forecasting Degradation as Historical Information Is Lost"
)

ax.grid(
    True,
    alpha=0.25
)

ax.legend(
    frameon=False
)

fig.tight_layout()

fig2_png = (
    FIGURES_DIR
    / "Figure_WAPE_by_Missing_Lag_Count.png"
)

fig2_pdf = (
    FIGURES_DIR
    / "Figure_WAPE_by_Missing_Lag_Count.pdf"
)

fig.savefig(
    fig2_png,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    fig2_pdf,
    bbox_inches="tight"
)

plt.show()

print("✓ Figure 2 saved")

# ============================================================
# FIGURE 3
# EVENT-LEVEL PAIRED ROBUSTNESS COMPARISON
# ============================================================

event_file = (
    DATA_DIR
    / "Controlled_Failure_Event_Level_Robustness_With_Paired_Comparison.parquet"
)

events = pd.read_parquet(
    event_file
)

figure3_data = events[
    [
        "Injection_Event_ID",
        "Detector_Code",
        "Failure_Duration_Hours",
        "LGB_MAE_Delta",
        "XGB_MAE_Delta",
        "XGB_minus_LGB_MAE_Degradation"
    ]
].copy()

fig3_data_file = (
    TABLES_DIR
    / "Figure_Data_Event_Level_Model_Robustness.csv"
)

figure3_data.to_csv(
    fig3_data_file,
    index=False
)

fig, ax = plt.subplots(
    figsize=(7.2, 6.4)
)

ax.scatter(
    figure3_data[
        "LGB_MAE_Delta"
    ],
    figure3_data[
        "XGB_MAE_Delta"
    ],
    alpha=0.55,
    s=34,
    edgecolors="none"
)

# Equality line
min_value = min(
    figure3_data[
        "LGB_MAE_Delta"
    ].min(),
    figure3_data[
        "XGB_MAE_Delta"
    ].min()
)

max_value = max(
    figure3_data[
        "LGB_MAE_Delta"
    ].max(),
    figure3_data[
        "XGB_MAE_Delta"
    ].max()
)

ax.plot(
    [
        min_value,
        max_value
    ],
    [
        min_value,
        max_value
    ],
    linestyle="--",
    linewidth=1.3,
    label="Equal degradation"
)

ax.set_xlabel(
    "LightGBM Event-Level MAE Degradation"
)

ax.set_ylabel(
    "XGBoost Event-Level MAE Degradation"
)

ax.set_title(
    "Paired Event-Level Robustness Under Identical Detector Failures"
)

ax.grid(
    True,
    alpha=0.25
)

ax.legend(
    frameon=False
)

# Add key result annotation
xgb_worse_percent = (
    100
    * (
        figure3_data[
            "XGB_minus_LGB_MAE_Degradation"
        ] > 0
    ).mean()
)

ax.text(
    0.04,
    0.95,
    f"XGBoost degraded more in "
    f"{xgb_worse_percent:.1f}% of events",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=dict(
        boxstyle="round",
        alpha=0.10
    )
)

fig.tight_layout()

fig3_png = (
    FIGURES_DIR
    / "Figure_Event_Level_LightGBM_vs_XGBoost_Robustness.png"
)

fig3_pdf = (
    FIGURES_DIR
    / "Figure_Event_Level_LightGBM_vs_XGBoost_Robustness.pdf"
)

fig.savefig(
    fig3_png,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    fig3_pdf,
    bbox_inches="tight"
)

plt.show()

print("✓ Figure 3 saved")

# ============================================================
# PAPER-READY FIGURE MANIFEST
# ============================================================

figure_manifest = pd.DataFrame({

    "Figure": [
        "Figure 1",
        "Figure 2",
        "Figure 3"
    ],

    "Title": [
        (
            "Forecasting Accuracy Under Empirically "
            "Grounded Detector Failures"
        ),
        (
            "Forecasting Degradation as Historical "
            "Information Is Lost"
        ),
        (
            "Paired Event-Level Robustness Under "
            "Identical Detector Failures"
        )
    ],

    "Primary_Message": [
        (
            "LightGBM retains substantially better accuracy "
            "than XGBoost after real-pattern failure injection."
        ),
        (
            "Forecast error increases as more historical lag "
            "features become unavailable, with XGBoost showing "
            "much stronger degradation."
        ),
        (
            "For the same controlled detector failure events, "
            "XGBoost experiences greater MAE degradation than "
            "LightGBM in approximately 91% of cases."
        )
    ],

    "PNG_File": [
        str(fig1_png),
        str(fig2_png),
        str(fig3_png)
    ],

    "PDF_File": [
        str(fig1_pdf),
        str(fig2_pdf),
        str(fig3_pdf)
    ],

    "Source_Data_File": [
        str(fig1_data_file),
        str(fig2_data_file),
        str(fig3_data_file)
    ]
})

manifest_file = (
    RESULTS_DIR
    / "Notebook_4_Paper_Figure_Manifest.csv"
)

figure_manifest.to_csv(
    manifest_file,
    index=False
)

print("\n" + "=" * 115)
print("PUBLICATION FIGURE MANIFEST")
print("=" * 115)

display(
    figure_manifest[
        [
            "Figure",
            "Title",
            "Primary_Message"
        ]
    ]
)

print("\n" + "=" * 115)
print("FILES SAVED")
print("=" * 115)

for path in [
    fig1_png,
    fig1_pdf,
    fig1_data_file,
    fig2_png,
    fig2_pdf,
    fig2_data_file,
    fig3_png,
    fig3_pdf,
    fig3_data_file,
    manifest_file
]:
    print(path)

print("\n" + "=" * 115)
print("STEP 16 COMPLETE")
print("=" * 115)

In [ ]:
# ============================================================
# STEP 17 — TRAINING MISSINGNESS & DEPLOYMENT-SHIFT AUDIT
#
# Purpose:
# Determine whether LightGBM / XGBoost were trained on
# complete lag vectors only, and quantify whether the
# failure-injection experiment represents an unseen
# missing-data regime at deployment.
# ============================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path

print("=" * 115)
print("STEP 17 — TRAINING MISSINGNESS & DEPLOYMENT-SHIFT AUDIT")
print("=" * 115)

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------
forecast_dir = (
    NOTEBOOK3_DIR
    / "Data"
    / "Forecasting_Monthly"
)

lag_cols = [
    "Lag_1",
    "Lag_2",
    "Lag_4",
    "Lag_8",
    "Lag_96"
]

months = [
    "2026-01",
    "2026-02",
    "2026-03",
    "2026-04",
    "2026-05",
    "2026-06"
]

monthly_rows = []

total_rows = 0
total_ready = 0
total_missing_any = 0
total_ready_with_missing = 0
total_not_ready_without_missing = 0

lag_missing_totals = {
    lag: 0
    for lag in lag_cols
}

lag_missing_ready_totals = {
    lag: 0
    for lag in lag_cols
}

# ------------------------------------------------------------
# 2. Audit training months one by one
# ------------------------------------------------------------
for month in months:

    file = (
        forecast_dir
        / f"FORECASTING_{month}.parquet"
    )

    print("\n" + "-" * 115)
    print(f"Auditing {month}")
    print("-" * 115)

    df = pd.read_parquet(
        file,
        columns=[
            "Timestamp",
            "Detector_Code",
            "All_Lags_Available",
            *lag_cols
        ]
    )

    rows = len(df)

    missing_matrix = (
        df[lag_cols]
        .isna()
    )

    any_missing = (
        missing_matrix
        .any(axis=1)
    )

    ready = (
        df["All_Lags_Available"]
        == True
    )

    not_ready = ~ready

    ready_with_missing = (
        ready
        & any_missing
    )

    not_ready_without_missing = (
        not_ready
        & ~any_missing
    )

    month_result = {
        "Month":
            month,

        "Total_Rows":
            rows,

        "All_Lags_Available_Rows":
            int(ready.sum()),

        "Rows_With_Any_Missing_Lag":
            int(any_missing.sum()),

        "Ready_Rows_With_Missing_Lag":
            int(ready_with_missing.sum()),

        "NotReady_Rows_Without_Missing_Lag":
            int(not_ready_without_missing.sum()),

        "Model_Ready_Percent":
            100 * ready.mean()
    }

    for lag in lag_cols:

        month_result[
            f"{lag}_Missing_All_Rows"
        ] = int(
            missing_matrix[lag].sum()
        )

        month_result[
            f"{lag}_Missing_ModelReady_Rows"
        ] = int(
            (
                ready
                & missing_matrix[lag]
            ).sum()
        )

    monthly_rows.append(
        month_result
    )

    # Aggregate totals
    total_rows += rows
    total_ready += int(
        ready.sum()
    )

    total_missing_any += int(
        any_missing.sum()
    )

    total_ready_with_missing += int(
        ready_with_missing.sum()
    )

    total_not_ready_without_missing += int(
        not_ready_without_missing.sum()
    )

    for lag in lag_cols:

        lag_missing_totals[lag] += int(
            missing_matrix[lag].sum()
        )

        lag_missing_ready_totals[lag] += int(
            (
                ready
                & missing_matrix[lag]
            ).sum()
        )

    print(
        f"Rows                    : "
        f"{rows:,}"
    )

    print(
        f"All lags available      : "
        f"{ready.sum():,}"
    )

    print(
        f"Any missing lag         : "
        f"{any_missing.sum():,}"
    )

    print(
        f"Ready rows with missing : "
        f"{ready_with_missing.sum():,}"
    )

# ------------------------------------------------------------
# 3. Monthly summary table
# ------------------------------------------------------------
monthly_summary = pd.DataFrame(
    monthly_rows
)

print("\n" + "=" * 115)
print("MONTHLY TRAINING MISSINGNESS SUMMARY")
print("=" * 115)

display(
    monthly_summary[
        [
            "Month",
            "Total_Rows",
            "All_Lags_Available_Rows",
            "Rows_With_Any_Missing_Lag",
            "Ready_Rows_With_Missing_Lag",
            "Model_Ready_Percent"
        ]
    ]
)

# ------------------------------------------------------------
# 4. Overall training-period summary
# ------------------------------------------------------------
print("\n" + "=" * 115)
print("OVERALL TRAINING-PERIOD AUDIT")
print("=" * 115)

print(
    f"\nTraining rows (Jan–Jun)          : "
    f"{total_rows:,}"
)

print(
    f"All-lags-available rows          : "
    f"{total_ready:,}"
)

print(
    f"Rows with ≥1 missing lag         : "
    f"{total_missing_any:,}"
)

print(
    f"Model-ready rows with missing lag: "
    f"{total_ready_with_missing:,}"
)

print(
    f"Not-ready rows with no missing lag: "
    f"{total_not_ready_without_missing:,}"
)

print(
    f"\nModel-ready percentage           : "
    f"{100 * total_ready / total_rows:.4f}%"
)

# ------------------------------------------------------------
# 5. Missingness by lag feature
# ------------------------------------------------------------
lag_summary_rows = []

for lag in lag_cols:

    lag_summary_rows.append({

        "Lag_Feature":
            lag,

        "Missing_All_Training_Rows":
            lag_missing_totals[lag],

        "Missing_ModelReady_Rows":
            lag_missing_ready_totals[lag],

        "Missing_All_Rows_Percent":
            100
            * lag_missing_totals[lag]
            / total_rows,

        "Missing_ModelReady_Percent":
            (
                100
                * lag_missing_ready_totals[lag]
                / total_ready
                if total_ready > 0
                else np.nan
            )
    })

lag_summary = pd.DataFrame(
    lag_summary_rows
)

print("\n" + "=" * 115)
print("TRAINING MISSINGNESS BY LAG FEATURE")
print("=" * 115)

display(
    lag_summary
)

# ------------------------------------------------------------
# 6. Determine deployment-shift status
# ------------------------------------------------------------
training_complete_only = (
    total_ready_with_missing == 0
)

if training_complete_only:

    deployment_shift_status = (
        "YES — Model-ready training rows contained no missing "
        "lag values. Failure injection therefore creates an "
        "unseen missing-input regime relative to training."
    )

else:

    deployment_shift_status = (
        "NO — Some model-ready training rows already contained "
        "missing lag values."
    )

print("\n" + "=" * 115)
print("DEPLOYMENT-SHIFT INTERPRETATION")
print("=" * 115)

print(
    f"\n{deployment_shift_status}"
)

# ------------------------------------------------------------
# 7. Cross-check against injected experiment
# ------------------------------------------------------------
affected_file = (
    DATA_DIR
    / "Controlled_Failure_Injected_Affected_Predictions.parquet"
)

injected = pd.read_parquet(
    affected_file,
    columns=[
        "Missing_Lag_Count",
        "Missing_Lag_Pattern"
    ]
)

print("\nInjected experiment:")
print(
    f"  Affected rows with missing lags : "
    f"{len(injected):,}"
)

print(
    f"  Minimum missing lag count       : "
    f"{injected['Missing_Lag_Count'].min()}"
)

print(
    f"  Maximum missing lag count       : "
    f"{injected['Missing_Lag_Count'].max()}"
)

print(
    f"  Unique missing-lag patterns     : "
    f"{injected['Missing_Lag_Pattern'].nunique():,}"
)

# ------------------------------------------------------------
# 8. Concise paper-ready audit summary
# ------------------------------------------------------------
overall_summary = pd.DataFrame({

    "Metric": [
        "Training_Period",
        "Total_Training_Rows",
        "Model_Ready_Rows",
        "Model_Ready_Percent",
        "Training_Rows_With_Any_Missing_Lag",
        "Model_Ready_Rows_With_Missing_Lag",
        "Injected_Affected_Rows",
        "Injected_Min_Missing_Lags",
        "Injected_Max_Missing_Lags",
        "Deployment_Shift_Confirmed"
    ],

    "Value": [
        "2026-01-01 to 2026-06-30",
        total_rows,
        total_ready,
        100 * total_ready / total_rows,
        total_missing_any,
        total_ready_with_missing,
        len(injected),
        int(
            injected[
                "Missing_Lag_Count"
            ].min()
        ),
        int(
            injected[
                "Missing_Lag_Count"
            ].max()
        ),
        training_complete_only
    ]
})

print("\n" + "=" * 115)
print("PAPER-READY TRAINING EXPOSURE SUMMARY")
print("=" * 115)

display(
    overall_summary
)

# ------------------------------------------------------------
# 9. Save only critical outputs
# ------------------------------------------------------------
monthly_file = (
    TABLES_DIR
    / "Table_Training_Missingness_Monthly_Audit.csv"
)

monthly_summary.to_csv(
    monthly_file,
    index=False
)

lag_file = (
    TABLES_DIR
    / "Table_Training_Missingness_by_Lag.csv"
)

lag_summary.to_csv(
    lag_file,
    index=False
)

summary_file = (
    TABLES_DIR
    / "Table_Training_vs_Injected_Missingness_Audit.csv"
)

overall_summary.to_csv(
    summary_file,
    index=False
)

metadata = {

    "training_period":
        "2026-01-01 to 2026-06-30",

    "training_missingness_audit":
        {
            "total_rows":
                int(total_rows),

            "model_ready_rows":
                int(total_ready),

            "model_ready_rows_with_missing_lags":
                int(total_ready_with_missing),

            "training_complete_lag_vectors_only":
                bool(training_complete_only)
        },

    "deployment_shift_definition":
        (
            "Controlled failure injection introduces missing "
            "lag-feature combinations into models trained on "
            "complete model-ready lag vectors."
        ),

    "interpretation":
        deployment_shift_status
}

metadata_file = (
    RESULTS_DIR
    / "Training_Missingness_Deployment_Shift_Audit.json"
)

with open(
    metadata_file,
    "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )

print("\n" + "=" * 115)
print("FILES SAVED")
print("=" * 115)

print(monthly_file)
print(lag_file)
print(summary_file)
print(metadata_file)

print("\n" + "=" * 115)
print("STEP 17 COMPLETE")
print("=" * 115)

In [ ]:
# ============================================================
# STEP 18 — FINAL ROBUSTNESS FIGURE REFINEMENT
#
# Replace the original missing-lag figure with a clearer
# publication-ready relative-degradation figure.
#
# Main figure uses 1–4 missing lags.
# 5-lag case remains in the numerical tables but is excluded
# from the main figure because only 8 affected rows exist.
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

print("=" * 115)
print("STEP 18 — FINAL ROBUSTNESS FIGURE REFINEMENT")
print("=" * 115)

# ------------------------------------------------------------
# 1. Load robustness-by-missing-lag results
# ------------------------------------------------------------
source_file = (
    TABLES_DIR
    / "Table_Robustness_by_Missing_Lag_Count.csv"
)

results = pd.read_csv(source_file)

print("\nAvailable groups:")
display(
    results[
        [
            "Missing_Lag_Count",
            "Model",
            "Rows",
            "Clean_WAPE",
            "Injected_WAPE"
        ]
    ]
)

# ------------------------------------------------------------
# 2. Confirm why 5-lag case should not enter main figure
# ------------------------------------------------------------
five_lag = results[
    results["Missing_Lag_Count"] == 5
].copy()

print("\n" + "=" * 115)
print("FIVE-MISSING-LAG SAMPLE SIZE CHECK")
print("=" * 115)

display(
    five_lag[
        [
            "Model",
            "Rows",
            "Clean_WAPE",
            "Injected_WAPE"
        ]
    ]
)

# ------------------------------------------------------------
# 3. Main-paper plotting subset: 1–4 missing lags
# ------------------------------------------------------------
plot_data = results[
    results["Missing_Lag_Count"].between(
        1,
        4
    )
].copy()

plot_data[
    "WAPE_Ratio"
] = (
    plot_data["Injected_WAPE"]
    / plot_data["Clean_WAPE"]
)

plot_data[
    "WAPE_Increase_Percent"
] = (
    100
    * (
        plot_data["WAPE_Ratio"]
        - 1
    )
)

# Sample size for each missing-lag level
sample_sizes = (
    plot_data[
        [
            "Missing_Lag_Count",
            "Rows"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Missing_Lag_Count"
    )
)

print("\n" + "=" * 115)
print("MAIN-FIGURE SOURCE DATA")
print("=" * 115)

display(
    plot_data[
        [
            "Missing_Lag_Count",
            "Model",
            "Rows",
            "Clean_WAPE",
            "Injected_WAPE",
            "WAPE_Ratio",
            "WAPE_Increase_Percent"
        ]
    ]
)

# ------------------------------------------------------------
# 4. Publication-ready figure
# ------------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(8.0, 6.0)
)

for model_name in [
    "LightGBM",
    "XGBoost"
]:

    subset = (
        plot_data[
            plot_data["Model"]
            == model_name
        ]
        .sort_values(
            "Missing_Lag_Count"
        )
    )

    ax.plot(
        subset["Missing_Lag_Count"],
        subset["WAPE_Ratio"],
        marker="o",
        linewidth=2.2,
        markersize=7,
        label=model_name
    )

    # Point labels
    for _, row in subset.iterrows():

        ax.annotate(
            f"{row['WAPE_Ratio']:.1f}×",
            (
                row["Missing_Lag_Count"],
                row["WAPE_Ratio"]
            ),
            xytext=(0, 7),
            textcoords="offset points",
            ha="center",
            fontsize=9
        )

# Reference: no degradation
ax.axhline(
    1,
    linestyle="--",
    linewidth=1.2,
    label="No degradation"
)

# Log scale keeps both models readable
ax.set_yscale(
    "log"
)

ax.set_yticks(
    [
        1,
        2,
        5,
        10,
        20,
        50,
        100
    ]
)

ax.get_yaxis().set_major_formatter(
    ScalarFormatter()
)

# X-axis labels include sample size
x_values = (
    sample_sizes[
        "Missing_Lag_Count"
    ]
    .astype(int)
    .tolist()
)

x_labels = [
    f"{int(row.Missing_Lag_Count)}\n(n={int(row.Rows):,})"
    for row in sample_sizes.itertuples()
]

ax.set_xticks(
    x_values
)

ax.set_xticklabels(
    x_labels
)

ax.set_xlabel(
    "Number of Missing Historical Lag Features"
)

ax.set_ylabel(
    "WAPE Ratio (Failure-Injected / Clean)"
)

ax.set_title(
    "Relative Forecast Degradation as Historical Information Is Lost"
)

ax.grid(
    True,
    which="both",
    alpha=0.25
)

ax.legend(
    frameon=False
)

fig.tight_layout()

# ------------------------------------------------------------
# 5. Save publication figure
# ------------------------------------------------------------
png_file = (
    FIGURES_DIR
    / "Figure_Relative_WAPE_Degradation_by_Missing_Lags.png"
)

pdf_file = (
    FIGURES_DIR
    / "Figure_Relative_WAPE_Degradation_by_Missing_Lags.pdf"
)

fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    pdf_file,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 6. Save exact source data
# ------------------------------------------------------------
figure_data_file = (
    TABLES_DIR
    / "Figure_Data_Relative_WAPE_Degradation_by_Missing_Lags.csv"
)

plot_data.to_csv(
    figure_data_file,
    index=False
)

# ------------------------------------------------------------
# 7. Save exclusion note for transparency
# ------------------------------------------------------------
exclusion_note = pd.DataFrame({

    "Item": [
        "Main figure included missing-lag counts",
        "Main figure excluded missing-lag count",
        "Reason",
        "Excluded affected rows",
        "Numerical results retained elsewhere"
    ],

    "Value": [
        "1–4",
        "5",
        (
            "Only 8 affected observations were available "
            "for the five-missing-lag condition; displaying "
            "this sparse extreme condition on the primary "
            "figure would dominate the plotting scale."
        ),
        8,
        True
    ]
})

note_file = (
    RESULTS_DIR
    / "Figure_2_Main_Manuscript_Exclusion_Note.csv"
)

exclusion_note.to_csv(
    note_file,
    index=False
)

# ------------------------------------------------------------
# 8. Update paper figure manifest
# ------------------------------------------------------------
manifest_file = (
    RESULTS_DIR
    / "Notebook_4_Paper_Figure_Manifest.csv"
)

manifest = pd.read_csv(
    manifest_file
)

mask = (
    manifest["Figure"]
    == "Figure 2"
)

manifest.loc[
    mask,
    "Title"
] = (
    "Relative Forecast Degradation as Historical "
    "Information Is Lost"
)

manifest.loc[
    mask,
    "Primary_Message"
] = (
    "Accuracy degradation grows as historical lag "
    "information is lost, with XGBoost showing much "
    "greater sensitivity than LightGBM."
)

manifest.loc[
    mask,
    "PNG_File"
] = str(
    png_file
)

manifest.loc[
    mask,
    "PDF_File"
] = str(
    pdf_file
)

manifest.loc[
    mask,
    "Source_Data_File"
] = str(
    figure_data_file
)

manifest.to_csv(
    manifest_file,
    index=False
)

# ------------------------------------------------------------
# 9. Final output
# ------------------------------------------------------------
print("\n" + "=" * 115)
print("FINAL FIGURE 2 SAVED")
print("=" * 115)

print(f"\nPNG:\n{png_file}")
print(f"\nPDF:\n{pdf_file}")
print(f"\nSource data:\n{figure_data_file}")
print(f"\nTransparency note:\n{note_file}")
print(f"\nUpdated figure manifest:\n{manifest_file}")

print("\n" + "=" * 115)
print("STEP 18 COMPLETE")
print("=" * 115)

In [ ]:
# ============================================================
# STEP 19 — NOTEBOOK 4 FINAL AUDIT & RECOVERY PACKAGE
#
# Purpose:
#   1. Audit all critical Notebook 4 outputs
#   2. Build complete file inventory
#   3. Extract final paper-ready numerical findings
#   4. Save final JSON + CSV + human-readable TXT summary
#   5. Preserve methodological decisions for full recovery
# ============================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime

print("=" * 120)
print("STEP 19 — NOTEBOOK 4 FINAL AUDIT & RECOVERY PACKAGE")
print("=" * 120)

# ------------------------------------------------------------
# 1. Core paths
# ------------------------------------------------------------
N4 = NOTEBOOK4_DIR

print(f"\nNotebook 4 root:\n{N4}")

# ------------------------------------------------------------
# 2. Critical files expected
# ------------------------------------------------------------
critical_files = {

    # ---------------- DATA ----------------
    "August real failure runs":
        DATA_DIR / "August_Real_Failure_Runs_Audited.csv",

    "August real failure intervals":
        DATA_DIR / "August_Real_Failure_Intervals_15min.parquet",

    "Isolated real failure recovery events":
        DATA_DIR / "August_Isolated_Real_Failure_Recovery_Events.csv",

    "Failure blackout amplification data":
        DATA_DIR / "August_Isolated_Failure_Blackout_Amplification.csv",

    "Pre-test empirical failure library":
        DATA_DIR / "PreTest_Real_Failure_Pattern_Library_JanJul.csv",

    "Controlled August injection design":
        DATA_DIR / "Controlled_August_Failure_Injection_Events.csv",

    "Failure-injected affected predictions":
        DATA_DIR / "Controlled_Failure_Injected_Affected_Predictions.parquet",

    "Event-level robustness results":
        DATA_DIR / "Controlled_Failure_Event_Level_Robustness_With_Paired_Comparison.parquet",

    # ---------------- TABLES ----------------
    "Failure blackout headline results":
        TABLES_DIR / "Table_Failure_Blackout_Headline_Results.csv",

    "Injection design summary":
        TABLES_DIR / "Table_Controlled_Injection_Design_Summary.csv",

    "Overall robustness performance":
        TABLES_DIR / "Table_Controlled_Failure_Overall_Robustness_Performance.csv",

    "Robustness by missing lag count":
        TABLES_DIR / "Table_Robustness_by_Missing_Lag_Count.csv",

    "Robustness by missing lag pattern":
        TABLES_DIR / "Table_Robustness_by_Missing_Lag_Pattern.csv",

    "Robustness by failure duration":
        TABLES_DIR / "Table_Robustness_by_Failure_Duration.csv",

    "Event-level statistical inference":
        TABLES_DIR / "Table_Event_Level_Robustness_Statistical_Inference.csv",

    "Paired LightGBM vs XGBoost inference":
        TABLES_DIR / "Table_Paired_LightGBM_vs_XGBoost_Robustness_Inference.csv",

    "Training vs injected missingness audit":
        TABLES_DIR / "Table_Training_vs_Injected_Missingness_Audit.csv",

    # ---------------- FIGURES ----------------
    "Figure 1 PNG":
        FIGURES_DIR / "Figure_Clean_vs_Injected_WAPE.png",

    "Figure 1 PDF":
        FIGURES_DIR / "Figure_Clean_vs_Injected_WAPE.pdf",

    "Figure 2 FINAL PNG":
        FIGURES_DIR / "Figure_Relative_WAPE_Degradation_by_Missing_Lags.png",

    "Figure 2 FINAL PDF":
        FIGURES_DIR / "Figure_Relative_WAPE_Degradation_by_Missing_Lags.pdf",

    "Figure 3 PNG":
        FIGURES_DIR / "Figure_Event_Level_LightGBM_vs_XGBoost_Robustness.png",

    "Figure 3 PDF":
        FIGURES_DIR / "Figure_Event_Level_LightGBM_vs_XGBoost_Robustness.pdf",

    # ---------------- RESULTS ----------------
    "Paper figure manifest":
        RESULTS_DIR / "Notebook_4_Paper_Figure_Manifest.csv",

    "Failure injection metadata":
        RESULTS_DIR / "Controlled_Failure_Injection_Experiment_Metadata.json",

    "Deployment-shift audit":
        RESULTS_DIR / "Training_Missingness_Deployment_Shift_Audit.json",

    "Statistical inference metadata":
        RESULTS_DIR / "Event_Level_Statistical_Inference_Metadata.json",

    "Figure 2 transparency note":
        RESULTS_DIR / "Figure_2_Main_Manuscript_Exclusion_Note.csv"
}

# ------------------------------------------------------------
# 3. Critical file audit
# ------------------------------------------------------------
audit_rows = []

for description, path in critical_files.items():

    exists = path.exists()

    size_mb = (
        path.stat().st_size / (1024 ** 2)
        if exists
        else np.nan
    )

    audit_rows.append({
        "Description": description,
        "Path": str(path),
        "Exists": exists,
        "Size_MB": size_mb
    })

critical_audit = pd.DataFrame(audit_rows)

print("\n" + "=" * 120)
print("CRITICAL FILE AUDIT")
print("=" * 120)

display(critical_audit)

missing_critical = critical_audit[
    ~critical_audit["Exists"]
]

print(
    f"\nCritical files expected : "
    f"{len(critical_audit):,}"
)

print(
    f"Critical files present  : "
    f"{critical_audit['Exists'].sum():,}"
)

print(
    f"Critical files missing  : "
    f"{len(missing_critical):,}"
)

# ------------------------------------------------------------
# 4. Full Notebook 4 inventory
# ------------------------------------------------------------
all_files = sorted(
    [
        p for p in N4.rglob("*")
        if p.is_file()
    ]
)

inventory_rows = []

for path in all_files:

    inventory_rows.append({
        "Relative_Path":
            str(path.relative_to(N4)),

        "Extension":
            path.suffix.lower(),

        "Size_MB":
            path.stat().st_size / (1024 ** 2),

        "Modified":
            datetime.fromtimestamp(
                path.stat().st_mtime
            ).isoformat(
                timespec="seconds"
            )
    })

inventory = pd.DataFrame(
    inventory_rows
)

print("\n" + "=" * 120)
print("FULL NOTEBOOK 4 FILE INVENTORY")
print("=" * 120)

print(
    f"\nTotal files currently stored: "
    f"{len(inventory):,}"
)

print(
    f"Total output size: "
    f"{inventory['Size_MB'].sum():.2f} MB"
)

# ------------------------------------------------------------
# 5. Helper for long-format Metric / Value tables
# ------------------------------------------------------------
def read_metric_table(path):

    df = pd.read_csv(path)

    if (
        "Metric" not in df.columns
        or "Value" not in df.columns
    ):
        raise ValueError(
            f"{path.name} does not contain Metric/Value columns."
        )

    return dict(
        zip(
            df["Metric"],
            df["Value"]
        )
    )

# ------------------------------------------------------------
# 6. Load headline result files
# ------------------------------------------------------------
blackout = read_metric_table(
    TABLES_DIR
    / "Table_Failure_Blackout_Headline_Results.csv"
)

injection_design = read_metric_table(
    TABLES_DIR
    / "Table_Controlled_Injection_Design_Summary.csv"
)

paired_inference = read_metric_table(
    TABLES_DIR
    / "Table_Paired_LightGBM_vs_XGBoost_Robustness_Inference.csv"
)

training_shift = read_metric_table(
    TABLES_DIR
    / "Table_Training_vs_Injected_Missingness_Audit.csv"
)

overall = pd.read_csv(
    TABLES_DIR
    / "Table_Controlled_Failure_Overall_Robustness_Performance.csv"
)

event_inference = pd.read_csv(
    TABLES_DIR
    / "Table_Event_Level_Robustness_Statistical_Inference.csv"
)

# ------------------------------------------------------------
# 7. Extract model results
# ------------------------------------------------------------
lgb = overall[
    overall["Model"] == "LightGBM"
].iloc[0]

xgb = overall[
    overall["Model"] == "XGBoost"
].iloc[0]

persistence = overall[
    overall["Model"] == "Persistence_15min"
].iloc[0]

seasonal = overall[
    overall["Model"] == "Seasonal_Naive_24h"
].iloc[0]

lgb_event = event_inference[
    event_inference["Model"] == "LightGBM"
].iloc[0]

xgb_event = event_inference[
    event_inference["Model"] == "XGBoost"
].iloc[0]

# ------------------------------------------------------------
# 8. Paper-ready final summary table
# ------------------------------------------------------------
paper_summary_rows = [

    # ========================================================
    # REAL FAILURE OBSERVATION
    # ========================================================
    {
        "Section": "Observed Real Failures",
        "Metric": "August isolated failure events",
        "Value": blackout["Isolated_Failure_Events"],
        "Interpretation":
            "Isolated real SCATS failures used for blackout analysis."
    },

    {
        "Section": "Observed Real Failures",
        "Metric": "Median actual failure duration (hours)",
        "Value": blackout["Median_Actual_Failure_Hours"],
        "Interpretation":
            "Typical observed isolated failure was short."
    },

    {
        "Section": "Observed Real Failures",
        "Metric": "Median forecast blackout (hours)",
        "Value": blackout["Median_Total_Forecast_Blackout_Hours"],
        "Interpretation":
            "Strict clean-data forecasting pipeline remained unavailable much longer than the physical failure."
    },

    {
        "Section": "Observed Real Failures",
        "Metric": "Median blackout amplification factor",
        "Value": blackout["Median_Blackout_Amplification_Factor"],
        "Interpretation":
            "Forecast unavailability was strongly amplified relative to physical failure duration."
    },

    # ========================================================
    # CONTROLLED EXPERIMENT DESIGN
    # ========================================================
    {
        "Section": "Controlled Experiment",
        "Metric": "Selected injection events",
        "Value": injection_design["Selected_Injection_Events"],
        "Interpretation":
            "Maximum one empirically grounded injected event per detector."
    },

    {
        "Section": "Controlled Experiment",
        "Metric": "Unique injected detectors",
        "Value": injection_design["Unique_Injected_Detectors"],
        "Interpretation":
            "Each selected detector contributed one controlled event."
    },

    {
        "Section": "Controlled Experiment",
        "Metric": "Injected failure intervals",
        "Value": injection_design["Total_Injected_Failure_Intervals"],
        "Interpretation":
            "Historical observations removed according to real Jan–Jul SCATS failure patterns."
    },

    # ========================================================
    # LIGHTGBM
    # ========================================================
    {
        "Section": "LightGBM Robustness",
        "Metric": "Prediction availability (%)",
        "Value": lgb["Availability_Percent"],
        "Interpretation":
            "Native missing-value handling produced predictions for every affected target."
    },

    {
        "Section": "LightGBM Robustness",
        "Metric": "Clean WAPE (%)",
        "Value": lgb["Paired_Clean_WAPE"],
        "Interpretation":
            "Baseline accuracy on the exact affected targets before failure injection."
    },

    {
        "Section": "LightGBM Robustness",
        "Metric": "Failure-injected WAPE (%)",
        "Value": lgb["Injected_WAPE"],
        "Interpretation":
            "Accuracy after empirically grounded lag missingness."
    },

    {
        "Section": "LightGBM Robustness",
        "Metric": "WAPE increase (percentage points)",
        "Value": lgb["WAPE_Delta_pp"],
        "Interpretation":
            "Absolute robustness loss under injected detector failures."
    },

    {
        "Section": "LightGBM Robustness",
        "Metric": "Injected bias",
        "Value": lgb["Injected_Bias"],
        "Interpretation":
            "Negative value indicates underprediction tendency."
    },

    # ========================================================
    # XGBOOST
    # ========================================================
    {
        "Section": "XGBoost Robustness",
        "Metric": "Prediction availability (%)",
        "Value": xgb["Availability_Percent"],
        "Interpretation":
            "Native missing-value handling also retained full availability."
    },

    {
        "Section": "XGBoost Robustness",
        "Metric": "Clean WAPE (%)",
        "Value": xgb["Paired_Clean_WAPE"],
        "Interpretation":
            "Clean performance was comparable to LightGBM."
    },

    {
        "Section": "XGBoost Robustness",
        "Metric": "Failure-injected WAPE (%)",
        "Value": xgb["Injected_WAPE"],
        "Interpretation":
            "Very large degradation under missing historical features."
    },

    {
        "Section": "XGBoost Robustness",
        "Metric": "WAPE increase (percentage points)",
        "Value": xgb["WAPE_Delta_pp"],
        "Interpretation":
            "Much larger robustness loss than LightGBM."
    },

    {
        "Section": "XGBoost Robustness",
        "Metric": "Injected bias",
        "Value": xgb["Injected_Bias"],
        "Interpretation":
            "Positive value indicates strong overprediction tendency."
    },

    # ========================================================
    # NAIVE MODEL AVAILABILITY
    # ========================================================
    {
        "Section": "Availability Trade-off",
        "Metric": "Persistence availability (%)",
        "Value": persistence["Availability_Percent"],
        "Interpretation":
            "Direct lag dependence prevents predictions when Lag_1 is unavailable."
    },

    {
        "Section": "Availability Trade-off",
        "Metric": "Seasonal naive availability (%)",
        "Value": seasonal["Availability_Percent"],
        "Interpretation":
            "Direct Lag_96 dependence creates forecast unavailability."
    },

    # ========================================================
    # EVENT-LEVEL STATISTICS
    # ========================================================
    {
        "Section": "Statistical Evidence",
        "Metric": "LightGBM worsened events (%)",
        "Value": lgb_event["Worsened_Percent"],
        "Interpretation":
            "Share of controlled failure events with increased event-level MAE."
    },

    {
        "Section": "Statistical Evidence",
        "Metric": "XGBoost worsened events (%)",
        "Value": xgb_event["Worsened_Percent"],
        "Interpretation":
            "Failure-induced degradation was nearly universal for XGBoost."
    },

    {
        "Section": "Statistical Evidence",
        "Metric": "Events where XGBoost degraded more (%)",
        "Value": paired_inference["XGB_Degraded_More_Percent"],
        "Interpretation":
            "Direct paired comparison under identical injected failure events."
    },

    {
        "Section": "Statistical Evidence",
        "Metric": "Mean XGB minus LGB MAE degradation",
        "Value": paired_inference["Mean_XGB_minus_LGB_MAE_Degradation"],
        "Interpretation":
            "Positive values indicate greater XGBoost robustness loss."
    },

    {
        "Section": "Statistical Evidence",
        "Metric": "Paired Wilcoxon p-value",
        "Value": paired_inference["Paired_Wilcoxon_OneSided_P"],
        "Interpretation":
            "Tests whether XGBoost degradation exceeds LightGBM degradation across matched events."
    },

    # ========================================================
    # DEPLOYMENT SHIFT
    # ========================================================
    {
        "Section": "Deployment Shift",
        "Metric": "Training model-ready rows",
        "Value": training_shift["Model_Ready_Rows"],
        "Interpretation":
            "Rows actually eligible for model training."
    },

    {
        "Section": "Deployment Shift",
        "Metric": "Model-ready rows with missing lags",
        "Value": training_shift["Model_Ready_Rows_With_Missing_Lag"],
        "Interpretation":
            "Confirms whether training exposed the models to missing lag vectors."
    },

    {
        "Section": "Deployment Shift",
        "Metric": "Deployment shift confirmed",
        "Value": training_shift["Deployment_Shift_Confirmed"],
        "Interpretation":
            "Injected missing-lag combinations were unseen in model-ready training data."
    }
]

paper_summary = pd.DataFrame(
    paper_summary_rows
)

print("\n" + "=" * 120)
print("FINAL PAPER-READY NUMERICAL SUMMARY")
print("=" * 120)

display(paper_summary)

# ------------------------------------------------------------
# 9. Final methodological record
# ------------------------------------------------------------
methodology = {

    "notebook_title":
        (
            "Notebook 4 — Robustness and Reliability Analysis "
            "Under Real-World Detector Failures"
        ),

    "primary_research_question":
        (
            "How robust are 15-minute traffic-volume forecasting "
            "models when real-world detector failure patterns "
            "remove historical traffic observations?"
        ),

    "data_period":
        "2026-01-01 to 2026-08-30",

    "forecasting_task":
        "15-minute-ahead traffic volume forecasting",

    "models":
        [
            "LightGBM",
            "XGBoost",
            "Persistence 15-minute",
            "Seasonal Naive 24-hour"
        ],

    "real_failure_analysis": {

        "observed_period":
            "August 2026 held-out test month",

        "important_interpretation":
            (
                "Exact real-failure intervals were absent from "
                "the clean forecasting dataset because corrupted "
                "targets had been excluded during preprocessing."
            ),

        "recovery_mechanism":
            (
                "Strict detector-day cleaning delayed clean-data "
                "return, and Lag_96 generally added an additional "
                "24-hour feature-readiness delay."
            )
    },

    "controlled_failure_experiment": {

        "failure_pattern_source":
            "Observed real SCATS failures from Jan-Jul 2026",

        "test_period":
            "August 2026",

        "events":
            int(
                float(
                    injection_design[
                        "Selected_Injection_Events"
                    ]
                )
            ),

        "event_selection":
            (
                "Maximum one event per detector; same detector, "
                "failure duration, start interval and day-of-week "
                "preserved."
            ),

        "actual_target_modified":
            False,

        "model_retrained":
            False,

        "injected_features":
            [
                "Lag_1",
                "Lag_2",
                "Lag_4",
                "Lag_8",
                "Lag_96"
            ]
    },

    "training_exposure": {

        "model_ready_missing_lags":
            int(
                float(
                    training_shift[
                        "Model_Ready_Rows_With_Missing_Lag"
                    ]
                )
            ),

        "deployment_shift_confirmed":
            str(
                training_shift[
                    "Deployment_Shift_Confirmed"
                ]
            )
    },

    "statistical_unit":
        "Controlled detector failure event",

    "statistical_methods":
        [
            "Event-level MAE degradation",
            "10,000-resample bootstrap 95% confidence intervals",
            "One-sided Wilcoxon signed-rank tests",
            "Event-level sign tests",
            "Direct paired LightGBM vs XGBoost comparison"
        ],

    "figure_policy": {

        "main_figures":
            [
                "Clean vs failure-injected WAPE",
                "Relative WAPE degradation by missing-lag count",
                "Paired event-level LightGBM vs XGBoost robustness"
            ],

        "five_missing_lag_case":
            (
                "Retained numerically but excluded from main Figure 2 "
                "because only 8 affected observations were available."
            )
    },

    "scope_decision":
        (
            "Mitigation strategies such as imputation, failure-aware "
            "training, missingness augmentation or model retraining "
            "were intentionally NOT added to Notebook 4. They form "
            "a separate robustness-mitigation research stage."
        )
}

# ------------------------------------------------------------
# 10. Human-readable recovery document
# ------------------------------------------------------------
recovery_text = f"""
NOTEBOOK 4 — FINAL RECOVERY SUMMARY
===================================

TITLE
-----
Robustness and Reliability Analysis Under Real-World Detector Failures

PROJECT CONTEXT
---------------
Forecasting task:
15-minute-ahead traffic-volume forecasting.

Data coverage:
2026-01-01 to 2026-08-30.

Notebook 3 supplied the final forecasting models:
- LightGBM
- XGBoost
- Persistence 15-minute baseline
- Seasonal Naive 24-hour baseline

Notebook 4 studied what happens when historical traffic observations
become unavailable because of detector failures.


============================================================
1. REAL DETECTOR FAILURE ANALYSIS
============================================================

August real failure library:
- 242 contiguous real failure runs
- 146 affected detectors
- 29 affected dates
- 3,010 failed 15-minute intervals

Exact failure intervals were NOT present in the clean forecasting
or final prediction datasets. This was expected because corrupted
targets had already been excluded during preprocessing.

Therefore, prediction error was NOT evaluated against corrupted
sensor observations during the actual failure itself.

The correct operational question became:
How long does failure influence forecasting availability after the
detector problem occurs?


ISOLATED REAL FAILURE ANALYSIS
------------------------------
Isolated recovery events:
{blackout['Isolated_Failure_Events']}

Median observed physical failure duration:
{float(blackout['Median_Actual_Failure_Hours']):.2f} hours

Median total forecast blackout:
{float(blackout['Median_Total_Forecast_Blackout_Hours']):.2f} hours

Median blackout amplification:
{float(blackout['Median_Blackout_Amplification_Factor']):.2f}x

Key mechanism:
The strict clean-data pipeline typically returned clean detector data
at the start of a later valid detector-day. After clean data returned,
Lag_96 usually imposed an additional 96 intervals = 24 hours before
all model inputs became available.

This is a PIPELINE / FEATURE-DEPENDENCY effect, not a computational
delay caused by LightGBM or XGBoost.


============================================================
2. EMPIRICAL FAILURE PATTERN LIBRARY
============================================================

Real detector failure patterns from Jan-Jul 2026 were used to design
the controlled experiment.

Pre-test failure runs:
2,223

Affected detectors:
604

Median real failure run length:
2 intervals = 30 minutes

The Jan-Jul failure-duration distribution was highly similar to the
held-out August observed failure distribution.

KS statistic:
0.0350

KS p-value:
0.941884


============================================================
3. CONTROLLED FAILURE-INJECTION DESIGN
============================================================

Controlled events:
{int(float(injection_design['Selected_Injection_Events']))}

Unique injected detectors:
{int(float(injection_design['Unique_Injected_Detectors']))}

Injected failed detector intervals:
{int(float(injection_design['Total_Injected_Failure_Intervals']))}

Design rules:
- Maximum one injected event per detector.
- Same detector preserved.
- Real failure run length preserved.
- Real start interval preserved.
- Day-of-week preserved.
- August remained the held-out test period.
- Every injection window had complete clean baseline predictions.
- Ground-truth Actual values were never modified.
- Models were never retrained.

Injected missingness propagated through:
Lag_1, Lag_2, Lag_4, Lag_8 and Lag_96.

Affected forecasting targets:
12,662


============================================================
4. MODEL REPRODUCIBILITY
============================================================

Notebook 4 reproduced Notebook 3 predictions successfully.

XGBoost:
Exact after applying Notebook 3 zero-clipping post-processing.

LightGBM:
Numerically equivalent after clipping.
Mean absolute reproduction difference was approximately 6.5e-7,
with maximum difference approximately 2.1e-5.

Therefore the robustness results are attributable to failure
injection rather than a changed model implementation.


============================================================
5. PREDICTION AVAILABILITY
============================================================

LightGBM availability:
{float(lgb['Availability_Percent']):.2f}%

XGBoost availability:
{float(xgb['Availability_Percent']):.2f}%

Persistence 15-minute availability:
{float(persistence['Availability_Percent']):.2f}%

Seasonal Naive 24-hour availability:
{float(seasonal['Availability_Percent']):.2f}%

Interpretation:
LightGBM and XGBoost can technically score NaN inputs and therefore
retain 100% availability.

The naive models lose forecasts whenever their directly required
lag is missing.

Availability and accuracy MUST therefore be reported separately.


============================================================
6. MAIN ROBUSTNESS RESULTS
============================================================

LIGHTGBM
--------
Clean WAPE:
{float(lgb['Paired_Clean_WAPE']):.3f}%

Failure-injected WAPE:
{float(lgb['Injected_WAPE']):.3f}%

WAPE increase:
{float(lgb['WAPE_Delta_pp']):.3f} percentage points

Clean MAE:
{float(lgb['Paired_Clean_MAE']):.3f}

Failure-injected MAE:
{float(lgb['Injected_MAE']):.3f}

Injected bias:
{float(lgb['Injected_Bias']):.3f}

Interpretation:
LightGBM remained available but showed substantial degradation and
tended toward underprediction.


XGBOOST
-------
Clean WAPE:
{float(xgb['Paired_Clean_WAPE']):.3f}%

Failure-injected WAPE:
{float(xgb['Injected_WAPE']):.3f}%

WAPE increase:
{float(xgb['WAPE_Delta_pp']):.3f} percentage points

Clean MAE:
{float(xgb['Paired_Clean_MAE']):.3f}

Failure-injected MAE:
{float(xgb['Injected_MAE']):.3f}

Injected bias:
{float(xgb['Injected_Bias']):.3f}

Interpretation:
XGBoost remained technically available but suffered severe
accuracy degradation and strong overprediction.


============================================================
7. EVENT-LEVEL STATISTICAL INFERENCE
============================================================

Statistical unit:
604 controlled failure events.

This avoids treating correlated 15-minute rows within the same
failure event as statistically independent.


LIGHTGBM
--------
Mean event-level MAE degradation:
{float(lgb_event['Mean_MAE_Delta']):.3f}

95% bootstrap CI:
[{float(lgb_event['Mean_MAE_Delta_CI_Lower']):.3f},
 {float(lgb_event['Mean_MAE_Delta_CI_Upper']):.3f}]

Events with worsened MAE:
{int(lgb_event['Worsened_Events'])} / 604
({float(lgb_event['Worsened_Percent']):.2f}%)


XGBOOST
-------
Mean event-level MAE degradation:
{float(xgb_event['Mean_MAE_Delta']):.3f}

95% bootstrap CI:
[{float(xgb_event['Mean_MAE_Delta_CI_Lower']):.3f},
 {float(xgb_event['Mean_MAE_Delta_CI_Upper']):.3f}]

Events with worsened MAE:
{int(xgb_event['Worsened_Events'])} / 604
({float(xgb_event['Worsened_Percent']):.2f}%)


DIRECT MODEL COMPARISON
-----------------------
Events where XGBoost degraded more than LightGBM:
{float(paired_inference['XGB_Degraded_More_Percent']):.2f}%

Mean XGBoost-minus-LightGBM MAE degradation:
{float(paired_inference['Mean_XGB_minus_LGB_MAE_Degradation']):.3f}

95% bootstrap CI:
[{float(paired_inference['Mean_95CI_Lower']):.3f},
 {float(paired_inference['Mean_95CI_Upper']):.3f}]

Paired one-sided Wilcoxon p-value:
{float(paired_inference['Paired_Wilcoxon_OneSided_P']):.6e}

The controlled-event evidence therefore strongly supports greater
native robustness of LightGBM than XGBoost under the tested missing-
historical-input regime.


============================================================
8. TRAINING / DEPLOYMENT SHIFT
============================================================

Total Jan-Jun training rows:
{training_shift['Total_Training_Rows']}

Model-ready training rows:
{training_shift['Model_Ready_Rows']}

Rows in full training-period forecasting data with >=1 missing lag:
{training_shift['Training_Rows_With_Any_Missing_Lag']}

MODEL-READY training rows with missing lag values:
{training_shift['Model_Ready_Rows_With_Missing_Lag']}

Deployment shift confirmed:
{training_shift['Deployment_Shift_Confirmed']}

CRITICAL INTERPRETATION:
Neither final model was trained on missing-lag model-ready samples.
The controlled failure experiment therefore created an unseen
missing-input deployment regime.

The correct conclusion is NOT that XGBoost is universally inferior.

The correct conclusion is:
Under comparable clean-data accuracy, LightGBM demonstrated much
greater native resilience than XGBoost when real-failure-derived
missing historical inputs were introduced outside the models'
training regime.


============================================================
9. PAPER FIGURES
============================================================

MAIN FIGURE 1
Clean vs Failure-Injected WAPE

MAIN FIGURE 2
Relative WAPE Degradation by Number of Missing Historical Lags

The five-missing-lag condition is NOT displayed in the main figure
because it contained only n=8 affected observations.
It remains fully preserved in the numerical tables.

MAIN FIGURE 3
Paired Event-Level LightGBM vs XGBoost Robustness

All main figures were saved as:
- PNG at 600 dpi
- PDF vector format
- Separate CSV source data


============================================================
10. IMPORTANT PAPER INTERPRETATION
============================================================

Notebook 4 identifies TWO distinct operational responses:

A) STRICT CLEAN-DATA PIPELINE
Missing historical observations cause forecasts to become unavailable
until feature requirements are restored.

B) NATIVE MISSING-TOLERANT MODEL SCORING
LightGBM and XGBoost can still return numerical predictions with NaN
lag inputs, but forecast accuracy may degrade substantially.

This distinction between:
- forecast availability
and
- forecast accuracy

is central to the paper.


============================================================
11. SCOPE BOUNDARY
============================================================

Notebook 4 intentionally does NOT introduce mitigation strategies.

Do NOT add ad-hoc mean filling, zero filling, forward filling or
retraining to Notebook 4.

Potential mitigation methods belong in the next research stage:
- Missingness-aware model training
- Failure-pattern augmentation
- Robust imputation
- Feature fallback strategies
- Failure-aware forecasting architectures
- Availability-accuracy tradeoff analysis


============================================================
12. NOTEBOOK 4 STATUS
============================================================

Notebook 4 analytical workflow:
COMPLETE

Critical outputs present:
{int(critical_audit['Exists'].sum())} / {len(critical_audit)}

Missing critical files:
{len(missing_critical)}

Total Notebook 4 files before final recovery outputs:
{len(inventory)}

Notebook 4 can be reconstructed from the saved Google Drive outputs
without relying on the ChatGPT conversation.
"""

# ------------------------------------------------------------
# 11. Save final outputs
# ------------------------------------------------------------
critical_audit_file = (
    RESULTS_DIR
    / "Notebook_4_Critical_File_Audit.csv"
)

critical_audit.to_csv(
    critical_audit_file,
    index=False
)

inventory_file = (
    RESULTS_DIR
    / "Notebook_4_FULL_File_Inventory.csv"
)

inventory.to_csv(
    inventory_file,
    index=False
)

paper_summary_file = (
    RESULTS_DIR
    / "Notebook_4_FINAL_Paper_Ready_Summary.csv"
)

paper_summary.to_csv(
    paper_summary_file,
    index=False
)

methodology_file = (
    RESULTS_DIR
    / "Notebook_4_FINAL_Methodology.json"
)

with open(
    methodology_file,
    "w"
) as f:

    json.dump(
        methodology,
        f,
        indent=2
    )

recovery_file = (
    RESULTS_DIR
    / "Notebook_4_FINAL_Recovery_Summary.txt"
)

with open(
    recovery_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        recovery_text.strip()
        + "\n"
    )

# ------------------------------------------------------------
# 12. Compact final JSON summary
# ------------------------------------------------------------
final_json = {

    "notebook":
        4,

    "title":
        methodology["notebook_title"],

    "status":
        (
            "COMPLETE"
            if len(missing_critical) == 0
            else "REVIEW REQUIRED"
        ),

    "critical_files_present":
        int(
            critical_audit[
                "Exists"
            ].sum()
        ),

    "critical_files_expected":
        int(
            len(critical_audit)
        ),

    "main_results": {

        "median_real_failure_hours":
            float(
                blackout[
                    "Median_Actual_Failure_Hours"
                ]
            ),

        "median_forecast_blackout_hours":
            float(
                blackout[
                    "Median_Total_Forecast_Blackout_Hours"
                ]
            ),

        "controlled_events":
            int(
                float(
                    injection_design[
                        "Selected_Injection_Events"
                    ]
                )
            ),

        "lightgbm_clean_wape":
            float(
                lgb[
                    "Paired_Clean_WAPE"
                ]
            ),

        "lightgbm_injected_wape":
            float(
                lgb[
                    "Injected_WAPE"
                ]
            ),

        "xgboost_clean_wape":
            float(
                xgb[
                    "Paired_Clean_WAPE"
                ]
            ),

        "xgboost_injected_wape":
            float(
                xgb[
                    "Injected_WAPE"
                ]
            ),

        "xgboost_degraded_more_events_percent":
            float(
                paired_inference[
                    "XGB_Degraded_More_Percent"
                ]
            )
    },

    "next_stage":
        (
            "Failure-aware mitigation and robust forecasting "
            "strategies should be evaluated separately."
        )
}

final_json_file = (
    RESULTS_DIR
    / "Notebook_4_FINAL_Summary.json"
)

with open(
    final_json_file,
    "w"
) as f:

    json.dump(
        final_json,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 13. Final validation
# ------------------------------------------------------------
print("\n" + "=" * 120)
print("FINAL NOTEBOOK 4 STATUS")
print("=" * 120)

print(
    f"\nCritical files present : "
    f"{critical_audit['Exists'].sum()} / "
    f"{len(critical_audit)}"
)

print(
    f"Missing critical files : "
    f"{len(missing_critical)}"
)

print(
    f"\nFinal status: "
    f"{final_json['status']}"
)

print("\nFinal recovery outputs:")

for path in [
    critical_audit_file,
    inventory_file,
    paper_summary_file,
    methodology_file,
    recovery_file,
    final_json_file
]:
    print(f"  ✓ {path}")

print("\n" + "=" * 120)

if len(missing_critical) == 0:

    print("✅ NOTEBOOK 4 IS COMPLETE AND FULLY RECOVERABLE.")

else:

    print("⚠️ NOTEBOOK 4 REQUIRES REVIEW BEFORE CLOSING COLAB.")

print("=" * 120)